# HRM Fine-Tuning - HRB Architecture (6x6)

This notebook fine-tunes the HRM base model using the **hrm_hrb_v1** architecture
on 6x6 Sudoku puzzles using LoRA.


In [ ]:
!mkdir -p dataset/raw-data models/hrm utils config/arch


In [ ]:
%%writefile create_eval_notebook.py
import json
import os

nb = {
 "cells": [
  {
   "cell_type": "code",
   "execution_count": None,
   "metadata": {},
   "outputs": [],
   "source": [
    "!git clone https://github.com/jagan25-mj/NHRT.git\n",
    "%cd NHRT"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": None,
   "metadata": {},
   "outputs": [],
   "source": [
    "!python dataset/build_6x6_sudoku_dataset.py"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": None,
   "metadata": {},
   "outputs": [],
   "source": [
    "import os\n",
    "import glob\n",
    "\n",
    "# In Kaggle, attached notebook outputs usually appear in /kaggle/input/\n",
    "# Let's search for the highest step checkpoint in /kaggle/input/\n",
    "checkpoint_files = glob.glob('/kaggle/input/**/step_*', recursive=True)\n",
    "checkpoint_files = [f for f in checkpoint_files if 'all_preds' not in f]\n",
    "\n",
    "if not checkpoint_files:\n",
    "    print('No checkpoints found! Make sure you attached the output of the previous notebook.')\n",
    "else:\n",
    "    # Sort by step number\n",
    "    def extract_step(f):\n",
    "        basename = os.path.basename(f)\n",
    "        return int(basename.replace('step_', ''))\n",
    "    \n",
    "    latest_checkpoint = max(checkpoint_files, key=extract_step)\n",
    "    print(f'Found checkpoint: {latest_checkpoint}')\n",
    "    \n",
    "    # Run evaluation\n",
    "    cmd = f'python evaluate.py checkpoint=\"{latest_checkpoint}\"'\n",
    "    print(f'Running: {cmd}')\n",
    "    os.system(cmd)\n"
   ]
  }
 ],
 "metadata": {
  "kernelspec": {
   "display_name": "Python 3",
   "language": "python",
   "name": "python3"
  },
  "language_info": {
   "codemirror_mode": {
    "name": "ipython",
    "version": 3
   },
   "file_extension": ".py",
   "mimetype": "text/x-python",
   "name": "python",
   "nbconvert_exporter": "python",
   "pygments_lexer": "ipython3",
   "version": "3.10.12"
  }
 },
 "nbformat": 4,
 "nbformat_minor": 4
}

with open('kaggle_evaluate_6x6.ipynb', 'w') as f:
    json.dump(nb, f, indent=1)



In [ ]:
%%writefile docx_content.txt

Nested Hierarchical Reasoning Transformer (NHRT)
Methodology and Results Report
Base model: Qwen2.5-0.5B  ·  Datasets: GSM8K, SVAMP, ARC-Challenge
Covers Phase 1 (architecture ablation), Phase 2 (bug fixes and extensions),
and Phase 3 (statistical validation and preference optimisation)


1. Methodology
1.1 Experimental Setup
All experiments use a single frozen base model and a fixed set of default hyperparameters unless a section explicitly states an override (e.g. the scaled-training or seed-variance runs).
Parameter
Value
Base model
Qwen/Qwen2.5-0.5B (494.0M parameters, 24 decoder layers, bfloat16)
Hidden size / attention heads
896 hidden · 14 query heads · 2 KV heads (GQA 7:1)
Default insertion layer (TARGET_LAYER)
12 (Phase 1 ablation winner)
Default training set size (N_TRAIN)
200 GSM8K training examples
Default evaluation set size (N_EVAL)
100 (Phases 1–2) → 300 from Phase 2 §2.6 onward
Epochs / learning rate
3 epochs · 2e-4, cosine schedule with 10% warmup
Optimizer / grad clipping
AdamW, weight decay 0.01 · gradient-norm clip = 1.0
Precision
bfloat16 throughout (torch_dtype="auto") (bfloat16 avoids the loss-scaling issues float16 introduced in an earlier debugging attempt)
Datasets
GSM8K (train/test), SVAMP (test, zero-shot transfer), ARC-Challenge (test)
Table 1.1 — Fixed experimental configuration used across all phases.
1.2 Reasoning Module Architectures
Four trainable reasoning-module designs were implemented and compared, all inserted via the same gated-residual mechanism (Section 1.3).
Module
Mechanism
Trainable params (single layer)
HRB (HierarchicalReasoningBlock)
Gated blend of a shallow “fast” path and a 2-layer “deep” path, blended by a learned sigmoid controller
~8.84M
DeepReasoner
Universal-Transformer-style iterative residual loop, fixed steps=3, RMSNorm-stabilised
~1.61M
AdaptiveDeepReasoner (ACT)
Same iterative residual block, but each token learns its own halting probability (Graves 2016); ponder cost added to the loss
~1.61M + halting head
DeepReasoner + LoRA (joint)
DeepReasoner spliced into a layer whose projections are also LoRA-adapted (r=16); both trained jointly
~1.98M combined
Table 1.2 — Reasoning module designs, in order of introduction.
A fifth module, NestedReasoningModule (three parallel linear branches + aggregation), was defined for ablation completeness in Phase 1 but was superseded by HRB before any training run and does not appear in the results.
1.3 Core Mechanism: the Gated NestedDecoderLayer
Every reasoning module above is inserted into a target decoder layer using the same wrapper, shown in Figure 1.1. The base model's attention and MLP sublayers remain completely frozen; only the reasoning module and a single learnable scalar gate (initialised at 0.05, i.e. near-zero) are trained. This means the model is mathematically near-identical to the frozen baseline at initialisation, and training only opens the gate if the reasoning module's output is actually useful for the loss.

Figure 1.1 — Forward pass of NestedDecoderLayer. Gray = frozen; green = trainable reasoning module; orange = trainable gate.
1.4 Overall Research Workflow
Figure 1.2 maps every experiment across all three phases to the stage of the pipeline it belongs to. This is the master reference for how Sections 2.1–4.13 relate to one another chronologically and methodologically.

Figure 1.2 — End-to-end workflow, Phase 1 through Phase 3. Red-outlined box marks the ARC-Challenge methodology later found to be flawed and subsequently corrected.

1.5 Training Procedure
All architectures are trained with an identical TorchTune-inspired supervised fine-tuning loop, shown in Figure 1.3: only parameters belonging to the reasoning module and the gate have requires_grad=True; the rest of the 494M-parameter base model is frozen throughout.

Figure 1.3 — Per-architecture SFT training loop. The ponder-cost term (dashed path) applies only to AdaptiveDeepReasoner.
1.6 Evaluation Protocol
GSM8K / SVAMP accuracy: greedy generation (do_sample=False, max_new_tokens=64), answer extracted via a regex preferring the “#### N” marker and falling back to the last number in the completion.
Statistical significance: McNemar's exact binomial test on paired per-question correctness between any two configurations, reported at p<0.05.
ARC-Challenge (original method, Phase 2 §2.7): free-form generation (16 new tokens) + regex matching “Answer: X”, later found to produce a floor effect for this non-instruction-tuned 0.5B base model.
ARC-Challenge (corrected method, Phase 3 §2.10): log-likelihood scoring — each answer choice's total token log-probability is computed as a continuation of the question, and the highest-scoring choice is the prediction. This does not require the model to follow an output format and is the standard approach for small base models on multiple-choice tasks.
Seed variance (Phase 3 §2.9): identical architecture and hyperparameters retrained from 3 different random seeds, evaluated at n=300, to distinguish real effects from run-to-run noise.
1.7 DPO Procedure (Phase 3 refinement)
As a final refinement stage, Direct Preference Optimization was applied on top of the already SFT-trained AdaptiveDeepReasoner checkpoint. Since no human preference labels exist for this task, preference pairs are self-generated in a STaR-style rejection-sampling procedure: multiple completions are sampled per training question from the SFT'd model, completions reaching the correct final answer are kept as “chosen” (preferring the shortest among them, reinforcing the efficiency story already told by ACT's average-step count), and incorrect completions are kept as “rejected.” Only the reasoning module and gate remain trainable during DPO, matching the SFT stage's trainable scope; the reference policy is a frozen deep copy of the SFT checkpoint.

Figure 1.4 — Self-generated preference-pair construction and DPO update, Phase 3 §2.12.

2. Results
Results are presented in the same phase order as the methodology (Section 1.4, Figure 1.2). Each subsection states the sample size (n) used, since n=100 estimates were later shown (§2.6) to be materially less stable than n=300 estimates for this effect size.
2.1 Phase 1 — Core Architecture Comparison (n=100 GSM8K/SVAMP test)
Configuration
GSM8K
SVAMP
Δ GSM8K vs baseline
Baseline (frozen)
3.00%
14.00%
—
LoRA, Layer 12 (param-matched)
10.00%
40.00%
+7.00pp
HRB, Layer 12
8.00%
35.00%
+5.00pp
Multi-Layer HRB [6, 12, 18]
11.00%
34.00%
+8.00pp
DeepReasoner, Layer 12
11.00%
33.00%
+8.00pp
Table 2.1 — Phase 1 core ablation. All configurations beat baseline on both metrics; see Table 2.2 for which differences are statistically confirmed.
2.2 Phase 1 — Statistical Significance (McNemar, n=100)
Comparison
Favours
p-value
Result
HRB vs Baseline
HRB
0.1797
not significant
HRB vs LoRA
LoRA
0.7539
not significant
Multi-Layer vs Single-Layer HRB
Multi-Layer
0.5488
not significant
DeepReasoner vs HRB
DeepReasoner
0.5078
not significant
Table 2.2 — None of the Phase 1 pairwise comparisons reached significance at n=100 — the motivation for the n=300 re-evaluation in §2.6 and the seed-variance harness in §2.9.
Gate sensitivity sweep (post-training diagnostic, n=20 mini-eval)
Gate value
0.00
0.01
0.03
0.05
0.07
0.10
0.15
0.20
Accuracy
0%
0%
10%
20%
10%
10%
5%
0%
Table 2.3 — Accuracy peaks at gate≈0.05, the value HRB training converged to independently — supporting evidence that the learned gate is meaningful rather than arbitrary.
2.4 Phase 2 — New Architectures (n=100)
Configuration
GSM8K
SVAMP
Gate (Δ from init)
Note
Multi-Layer HRB, init_gate=0.15
12.00%
n/a
+0.0004 (all 3 layers)
Gate movement within noise (not meaningful)
AdaptiveDeepReasoner (ACT)
12.00%
36.00%
+0.0125
Avg. 2.05 / 6 max steps used
DeepReasoner + LoRA (joint, r=16)
9.00%
35.00%
−0.0021 (closed)
Underperforms both standalone components
Scaled joint (1000 samples, 5 epochs)
8.00%
38.00%
—
Scaling the wrong candidate (misconfiguration)
Table 2.4 — AdaptiveDeepReasoner is the best-performing single architecture found; joint training underperforms.
2.5 Phase 2 — Layer Position Sweep (2-epoch screen, n=100)
Layer
4
8
12
16
20
23
Accuracy
8%
12%
11%
8%
6%
4%
Gate Δ
+0.0125
+0.0125
+0.0125
+0.0125
+0.0125
+0.0125
Table 2.5 — Lightweight screen (2 epochs) suggested Layer 8 may exceed the Phase 1 winner (Layer 12); confirmed at full training length in §2.11.
2.6 Phase 2 — Larger Evaluation Set (n=300)
Configuration
GSM8K (n=100)
GSM8K (n=300)
McNemar vs Baseline
Baseline (frozen)
3.00%
6.33%
—
Scaled joint (Phase 2 candidate)
8.00%
11.00%
significant (p<0.05)
Table 2.6 — The baseline estimate itself moved from 3.00% to 6.33% between n=100 and n=300 — direct evidence that n=100 is not a stable estimate for this model/task pair.
2.7 Phase 2 — ARC-Challenge, Original Method (flawed — superseded by §2.10)
Model
ARC accuracy (free-gen + regex)
McNemar vs Baseline
Baseline
2.00%
—
Scaled joint
2.33%
not significant
Table 2.7 — Both scores sit at a measurement floor caused by the evaluation method, not genuine multiple-choice competence (see §1.6). Do not cite this table — use Table 2.10 instead.

2.8 Phase 3 — Significance Tests on Outstanding Comparisons (n=100 unless noted)
Comparison
Favours
p-value
Result
AdaptiveDeepReasoner vs fixed-step DeepReasoner
Adaptive
1.0000
not significant
AdaptiveDeepReasoner vs Baseline
Adaptive
0.0225
significant (p<0.05)
Layer 8 (full run) vs Layer 12 DeepReasoner
Layer 8
0.7744
not significant
AdaptiveDeepReasoner + DPO vs SFT-only
+DPO
0.5078
not significant
ARC-Challenge, log-likelihood: scaled joint vs Baseline
scaled joint
0.5034
not significant
Table 2.8 — Only the Adaptive-vs-Baseline comparison is currently statistically confirmed; the project's best point-estimates (Adaptive vs fixed-step, DPO vs SFT) are directionally positive but not yet significant at current sample sizes.
2.9 Phase 3 — Seed Variance (n=300 per seed; run incomplete)
Architecture
Seed 42
Seed 123
Seed 2024
Status
HRB
10.33%
11.00%
13.00%
Complete — 3/3 seeds
DeepReasoner
13.67%
12.33%
n/a
Incomplete — 2/3 seeds (run truncated)
AdaptiveDeepReasoner
n/a
n/a
n/a
Not started — 0/3 seeds
Baseline (deterministic, no training)
6.33%
6.33%
6.33%
Single value — no seed dependence
Table 2.9 — HRB is confirmed to beat baseline (6.33%) on all 3 seeds tested, though the margin varies (10.33–13.00%). This harness must be completed — DeepReasoner and Adaptive variance are not yet known — before quoting a mean±std for either in a publication.
2.10 Phase 3 — ARC-Challenge, Corrected Method (log-likelihood scoring, n=300)
Model
ARC accuracy (log-likelihood)
ARC accuracy (original, flawed)
McNemar vs Baseline
Baseline
29.67%
2.00%
—
Scaled joint
31.00%
2.33%
not significant
Table 2.10 — Corrected methodology restores ARC-Challenge to a sane accuracy range (~30%, consistent with near-random performance on a 4–5-choice task for a 0.5B non-instruction-tuned model). The honest, current conclusion is: no statistically confirmed generalisation beyond arithmetic reasoning. This supersedes the positive-looking claim implied by Table 2.7.
2.11 Phase 3 — Layer 8 Full-Length Confirmation (n=100, 3 epochs)
Layer
GSM8K
SVAMP
Gate Δ
vs Layer 12 (McNemar)
Layer 8 (DeepReasoner, full run)
13.00%
26.00%
+0.0125
not significant
Layer 12 (DeepReasoner, Phase 1 reference)
11.00%
33.00%
+0.0125
—
Table 2.11 — Layer 8's GSM8K lead over Layer 12 replicated at full training length (13% vs 12% in the light screen), but comes with a SVAMP cost (26% vs 33%) and the GSM8K gap itself is not significant. This is a trade-off, not a confirmed win.
2.12 Phase 3 — DPO on Self-Generated Preference Pairs
Metric
Pre-DPO (SFT only)
Post-DPO
GSM8K accuracy
12.00%
15.00%
SVAMP accuracy
36.00%
39.00%
Gate value
0.0625
0.0625 (unchanged)
Preference pairs used
—
55 (37% yield from 150 candidates)
DPO preference accuracy (epoch 1 → 2)
—
61.82% → 94.55%
McNemar vs pre-DPO
—
p = 0.5078 (not significant)
Table 2.12 — DPO produced the largest raw accuracy gain in the project (+3pp GSM8K), but on only 55 preference pairs with no held-out validation split — the 94.55% epoch-2 preference accuracy is consistent with genuine learning or with overfitting to this small pool, and current data cannot distinguish the two.

2.13 Consolidated Best-Known Results (All Phases)
Configuration
GSM8K
SVAMP
ARC (corrected)
vs Baseline
Baseline (frozen)
3.00% / 6.33%*
14.00%
29.67%
—
LoRA, Layer 12
10.00%
40.00%
n/a
not significant
HRB, Layer 12 (mean of 3 seeds)
~11.4%
35.00%
n/a
significant (p<0.05)
DeepReasoner, Layer 12
11.00%
33.00%
n/a
not significant
DeepReasoner, Layer 8 (full run)
13.00%
26.00%
n/a
not significant
AdaptiveDeepReasoner (ACT)
12.00%
36.00%
n/a
significant (p<0.05)
DeepReasoner + LoRA (joint)
9.00%
35.00%
31.00%
not significant
AdaptiveDeepReasoner + DPO (best overall)
15.00%
39.00%
n/a
not significant
Table 2.13 — *Baseline GSM8K shown at both n=100 and n=300 (see §2.6). AdaptiveDeepReasoner + DPO is the highest raw score reached; AdaptiveDeepReasoner alone is the highest score with confirmed significance over baseline.


In [ ]:
%%writefile download (2).txt
11.0s 1 0.00s - Debugger warning: It seems that frozen modules are being used, which may
11.0s 2 0.00s - make the debugger miss breakpoints. Please pass -Xfrozen_modules=off
11.0s 3 0.00s - to python to disable frozen modules.
11.0s 4 0.00s - Note: Debugging will proceed. Set PYDEVD_DISABLE_FILE_VALIDATION=1 to disable this validation.
11.7s 5 0.00s - Debugger warning: It seems that frozen modules are being used, which may
11.7s 6 0.00s - make the debugger miss breakpoints. Please pass -Xfrozen_modules=off
11.7s 7 0.00s - to python to disable frozen modules.
11.7s 8 0.00s - Note: Debugging will proceed. Set PYDEVD_DISABLE_FILE_VALIDATION=1 to disable this validation.
13.2s 9 Cloning into 'HRM'...
14.1s 10 remote: Enumerating objects: 68, done.[K
14.1s 11 remote: Counting objects:   3% (1/32)[K
remote: Counting objects:   6% (2/32)[K
remote: Counting objects:   9% (3/32)[K
remote: Counting objects:  12% (4/32)[K
remote: Counting objects:  15% (5/32)[K
remote: Counting objects:  18% (6/32)[K
remote: Counting objects:  21% (7/32)[K
remote: Counting objects:  25% (8/32)[K
remote: Counting objects:  28% (9/32)[K
remote: Counting objects:  31% (10/32)[K
remote: Counting objects:  34% (11/32)[K
remote: Counting objects:  37% (12/32)[K
remote: Counting objects:  40% (13/32)[K
remote: Counting objects:  43% (14/32)[K
remote: Counting objects:  46% (15/32)[K
remote: Counting objects:  50% (16/32)[K
remote: Counting objects:  53% (17/32)[K
remote: Counting objects:  56% (18/32)[K
remote: Counting objects:  59% (19/32)[K
remote: Counting objects:  62% (20/32)[K
remote: Counting objects:  65% (21/32)[K
remote: Counting objects:  68% (22/32)[K
remote: Counting objects:  71% (23/32)[K
remote: Counting objects:  75% (24/32)[K
remote: Counting objects:  78% (25/32)[K
remote: Counting objects:  81% (26/32)[K
remote: Counting objects:  84% (27/32)[K
remote: Counting objects:  87% (28/32)[K
remote: Counting objects:  90% (29/32)[K
remote: Counting objects:  93% (30/32)[K
remote: Counting objects:  96% (31/32)[K
remote: Counting objects: 100% (32/32)[K
remote: Counting objects: 100% (32/32), done.[K
14.1s 12 remote: Compressing objects:   4% (1/22)[K
remote: Compressing objects:   9% (2/22)[K
remote: Compressing objects:  13% (3/22)[K
remote: Compressing objects:  18% (4/22)[K
remote: Compressing objects:  22% (5/22)[K
remote: Compressing objects:  27% (6/22)[K
remote: Compressing objects:  31% (7/22)[K
remote: Compressing objects:  36% (8/22)[K
remote: Compressing objects:  40% (9/22)[K
remote: Compressing objects:  45% (10/22)[K
remote: Compressing objects:  50% (11/22)[K
remote: Compressing objects:  54% (12/22)[K
remote: Compressing objects:  59% (13/22)[K
remote: Compressing objects:  63% (14/22)[K
remote: Compressing objects:  68% (15/22)[K
remote: Compressing objects:  72% (16/22)[K
remote: Compressing objects:  77% (17/22)[K
remote: Compressing objects:  81% (18/22)[K
remote: Compressing objects:  86% (19/22)[K
remote: Compressing objects:  90% (20/22)[K
remote: Compressing objects:  95% (21/22)[K
remote: Compressing objects: 100% (22/22)[K
remote: Compressing objects: 100% (22/22), done.[K
14.2s 13 Receiving objects:   1% (1/68)
Receiving objects:   2% (2/68)
Receiving objects:   4% (3/68)
Receiving objects:   5% (4/68)
Receiving objects:   7% (5/68)
Receiving objects:   8% (6/68)
Receiving objects:  10% (7/68)
Receiving objects:  11% (8/68)
Receiving objects:  13% (9/68)
Receiving objects:  14% (10/68)
Receiving objects:  16% (11/68)
Receiving objects:  17% (12/68)
Receiving objects:  19% (13/68)
Receiving objects:  20% (14/68)
Receiving objects:  22% (15/68)
Receiving objects:  23% (16/68)
Receiving objects:  25% (17/68)
Receiving objects:  26% (18/68)
Receiving objects:  27% (19/68)
Receiving objects:  29% (20/68)
Receiving objects:  30% (21/68)
Receiving objects:  32% (22/68)
remote: Total 68 (delta 13), reused 10 (delta 10), pack-reused 36 (from 2)[K
14.2s 14 Receiving objects:  33% (23/68)
Receiving objects:  35% (24/68)
Receiving objects:  36% (25/68)
Receiving objects:  38% (26/68)
Receiving objects:  39% (27/68)
Receiving objects:  41% (28/68)
Receiving objects:  42% (29/68)
Receiving objects:  44% (30/68)
Receiving objects:  45% (31/68)
Receiving objects:  47% (32/68)
Receiving objects:  48% (33/68)
Receiving objects:  50% (34/68)
Receiving objects:  51% (35/68)
Receiving objects:  52% (36/68)
Receiving objects:  54% (37/68)
Receiving objects:  55% (38/68)
Receiving objects:  57% (39/68)
Receiving objects:  58% (40/68)
Receiving objects:  60% (41/68)
Receiving objects:  61% (42/68)
Receiving objects:  63% (43/68)
Receiving objects:  64% (44/68)
Receiving objects:  66% (45/68)
Receiving objects:  67% (46/68)
Receiving objects:  69% (47/68)
Receiving objects:  70% (48/68)
Receiving objects:  72% (49/68)
Receiving objects:  73% (50/68)
Receiving objects:  75% (51/68)
Receiving objects:  76% (52/68)
Receiving objects:  77% (53/68)
Receiving objects:  79% (54/68)
Receiving objects:  80% (55/68)
Receiving objects:  82% (56/68)
Receiving objects:  83% (57/68)
Receiving objects:  85% (58/68)
Receiving objects:  86% (59/68)
Receiving objects:  88% (60/68)
Receiving objects:  89% (61/68)
Receiving objects:  91% (62/68)
Receiving objects:  92% (63/68)
Receiving objects:  94% (64/68)
Receiving objects:  95% (65/68)
Receiving objects:  97% (66/68)
Receiving objects:  98% (67/68)
Receiving objects: 100% (68/68)
Receiving objects: 100% (68/68), 141.52 KiB | 1.96 MiB/s, done.
14.2s 15 Resolving deltas:   0% (0/19)
Resolving deltas:   5% (1/19)
Resolving deltas:  10% (2/19)
Resolving deltas:  15% (3/19)
Resolving deltas:  21% (4/19)
Resolving deltas:  26% (5/19)
Resolving deltas:  31% (6/19)
Resolving deltas:  36% (7/19)
Resolving deltas:  42% (8/19)
Resolving deltas:  47% (9/19)
Resolving deltas:  52% (10/19)
Resolving deltas:  57% (11/19)
Resolving deltas:  63% (12/19)
Resolving deltas:  68% (13/19)
Resolving deltas:  73% (14/19)
Resolving deltas:  78% (15/19)
Resolving deltas:  84% (16/19)
Resolving deltas:  89% (17/19)
Resolving deltas:  94% (18/19)
Resolving deltas: 100% (19/19)
Resolving deltas: 100% (19/19), done.
14.3s 16 /kaggle/working/HRM
17.7s 17 Requirement already satisfied: peft in /usr/local/lib/python3.12/dist-packages (0.19.1)
17.7s 18 Requirement already satisfied: huggingface_hub in /usr/local/lib/python3.12/dist-packages (1.11.0)
17.7s 19 Requirement already satisfied: wandb in /usr/local/lib/python3.12/dist-packages (0.26.1)
17.7s 20 Requirement already satisfied: torch in /usr/local/lib/python3.12/dist-packages (from -r requirements.txt (line 1)) (2.10.0+cu128)
17.7s 21 Requirement already satisfied: einops in /usr/local/lib/python3.12/dist-packages (from -r requirements.txt (line 2)) (0.8.2)
17.7s 22 Requirement already satisfied: tqdm in /usr/local/lib/python3.12/dist-packages (from -r requirements.txt (line 3)) (4.67.3)
17.8s 23 Collecting coolname (from -r requirements.txt (line 4))
17.9s 24 Downloading coolname-5.0.0-py3-none-any.whl.metadata (4.9 kB)
17.9s 25 Requirement already satisfied: pydantic in /usr/local/lib/python3.12/dist-packages (from -r requirements.txt (line 5)) (2.12.3)
17.9s 26 Collecting argdantic (from -r requirements.txt (line 6))
18.0s 27 Downloading argdantic-1.3.3-py2.py3-none-any.whl.metadata (7.2 kB)
18.0s 28 Requirement already satisfied: omegaconf in /usr/local/lib/python3.12/dist-packages (from -r requirements.txt (line 8)) (2.3.0)
18.0s 29 Collecting hydra-core (from -r requirements.txt (line 9))
18.0s 30 Downloading hydra_core-1.3.4-py3-none-any.whl.metadata (5.7 kB)
18.1s 31 Requirement already satisfied: numpy>=1.17 in /usr/local/lib/python3.12/dist-packages (from peft) (2.0.2)
18.1s 32 Requirement already satisfied: packaging>=20.0 in /usr/local/lib/python3.12/dist-packages (from peft) (26.1)
18.1s 33 Requirement already satisfied: psutil in /usr/local/lib/python3.12/dist-packages (from peft) (5.9.5)
18.1s 34 Requirement already satisfied: pyyaml in /usr/local/lib/python3.12/dist-packages (from peft) (6.0.3)
18.1s 35 Requirement already satisfied: transformers in /usr/local/lib/python3.12/dist-packages (from peft) (5.0.0)
18.1s 36 Requirement already satisfied: accelerate>=0.21.0 in /usr/local/lib/python3.12/dist-packages (from peft) (1.13.0)
18.1s 37 Requirement already satisfied: safetensors in /usr/local/lib/python3.12/dist-packages (from peft) (0.7.0)
18.1s 38 Requirement already satisfied: filelock>=3.10.0 in /usr/local/lib/python3.12/dist-packages (from huggingface_hub) (3.29.0)
18.1s 39 Requirement already satisfied: fsspec>=2023.5.0 in /usr/local/lib/python3.12/dist-packages (from huggingface_hub) (2025.3.0)
18.1s 40 Requirement already satisfied: hf-xet<2.0.0,>=1.4.3 in /usr/local/lib/python3.12/dist-packages (from huggingface_hub) (1.4.3)
18.1s 41 Requirement already satisfied: httpx<1,>=0.23.0 in /usr/local/lib/python3.12/dist-packages (from huggingface_hub) (0.28.1)
18.1s 42 Requirement already satisfied: typer in /usr/local/lib/python3.12/dist-packages (from huggingface_hub) (0.24.2)
18.1s 43 Requirement already satisfied: typing-extensions>=4.1.0 in /usr/local/lib/python3.12/dist-packages (from huggingface_hub) (4.15.0)
18.1s 44 Requirement already satisfied: click>=8.0.1 in /usr/local/lib/python3.12/dist-packages (from wandb) (8.3.3)
18.1s 45 Requirement already satisfied: gitpython!=3.1.29,>=1.0.0 in /usr/local/lib/python3.12/dist-packages (from wandb) (3.1.47)
18.1s 46 Requirement already satisfied: platformdirs in /usr/local/lib/python3.12/dist-packages (from wandb) (4.9.6)
18.1s 47 Requirement already satisfied: protobuf!=5.28.0,!=5.29.0,<8,>4.21.0 in /usr/local/lib/python3.12/dist-packages (from wandb) (5.29.5)
18.1s 48 Requirement already satisfied: requests<3,>=2.0.0 in /usr/local/lib/python3.12/dist-packages (from wandb) (2.32.4)
18.1s 49 Requirement already satisfied: sentry-sdk>=2.0.0 in /usr/local/lib/python3.12/dist-packages (from wandb) (2.58.0)
18.1s 50 Requirement already satisfied: setuptools in /usr/local/lib/python3.12/dist-packages (from torch->-r requirements.txt (line 1)) (81.0.0)
18.1s 51 Requirement already satisfied: sympy>=1.13.3 in /usr/local/lib/python3.12/dist-packages (from torch->-r requirements.txt (line 1)) (1.14.0)
18.1s 52 Requirement already satisfied: networkx>=2.5.1 in /usr/local/lib/python3.12/dist-packages (from torch->-r requirements.txt (line 1)) (3.6.1)
18.1s 53 Requirement already satisfied: jinja2 in /usr/local/lib/python3.12/dist-packages (from torch->-r requirements.txt (line 1)) (3.1.6)
18.1s 54 Requirement already satisfied: cuda-bindings==12.9.4 in /usr/local/lib/python3.12/dist-packages (from torch->-r requirements.txt (line 1)) (12.9.4)
18.1s 55 Requirement already satisfied: nvidia-cuda-nvrtc-cu12==12.8.93 in /usr/local/lib/python3.12/dist-packages (from torch->-r requirements.txt (line 1)) (12.8.93)
18.1s 56 Requirement already satisfied: nvidia-cuda-runtime-cu12==12.8.90 in /usr/local/lib/python3.12/dist-packages (from torch->-r requirements.txt (line 1)) (12.8.90)
18.1s 57 Requirement already satisfied: nvidia-cuda-cupti-cu12==12.8.90 in /usr/local/lib/python3.12/dist-packages (from torch->-r requirements.txt (line 1)) (12.8.90)
18.1s 58 Requirement already satisfied: nvidia-cudnn-cu12==9.10.2.21 in /usr/local/lib/python3.12/dist-packages (from torch->-r requirements.txt (line 1)) (9.10.2.21)
18.1s 59 Requirement already satisfied: nvidia-cublas-cu12==12.8.4.1 in /usr/local/lib/python3.12/dist-packages (from torch->-r requirements.txt (line 1)) (12.8.4.1)
18.1s 60 Requirement already satisfied: nvidia-cufft-cu12==11.3.3.83 in /usr/local/lib/python3.12/dist-packages (from torch->-r requirements.txt (line 1)) (11.3.3.83)
18.1s 61 Requirement already satisfied: nvidia-curand-cu12==10.3.9.90 in /usr/local/lib/python3.12/dist-packages (from torch->-r requirements.txt (line 1)) (10.3.9.90)
18.1s 62 Requirement already satisfied: nvidia-cusolver-cu12==11.7.3.90 in /usr/local/lib/python3.12/dist-packages (from torch->-r requirements.txt (line 1)) (11.7.3.90)
18.1s 63 Requirement already satisfied: nvidia-cusparse-cu12==12.5.8.93 in /usr/local/lib/python3.12/dist-packages (from torch->-r requirements.txt (line 1)) (12.5.8.93)
18.1s 64 Requirement already satisfied: nvidia-cusparselt-cu12==0.7.1 in /usr/local/lib/python3.12/dist-packages (from torch->-r requirements.txt (line 1)) (0.7.1)
18.1s 65 Requirement already satisfied: nvidia-nccl-cu12==2.27.5 in /usr/local/lib/python3.12/dist-packages (from torch->-r requirements.txt (line 1)) (2.27.5)
18.1s 66 Requirement already satisfied: nvidia-nvshmem-cu12==3.4.5 in /usr/local/lib/python3.12/dist-packages (from torch->-r requirements.txt (line 1)) (3.4.5)
18.1s 67 Requirement already satisfied: nvidia-nvtx-cu12==12.8.90 in /usr/local/lib/python3.12/dist-packages (from torch->-r requirements.txt (line 1)) (12.8.90)
18.1s 68 Requirement already satisfied: nvidia-nvjitlink-cu12==12.8.93 in /usr/local/lib/python3.12/dist-packages (from torch->-r requirements.txt (line 1)) (12.8.93)
18.1s 69 Requirement already satisfied: nvidia-cufile-cu12==1.13.1.3 in /usr/local/lib/python3.12/dist-packages (from torch->-r requirements.txt (line 1)) (1.13.1.3)
18.1s 70 Requirement already satisfied: triton==3.6.0 in /usr/local/lib/python3.12/dist-packages (from torch->-r requirements.txt (line 1)) (3.6.0)
18.2s 71 Requirement already satisfied: cuda-pathfinder~=1.1 in /usr/local/lib/python3.12/dist-packages (from cuda-bindings==12.9.4->torch->-r requirements.txt (line 1)) (1.5.3)
18.2s 72 Requirement already satisfied: annotated-types>=0.6.0 in /usr/local/lib/python3.12/dist-packages (from pydantic->-r requirements.txt (line 5)) (0.7.0)
18.2s 73 Requirement already satisfied: pydantic-core==2.41.4 in /usr/local/lib/python3.12/dist-packages (from pydantic->-r requirements.txt (line 5)) (2.41.4)
18.2s 74 Requirement already satisfied: typing-inspection>=0.4.2 in /usr/local/lib/python3.12/dist-packages (from pydantic->-r requirements.txt (line 5)) (0.4.2)
18.2s 75 Requirement already satisfied: pydantic-settings<3,>=2.4.0 in /usr/local/lib/python3.12/dist-packages (from argdantic->-r requirements.txt (line 6)) (2.14.0)
18.2s 76 Requirement already satisfied: antlr4-python3-runtime==4.9.* in /usr/local/lib/python3.12/dist-packages (from omegaconf->-r requirements.txt (line 8)) (4.9.3)
18.2s 77 Requirement already satisfied: gitdb<5,>=4.0.1 in /usr/local/lib/python3.12/dist-packages (from gitpython!=3.1.29,>=1.0.0->wandb) (4.0.12)
18.2s 78 Requirement already satisfied: anyio in /usr/local/lib/python3.12/dist-packages (from httpx<1,>=0.23.0->huggingface_hub) (4.13.0)
18.2s 79 Requirement already satisfied: certifi in /usr/local/lib/python3.12/dist-packages (from httpx<1,>=0.23.0->huggingface_hub) (2026.4.22)
18.2s 80 Requirement already satisfied: httpcore==1.* in /usr/local/lib/python3.12/dist-packages (from httpx<1,>=0.23.0->huggingface_hub) (1.0.9)
18.2s 81 Requirement already satisfied: idna in /usr/local/lib/python3.12/dist-packages (from httpx<1,>=0.23.0->huggingface_hub) (3.13)
18.2s 82 Requirement already satisfied: h11>=0.16 in /usr/local/lib/python3.12/dist-packages (from httpcore==1.*->httpx<1,>=0.23.0->huggingface_hub) (0.16.0)
18.3s 83 Requirement already satisfied: python-dotenv>=0.21.0 in /usr/local/lib/python3.12/dist-packages (from pydantic-settings<3,>=2.4.0->argdantic->-r requirements.txt (line 6)) (1.2.2)
18.3s 84 Requirement already satisfied: charset_normalizer<4,>=2 in /usr/local/lib/python3.12/dist-packages (from requests<3,>=2.0.0->wandb) (3.4.7)
18.3s 85 Requirement already satisfied: urllib3<3,>=1.21.1 in /usr/local/lib/python3.12/dist-packages (from requests<3,>=2.0.0->wandb) (2.5.0)
18.3s 86 Requirement already satisfied: mpmath<1.4,>=1.1.0 in /usr/local/lib/python3.12/dist-packages (from sympy>=1.13.3->torch->-r requirements.txt (line 1)) (1.3.0)
18.3s 87 Requirement already satisfied: MarkupSafe>=2.0 in /usr/local/lib/python3.12/dist-packages (from jinja2->torch->-r requirements.txt (line 1)) (3.0.3)
18.4s 88 Requirement already satisfied: regex!=2019.12.17 in /usr/local/lib/python3.12/dist-packages (from transformers->peft) (2025.11.3)
18.4s 89 Requirement already satisfied: tokenizers<=0.23.0,>=0.22.0 in /usr/local/lib/python3.12/dist-packages (from transformers->peft) (0.22.2)
18.4s 90 Requirement already satisfied: typer-slim in /usr/local/lib/python3.12/dist-packages (from transformers->peft) (0.24.0)
18.4s 91 Requirement already satisfied: shellingham>=1.3.0 in /usr/local/lib/python3.12/dist-packages (from typer->huggingface_hub) (1.5.4)
18.4s 92 Requirement already satisfied: rich>=12.3.0 in /usr/local/lib/python3.12/dist-packages (from typer->huggingface_hub) (13.9.4)
18.4s 93 Requirement already satisfied: annotated-doc>=0.0.2 in /usr/local/lib/python3.12/dist-packages (from typer->huggingface_hub) (0.0.4)
18.4s 94 Requirement already satisfied: smmap<6,>=3.0.1 in /usr/local/lib/python3.12/dist-packages (from gitdb<5,>=4.0.1->gitpython!=3.1.29,>=1.0.0->wandb) (5.0.3)
18.4s 95 Requirement already satisfied: markdown-it-py>=2.2.0 in /usr/local/lib/python3.12/dist-packages (from rich>=12.3.0->typer->huggingface_hub) (4.0.0)
18.4s 96 Requirement already satisfied: pygments<3.0.0,>=2.13.0 in /usr/local/lib/python3.12/dist-packages (from rich>=12.3.0->typer->huggingface_hub) (2.20.0)
18.4s 97 Requirement already satisfied: mdurl~=0.1 in /usr/local/lib/python3.12/dist-packages (from markdown-it-py>=2.2.0->rich>=12.3.0->typer->huggingface_hub) (0.1.2)
18.4s 98 Downloading coolname-5.0.0-py3-none-any.whl (47 kB)
18.5s 99 [?25l   [90m━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━[0m [32m0.0/47.4 kB[0m [31m?[0m eta [36m-:--:--[0m
[2K   [90m━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━[0m [32m47.4/47.4 kB[0m [31m1.7 MB/s[0m eta [36m0:00:00[0m
18.5s 100 [?25hDownloading argdantic-1.3.3-py2.py3-none-any.whl (26 kB)
18.5s 101 Downloading hydra_core-1.3.4-py3-none-any.whl (155 kB)
18.5s 102 [?25l   [90m━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━[0m [32m0.0/155.5 kB[0m [31m?[0m eta [36m-:--:--[0m
[2K   [90m━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━[0m [32m155.5/155.5 kB[0m [31m6.2 MB/s[0m eta [36m0:00:00[0m
20.6s 103 [?25hInstalling collected packages: coolname, hydra-core, argdantic
20.8s 104 Successfully installed argdantic-1.3.3 coolname-5.0.0 hydra-core-1.3.4
28.3s 105 train.csv:   0%|                                     | 0.00/719M [00:00<?, ?B/s]
train.csv:   0%|                                     | 0.00/719M [00:00<?, ?B/s]
train.csv:   0%|                                     | 0.00/719M [00:00<?, ?B/s]
train.csv:   0%|                                     | 0.00/719M [00:00<?, ?B/s]
train.csv:   0%|                                     | 0.00/719M [00:00<?, ?B/s]
train.csv:   0%|                                     | 0.00/719M [00:01<?, ?B/s]
train.csv:   0%|                                     | 0.00/719M [00:01<?, ?B/s]
train.csv:   0%|                                     | 0.00/719M [00:01<?, ?B/s]
train.csv:   0%|                                     | 0.00/719M [00:01<?, ?B/s]
train.csv:   0%|                                     | 0.00/719M [00:01<?, ?B/s]
train.csv:   0%|                                     | 0.00/719M [00:02<?, ?B/s]
train.csv:   0%|                                     | 0.00/719M [00:02<?, ?B/s]
train.csv:   0%|                                     | 0.00/719M [00:02<?, ?B/s]
train.csv:  19%|█████▌                        | 134M/719M [00:02<00:00, 671MB/s]
train.csv:  37%|███████████▏                  | 268M/719M [00:03<00:01, 422MB/s]
train.csv:  65%|███████████████████▌          | 469M/719M [00:03<00:00, 306MB/s]
train.csv:  93%|███████████████████████████▉  | 671M/719M [00:04<00:00, 443MB/s]
train.csv: 100%|██████████████████████████████| 719M/719M [00:05<00:00, 191MB/s]
train.csv: 100%|██████████████████████████████| 719M/719M [00:05<00:00, 127MB/s]
172.0s 106 0%|                                                  | 0/1000 [00:00<?, ?it/s]
  0%|                                          | 1/1000 [00:00<02:33,  6.51it/s]
  0%|                                          | 2/1000 [00:00<02:16,  7.30it/s]
  0%|▏                                         | 3/1000 [00:00<02:11,  7.60it/s]
  0%|▏                                         | 4/1000 [00:00<02:09,  7.67it/s]
  0%|▏                                         | 5/1000 [00:00<02:09,  7.68it/s]
  1%|▎                                         | 6/1000 [00:00<02:08,  7.73it/s]
  1%|▎                                         | 7/1000 [00:00<02:07,  7.80it/s]
  1%|▎                                         | 8/1000 [00:01<02:07,  7.77it/s]
  1%|▍                                         | 9/1000 [00:01<02:07,  7.75it/s]
  1%|▍                                        | 10/1000 [00:01<02:07,  7.74it/s]
  1%|▍                                        | 11/1000 [00:01<02:08,  7.70it/s]
  1%|▍                                        | 12/1000 [00:01<02:09,  7.65it/s]
  1%|▌                                        | 13/1000 [00:01<02:07,  7.72it/s]
  1%|▌                                        | 14/1000 [00:01<02:06,  7.79it/s]
  2%|▌                                        | 15/1000 [00:01<02:07,  7.74it/s]
  2%|▋                                        | 16/1000 [00:02<02:07,  7.72it/s]
  2%|▋                                        | 17/1000 [00:02<02:08,  7.67it/s]
  2%|▋                                        | 18/1000 [00:02<02:07,  7.70it/s]
  2%|▊                                        | 19/1000 [00:02<02:06,  7.79it/s]
  2%|▊                                        | 20/1000 [00:02<02:05,  7.78it/s]
  2%|▊                                        | 21/1000 [00:02<02:06,  7.77it/s]
  2%|▉                                        | 22/1000 [00:02<02:04,  7.83it/s]
  2%|▉                                        | 23/1000 [00:02<02:04,  7.84it/s]
  2%|▉                                        | 24/1000 [00:03<02:03,  7.90it/s]
  2%|█                                        | 25/1000 [00:03<02:03,  7.87it/s]
  3%|█                                        | 26/1000 [00:03<02:03,  7.89it/s]
  3%|█                                        | 27/1000 [00:03<02:02,  7.93it/s]
  3%|█▏                                       | 28/1000 [00:03<02:02,  7.95it/s]
  3%|█▏                                       | 29/1000 [00:03<02:02,  7.93it/s]
  3%|█▏                                       | 30/1000 [00:03<02:03,  7.88it/s]
  3%|█▎                                       | 31/1000 [00:03<02:03,  7.87it/s]
  3%|█▎                                       | 32/1000 [00:04<02:02,  7.89it/s]
  3%|█▎                                       | 33/1000 [00:04<02:02,  7.91it/s]
  3%|█▍                                       | 34/1000 [00:04<02:02,  7.91it/s]
  4%|█▍                                       | 35/1000 [00:04<02:01,  7.96it/s]
  4%|█▍                                       | 36/1000 [00:04<02:01,  7.92it/s]
  4%|█▌                                       | 37/1000 [00:04<02:01,  7.94it/s]
  4%|█▌                                       | 38/1000 [00:04<02:00,  7.96it/s]
  4%|█▌                                       | 39/1000 [00:04<02:01,  7.92it/s]
  4%|█▋                                       | 40/1000 [00:05<02:00,  7.98it/s]
  4%|█▋                                       | 41/1000 [00:05<02:00,  7.97it/s]
  4%|█▋                                       | 42/1000 [00:05<02:01,  7.89it/s]
  4%|█▊                                       | 43/1000 [00:05<02:00,  7.94it/s]
  4%|█▊                                       | 44/1000 [00:05<02:00,  7.93it/s]
  4%|█▊                                       | 45/1000 [00:05<02:00,  7.90it/s]
  5%|█▉                                       | 46/1000 [00:05<02:00,  7.93it/s]
  5%|█▉                                       | 47/1000 [00:06<02:00,  7.92it/s]
  5%|█▉                                       | 48/1000 [00:06<01:59,  7.96it/s]
  5%|██                                       | 49/1000 [00:06<01:59,  7.96it/s]
  5%|██                                       | 50/1000 [00:06<02:01,  7.84it/s]
  5%|██                                       | 51/1000 [00:06<02:00,  7.91it/s]
  5%|██▏                                      | 52/1000 [00:06<01:59,  7.93it/s]
  5%|██▏                                      | 53/1000 [00:06<01:59,  7.94it/s]
  5%|██▏                                      | 54/1000 [00:06<01:58,  7.96it/s]
  6%|██▎                                      | 55/1000 [00:07<01:58,  7.95it/s]
  6%|██▎                                      | 56/1000 [00:07<01:58,  7.95it/s]
  6%|██▎                                      | 57/1000 [00:07<01:58,  7.93it/s]
  6%|██▍                                      | 58/1000 [00:07<01:59,  7.91it/s]
  6%|██▍                                      | 59/1000 [00:07<01:58,  7.93it/s]
  6%|██▍                                      | 60/1000 [00:07<01:58,  7.92it/s]
  6%|██▌                                      | 61/1000 [00:07<01:58,  7.95it/s]
  6%|██▌                                      | 62/1000 [00:07<01:57,  8.00it/s]
  6%|██▌                                      | 63/1000 [00:08<01:57,  7.95it/s]
  6%|██▌                                      | 64/1000 [00:08<01:58,  7.88it/s]
  6%|██▋                                      | 65/1000 [00:08<01:58,  7.87it/s]
  7%|██▋                                      | 66/1000 [00:08<02:00,  7.77it/s]
  7%|██▋                                      | 67/1000 [00:08<01:58,  7.84it/s]
  7%|██▊                                      | 68/1000 [00:08<01:58,  7.87it/s]
  7%|██▊                                      | 69/1000 [00:08<01:57,  7.92it/s]
  7%|██▊                                      | 70/1000 [00:08<01:56,  7.97it/s]
  7%|██▉                                      | 71/1000 [00:09<01:56,  7.95it/s]
  7%|██▉                                      | 72/1000 [00:09<01:56,  7.95it/s]
  7%|██▉                                      | 73/1000 [00:09<01:56,  7.95it/s]
  7%|███                                      | 74/1000 [00:09<01:58,  7.84it/s]
  8%|███                                      | 75/1000 [00:09<01:57,  7.88it/s]
  8%|███                                      | 76/1000 [00:09<01:57,  7.90it/s]
  8%|███▏                                     | 77/1000 [00:09<01:57,  7.87it/s]
  8%|███▏                                     | 78/1000 [00:09<01:57,  7.87it/s]
  8%|███▏                                     | 79/1000 [00:10<01:59,  7.68it/s]
  8%|███▎                                     | 80/1000 [00:10<02:04,  7.40it/s]
  8%|███▎                                     | 81/1000 [00:10<02:01,  7.57it/s]
  8%|███▎                                     | 82/1000 [00:10<01:59,  7.67it/s]
  8%|███▍                                     | 83/1000 [00:10<01:58,  7.71it/s]
  8%|███▍                                     | 84/1000 [00:10<01:58,  7.70it/s]
  8%|███▍                                     | 85/1000 [00:10<01:57,  7.79it/s]
  9%|███▌                                     | 86/1000 [00:10<01:56,  7.86it/s]
  9%|███▌                                     | 87/1000 [00:11<01:56,  7.87it/s]
  9%|███▌                                     | 88/1000 [00:11<01:55,  7.89it/s]
  9%|███▋                                     | 89/1000 [00:11<01:55,  7.87it/s]
  9%|███▋                                     | 90/1000 [00:11<01:55,  7.87it/s]
  9%|███▋                                     | 91/1000 [00:11<01:54,  7.93it/s]
  9%|███▊                                     | 92/1000 [00:11<01:54,  7.95it/s]
  9%|███▊                                     | 93/1000 [00:11<01:53,  7.98it/s]
  9%|███▊                                     | 94/1000 [00:11<01:53,  7.99it/s]
 10%|███▉                                     | 95/1000 [00:12<01:53,  7.97it/s]
 10%|███▉                                     | 96/1000 [00:12<01:53,  7.98it/s]
 10%|███▉                                     | 97/1000 [00:12<01:53,  7.96it/s]
 10%|████                                     | 98/1000 [00:12<01:53,  7.95it/s]
 10%|████                                     | 99/1000 [00:12<01:53,  7.96it/s]
 10%|████                                    | 100/1000 [00:12<01:54,  7.85it/s]
 10%|████                                    | 101/1000 [00:12<01:54,  7.82it/s]
 10%|████                                    | 102/1000 [00:13<02:03,  7.27it/s]
 10%|████                                    | 103/1000 [00:13<02:00,  7.45it/s]
 10%|████▏                                   | 104/1000 [00:13<01:58,  7.58it/s]
 10%|████▏                                   | 105/1000 [00:13<01:56,  7.67it/s]
 11%|████▏                                   | 106/1000 [00:13<01:54,  7.79it/s]
 11%|████▎                                   | 107/1000 [00:13<01:54,  7.83it/s]
 11%|████▎                                   | 108/1000 [00:13<01:53,  7.87it/s]
 11%|████▎                                   | 109/1000 [00:13<01:52,  7.94it/s]
 11%|████▍                                   | 110/1000 [00:14<01:52,  7.91it/s]
 11%|████▍                                   | 111/1000 [00:14<01:52,  7.90it/s]
 11%|████▍                                   | 112/1000 [00:14<01:52,  7.90it/s]
 11%|████▌                                   | 113/1000 [00:14<01:51,  7.93it/s]
 11%|████▌                                   | 114/1000 [00:14<01:51,  7.95it/s]
 12%|████▌                                   | 115/1000 [00:14<01:53,  7.83it/s]
 12%|████▋                                   | 116/1000 [00:14<01:52,  7.83it/s]
 12%|████▋                                   | 117/1000 [00:14<01:51,  7.88it/s]
 12%|████▋                                   | 118/1000 [00:15<01:51,  7.91it/s]
 12%|████▊                                   | 119/1000 [00:15<01:51,  7.91it/s]
 12%|████▊                                   | 120/1000 [00:15<01:52,  7.85it/s]
 12%|████▊                                   | 121/1000 [00:15<01:52,  7.79it/s]
 12%|████▉                                   | 122/1000 [00:15<01:52,  7.80it/s]
 12%|████▉                                   | 123/1000 [00:15<01:52,  7.79it/s]
 12%|████▉                                   | 124/1000 [00:15<01:52,  7.81it/s]
 12%|█████                                   | 125/1000 [00:15<01:51,  7.87it/s]
 13%|█████                                   | 126/1000 [00:16<01:50,  7.89it/s]
 13%|█████                                   | 127/1000 [00:16<01:50,  7.88it/s]
 13%|█████                                   | 128/1000 [00:16<01:51,  7.83it/s]
 13%|█████▏                                  | 129/1000 [00:16<01:51,  7.82it/s]
 13%|█████▏                                  | 130/1000 [00:16<01:50,  7.85it/s]
 13%|█████▏                                  | 131/1000 [00:16<01:51,  7.78it/s]
 13%|█████▎                                  | 132/1000 [00:16<01:51,  7.81it/s]
 13%|█████▎                                  | 133/1000 [00:16<01:50,  7.84it/s]
 13%|█████▎                                  | 134/1000 [00:17<01:50,  7.85it/s]
 14%|█████▍                                  | 135/1000 [00:17<01:51,  7.77it/s]
 14%|█████▍                                  | 136/1000 [00:17<01:51,  7.75it/s]
 14%|█████▍                                  | 137/1000 [00:17<01:50,  7.78it/s]
 14%|█████▌                                  | 138/1000 [00:17<01:49,  7.86it/s]
 14%|█████▌                                  | 139/1000 [00:17<01:49,  7.84it/s]
 14%|█████▌                                  | 140/1000 [00:17<01:49,  7.86it/s]
 14%|█████▋                                  | 141/1000 [00:17<01:48,  7.91it/s]
 14%|█████▋                                  | 142/1000 [00:18<01:48,  7.92it/s]
 14%|█████▋                                  | 143/1000 [00:18<01:48,  7.92it/s]
 14%|█████▊                                  | 144/1000 [00:18<01:50,  7.78it/s]
 14%|█████▊                                  | 145/1000 [00:18<01:49,  7.78it/s]
 15%|█████▊                                  | 146/1000 [00:18<01:51,  7.67it/s]
 15%|█████▉                                  | 147/1000 [00:18<01:52,  7.58it/s]
 15%|█████▉                                  | 148/1000 [00:18<01:52,  7.57it/s]
 15%|█████▉                                  | 149/1000 [00:19<01:50,  7.67it/s]
 15%|██████                                  | 150/1000 [00:19<01:49,  7.77it/s]
 15%|██████                                  | 151/1000 [00:19<01:49,  7.78it/s]
 15%|██████                                  | 152/1000 [00:19<01:47,  7.86it/s]
 15%|██████                                  | 153/1000 [00:19<01:47,  7.90it/s]
 15%|██████▏                                 | 154/1000 [00:19<01:46,  7.93it/s]
 16%|██████▏                                 | 155/1000 [00:19<01:47,  7.86it/s]
 16%|██████▏                                 | 156/1000 [00:19<01:48,  7.76it/s]
 16%|██████▎                                 | 157/1000 [00:20<01:49,  7.72it/s]
 16%|██████▎                                 | 158/1000 [00:20<01:54,  7.34it/s]
 16%|██████▎                                 | 159/1000 [00:20<01:51,  7.52it/s]
 16%|██████▍                                 | 160/1000 [00:20<01:49,  7.64it/s]
 16%|██████▍                                 | 161/1000 [00:20<01:48,  7.75it/s]
 16%|██████▍                                 | 162/1000 [00:20<01:49,  7.65it/s]
 16%|██████▌                                 | 163/1000 [00:20<01:48,  7.69it/s]
 16%|██████▌                                 | 164/1000 [00:20<01:47,  7.79it/s]
 16%|██████▌                                 | 165/1000 [00:21<01:46,  7.83it/s]
 17%|██████▋                                 | 166/1000 [00:21<01:46,  7.83it/s]
 17%|██████▋                                 | 167/1000 [00:21<01:46,  7.84it/s]
 17%|██████▋                                 | 168/1000 [00:21<01:45,  7.86it/s]
 17%|██████▊                                 | 169/1000 [00:21<01:44,  7.94it/s]
 17%|██████▊                                 | 170/1000 [00:21<01:45,  7.90it/s]
 17%|██████▊                                 | 171/1000 [00:21<01:44,  7.90it/s]
 17%|██████▉                                 | 172/1000 [00:21<01:44,  7.91it/s]
 17%|██████▉                                 | 173/1000 [00:22<01:44,  7.92it/s]
 17%|██████▉                                 | 174/1000 [00:22<01:45,  7.84it/s]
 18%|███████                                 | 175/1000 [00:22<01:46,  7.76it/s]
 18%|███████                                 | 176/1000 [00:22<01:47,  7.67it/s]
 18%|███████                                 | 177/1000 [00:22<01:46,  7.74it/s]
 18%|███████                                 | 178/1000 [00:22<01:49,  7.49it/s]
 18%|███████▏                                | 179/1000 [00:22<01:48,  7.55it/s]
 18%|███████▏                                | 180/1000 [00:23<01:46,  7.68it/s]
 18%|███████▏                                | 181/1000 [00:23<01:46,  7.70it/s]
 18%|███████▎                                | 182/1000 [00:23<01:44,  7.81it/s]
 18%|███████▎                                | 183/1000 [00:23<01:43,  7.87it/s]
 18%|███████▎                                | 184/1000 [00:23<01:43,  7.90it/s]
 18%|███████▍                                | 185/1000 [00:23<01:43,  7.91it/s]
 19%|███████▍                                | 186/1000 [00:23<01:42,  7.92it/s]
 19%|███████▍                                | 187/1000 [00:23<01:42,  7.94it/s]
 19%|███████▌                                | 188/1000 [00:24<01:42,  7.94it/s]
 19%|███████▌                                | 189/1000 [00:24<01:42,  7.89it/s]
 19%|███████▌                                | 190/1000 [00:24<01:42,  7.91it/s]
 19%|███████▋                                | 191/1000 [00:24<01:42,  7.88it/s]
 19%|███████▋                                | 192/1000 [00:24<01:42,  7.86it/s]
 19%|███████▋                                | 193/1000 [00:24<01:42,  7.88it/s]
 19%|███████▊                                | 194/1000 [00:24<01:42,  7.90it/s]
 20%|███████▊                                | 195/1000 [00:24<01:41,  7.92it/s]
 20%|███████▊                                | 196/1000 [00:25<01:41,  7.89it/s]
 20%|███████▉                                | 197/1000 [00:25<01:40,  7.95it/s]
 20%|███████▉                                | 198/1000 [00:25<01:41,  7.93it/s]
 20%|███████▉                                | 199/1000 [00:25<01:40,  7.94it/s]
 20%|████████                                | 200/1000 [00:25<01:41,  7.85it/s]
 20%|████████                                | 201/1000 [00:25<01:41,  7.88it/s]
 20%|████████                                | 202/1000 [00:25<01:41,  7.90it/s]
 20%|████████                                | 203/1000 [00:25<01:41,  7.85it/s]
 20%|████████▏                               | 204/1000 [00:26<01:40,  7.89it/s]
 20%|████████▏                               | 205/1000 [00:26<01:40,  7.92it/s]
 21%|████████▏                               | 206/1000 [00:26<01:41,  7.84it/s]
 21%|████████▎                               | 207/1000 [00:26<01:41,  7.83it/s]
 21%|████████▎                               | 208/1000 [00:26<01:41,  7.83it/s]
 21%|████████▎                               | 209/1000 [00:26<01:40,  7.83it/s]
 21%|████████▍                               | 210/1000 [00:26<01:40,  7.83it/s]
 21%|████████▍                               | 211/1000 [00:26<01:40,  7.85it/s]
 21%|████████▍                               | 212/1000 [00:27<01:40,  7.87it/s]
 21%|████████▌                               | 213/1000 [00:27<01:39,  7.90it/s]
 21%|████████▌                               | 214/1000 [00:27<01:39,  7.93it/s]
 22%|████████▌                               | 215/1000 [00:27<01:39,  7.93it/s]
 22%|████████▋                               | 216/1000 [00:27<01:38,  7.96it/s]
 22%|████████▋                               | 217/1000 [00:27<01:38,  7.95it/s]
 22%|████████▋                               | 218/1000 [00:27<01:38,  7.95it/s]
 22%|████████▊                               | 219/1000 [00:27<01:38,  7.95it/s]
 22%|████████▊                               | 220/1000 [00:28<01:38,  7.94it/s]
 22%|████████▊                               | 221/1000 [00:28<01:37,  8.00it/s]
 22%|████████▉                               | 222/1000 [00:28<01:36,  8.02it/s]
 22%|████████▉                               | 223/1000 [00:28<01:36,  8.05it/s]
 22%|████████▉                               | 224/1000 [00:28<01:36,  8.04it/s]
 22%|█████████                               | 225/1000 [00:28<01:36,  7.99it/s]
 23%|█████████                               | 226/1000 [00:28<01:36,  8.00it/s]
 23%|█████████                               | 227/1000 [00:28<01:36,  7.99it/s]
 23%|█████████                               | 228/1000 [00:29<01:36,  7.99it/s]
 23%|█████████▏                              | 229/1000 [00:29<01:36,  8.00it/s]
 23%|█████████▏                              | 230/1000 [00:29<01:36,  7.96it/s]
 23%|█████████▏                              | 231/1000 [00:29<01:36,  7.94it/s]
 23%|█████████▎                              | 232/1000 [00:29<01:37,  7.90it/s]
 23%|█████████▎                              | 233/1000 [00:29<01:36,  7.94it/s]
 23%|█████████▎                              | 234/1000 [00:29<01:37,  7.83it/s]
 24%|█████████▍                              | 235/1000 [00:29<01:37,  7.83it/s]
 24%|█████████▍                              | 236/1000 [00:30<01:44,  7.29it/s]
 24%|█████████▍                              | 237/1000 [00:30<01:41,  7.48it/s]
 24%|█████████▌                              | 238/1000 [00:30<01:39,  7.64it/s]
 24%|█████████▌                              | 239/1000 [00:30<01:38,  7.74it/s]
 24%|█████████▌                              | 240/1000 [00:30<01:37,  7.80it/s]
 24%|█████████▋                              | 241/1000 [00:30<01:37,  7.76it/s]
 24%|█████████▋                              | 242/1000 [00:30<01:37,  7.80it/s]
 24%|█████████▋                              | 243/1000 [00:31<01:36,  7.86it/s]
 24%|█████████▊                              | 244/1000 [00:31<01:36,  7.86it/s]
 24%|█████████▊                              | 245/1000 [00:31<01:35,  7.93it/s]
 25%|█████████▊                              | 246/1000 [00:31<01:34,  7.95it/s]
 25%|█████████▉                              | 247/1000 [00:31<01:34,  7.94it/s]
 25%|█████████▉                              | 248/1000 [00:31<01:34,  7.92it/s]
 25%|█████████▉                              | 249/1000 [00:31<01:34,  7.93it/s]
 25%|██████████                              | 250/1000 [00:31<01:34,  7.93it/s]
 25%|██████████                              | 251/1000 [00:32<01:34,  7.96it/s]
 25%|██████████                              | 252/1000 [00:32<01:34,  7.96it/s]
 25%|██████████                              | 253/1000 [00:32<01:33,  7.95it/s]
 25%|██████████▏                             | 254/1000 [00:32<01:33,  7.99it/s]
 26%|██████████▏                             | 255/1000 [00:32<01:33,  7.99it/s]
 26%|██████████▏                             | 256/1000 [00:32<01:33,  8.00it/s]
 26%|██████████▎                             | 257/1000 [00:32<01:33,  7.92it/s]
 26%|██████████▎                             | 258/1000 [00:32<01:33,  7.91it/s]
 26%|██████████▎                             | 259/1000 [00:33<01:33,  7.92it/s]
 26%|██████████▍                             | 260/1000 [00:33<01:33,  7.94it/s]
 26%|██████████▍                             | 261/1000 [00:33<01:32,  7.98it/s]
 26%|██████████▍                             | 262/1000 [00:33<01:32,  8.01it/s]
 26%|██████████▌                             | 263/1000 [00:33<01:31,  8.01it/s]
 26%|██████████▌                             | 264/1000 [00:33<01:31,  8.02it/s]
 26%|██████████▌                             | 265/1000 [00:33<01:31,  8.06it/s]
 27%|██████████▋                             | 266/1000 [00:33<01:31,  8.05it/s]
 27%|██████████▋                             | 267/1000 [00:34<01:31,  8.02it/s]
 27%|██████████▋                             | 268/1000 [00:34<01:31,  7.99it/s]
 27%|██████████▊                             | 269/1000 [00:34<01:31,  8.00it/s]
 27%|██████████▊                             | 270/1000 [00:34<01:31,  7.99it/s]
 27%|██████████▊                             | 271/1000 [00:34<01:31,  8.01it/s]
 27%|██████████▉                             | 272/1000 [00:34<01:31,  7.98it/s]
 27%|██████████▉                             | 273/1000 [00:34<01:31,  7.97it/s]
 27%|██████████▉                             | 274/1000 [00:34<01:30,  7.99it/s]
 28%|███████████                             | 275/1000 [00:35<01:30,  8.04it/s]
 28%|███████████                             | 276/1000 [00:35<01:30,  8.02it/s]
 28%|███████████                             | 277/1000 [00:35<01:30,  8.01it/s]
 28%|███████████                             | 278/1000 [00:35<01:30,  8.01it/s]
 28%|███████████▏                            | 279/1000 [00:35<01:30,  8.00it/s]
 28%|███████████▏                            | 280/1000 [00:35<01:29,  8.05it/s]
 28%|███████████▏                            | 281/1000 [00:35<01:29,  8.02it/s]
 28%|███████████▎                            | 282/1000 [00:35<01:29,  8.01it/s]
 28%|███████████▎                            | 283/1000 [00:36<01:29,  8.00it/s]
 28%|███████████▎                            | 284/1000 [00:36<01:29,  7.99it/s]
 28%|███████████▍                            | 285/1000 [00:36<01:29,  7.98it/s]
 29%|███████████▍                            | 286/1000 [00:36<01:30,  7.91it/s]
 29%|███████████▍                            | 287/1000 [00:36<01:29,  7.92it/s]
 29%|███████████▌                            | 288/1000 [00:36<01:29,  7.95it/s]
 29%|███████████▌                            | 289/1000 [00:36<01:29,  7.91it/s]
 29%|███████████▌                            | 290/1000 [00:36<01:29,  7.94it/s]
 29%|███████████▋                            | 291/1000 [00:37<01:29,  7.96it/s]
 29%|███████████▋                            | 292/1000 [00:37<01:29,  7.93it/s]
 29%|███████████▋                            | 293/1000 [00:37<01:28,  7.95it/s]
 29%|███████████▊                            | 294/1000 [00:37<01:28,  7.95it/s]
 30%|███████████▊                            | 295/1000 [00:37<01:29,  7.92it/s]
 30%|███████████▊                            | 296/1000 [00:37<01:28,  7.95it/s]
 30%|███████████▉                            | 297/1000 [00:37<01:28,  7.95it/s]
 30%|███████████▉                            | 298/1000 [00:37<01:28,  7.95it/s]
 30%|███████████▉                            | 299/1000 [00:38<01:28,  7.93it/s]
 30%|████████████                            | 300/1000 [00:38<01:28,  7.90it/s]
 30%|████████████                            | 301/1000 [00:38<01:28,  7.94it/s]
 30%|████████████                            | 302/1000 [00:38<01:27,  7.97it/s]
 30%|████████████                            | 303/1000 [00:38<01:27,  7.98it/s]
 30%|████████████▏                           | 304/1000 [00:38<01:27,  8.00it/s]
 30%|████████████▏                           | 305/1000 [00:38<01:27,  7.96it/s]
 31%|████████████▏                           | 306/1000 [00:38<01:27,  7.96it/s]
 31%|████████████▎                           | 307/1000 [00:39<01:26,  8.00it/s]
 31%|████████████▎                           | 308/1000 [00:39<01:26,  7.97it/s]
 31%|████████████▎                           | 309/1000 [00:39<01:26,  8.00it/s]
 31%|████████████▍                           | 310/1000 [00:39<01:26,  8.00it/s]
 31%|████████████▍                           | 311/1000 [00:39<01:26,  8.00it/s]
 31%|████████████▍                           | 312/1000 [00:39<01:25,  8.01it/s]
 31%|████████████▌                           | 313/1000 [00:39<01:25,  8.01it/s]
 31%|████████████▌                           | 314/1000 [00:39<01:26,  7.90it/s]
 32%|████████████▌                           | 315/1000 [00:40<01:27,  7.85it/s]
 32%|████████████▋                           | 316/1000 [00:40<01:31,  7.47it/s]
 32%|████████████▋                           | 317/1000 [00:40<01:29,  7.59it/s]
 32%|████████████▋                           | 318/1000 [00:40<01:28,  7.67it/s]
 32%|████████████▊                           | 319/1000 [00:40<01:28,  7.73it/s]
 32%|████████████▊                           | 320/1000 [00:40<01:27,  7.81it/s]
 32%|████████████▊                           | 321/1000 [00:40<01:26,  7.83it/s]
 32%|████████████▉                           | 322/1000 [00:40<01:25,  7.90it/s]
 32%|████████████▉                           | 323/1000 [00:41<01:25,  7.94it/s]
 32%|████████████▉                           | 324/1000 [00:41<01:24,  7.96it/s]
 32%|█████████████                           | 325/1000 [00:41<01:24,  7.95it/s]
 33%|█████████████                           | 326/1000 [00:41<01:24,  7.95it/s]
 33%|█████████████                           | 327/1000 [00:41<01:24,  7.98it/s]
 33%|█████████████                           | 328/1000 [00:41<01:23,  8.03it/s]
 33%|█████████████▏                          | 329/1000 [00:41<01:23,  8.01it/s]
 33%|█████████████▏                          | 330/1000 [00:41<01:23,  8.02it/s]
 33%|█████████████▏                          | 331/1000 [00:42<01:23,  8.04it/s]
 33%|█████████████▎                          | 332/1000 [00:42<01:23,  8.02it/s]
 33%|█████████████▎                          | 333/1000 [00:42<01:22,  8.06it/s]
 33%|█████████████▎                          | 334/1000 [00:42<01:22,  8.04it/s]
 34%|█████████████▍                          | 335/1000 [00:42<01:22,  8.03it/s]
 34%|█████████████▍                          | 336/1000 [00:42<01:22,  8.03it/s]
 34%|█████████████▍                          | 337/1000 [00:42<01:22,  8.03it/s]
 34%|█████████████▌                          | 338/1000 [00:42<01:22,  8.04it/s]
 34%|█████████████▌                          | 339/1000 [00:43<01:22,  8.04it/s]
 34%|█████████████▌                          | 340/1000 [00:43<01:22,  8.01it/s]
 34%|█████████████▋                          | 341/1000 [00:43<01:22,  7.94it/s]
 34%|█████████████▋                          | 342/1000 [00:43<01:23,  7.91it/s]
 34%|█████████████▋                          | 343/1000 [00:43<01:22,  7.93it/s]
 34%|█████████████▊                          | 344/1000 [00:43<01:22,  7.98it/s]
 34%|█████████████▊                          | 345/1000 [00:43<01:22,  7.98it/s]
 35%|█████████████▊                          | 346/1000 [00:43<01:21,  8.00it/s]
 35%|█████████████▉                          | 347/1000 [00:44<01:21,  7.99it/s]
 35%|█████████████▉                          | 348/1000 [00:44<01:21,  7.99it/s]
 35%|█████████████▉                          | 349/1000 [00:44<01:21,  8.01it/s]
 35%|██████████████                          | 350/1000 [00:44<01:21,  8.02it/s]
 35%|██████████████                          | 351/1000 [00:44<01:21,  8.01it/s]
 35%|██████████████                          | 352/1000 [00:44<01:20,  8.01it/s]
 35%|██████████████                          | 353/1000 [00:44<01:20,  8.00it/s]
 35%|██████████████▏                         | 354/1000 [00:44<01:22,  7.87it/s]
 36%|██████████████▏                         | 355/1000 [00:45<01:21,  7.89it/s]
 36%|██████████████▏                         | 356/1000 [00:45<01:21,  7.89it/s]
 36%|██████████████▎                         | 357/1000 [00:45<01:21,  7.86it/s]
 36%|██████████████▎                         | 358/1000 [00:45<01:22,  7.82it/s]
 36%|██████████████▎                         | 359/1000 [00:45<01:21,  7.85it/s]
 36%|██████████████▍                         | 360/1000 [00:45<01:21,  7.90it/s]
 36%|██████████████▍                         | 361/1000 [00:45<01:21,  7.88it/s]
 36%|██████████████▍                         | 362/1000 [00:45<01:22,  7.78it/s]
 36%|██████████████▌                         | 363/1000 [00:46<01:21,  7.82it/s]
 36%|██████████████▌                         | 364/1000 [00:46<01:20,  7.87it/s]
 36%|██████████████▌                         | 365/1000 [00:46<01:20,  7.90it/s]
 37%|██████████████▋                         | 366/1000 [00:46<01:20,  7.89it/s]
 37%|██████████████▋                         | 367/1000 [00:46<01:19,  7.95it/s]
 37%|██████████████▋                         | 368/1000 [00:46<01:19,  7.96it/s]
 37%|██████████████▊                         | 369/1000 [00:46<01:19,  7.97it/s]
 37%|██████████████▊                         | 370/1000 [00:46<01:19,  7.96it/s]
 37%|██████████████▊                         | 371/1000 [00:47<01:19,  7.96it/s]
 37%|██████████████▉                         | 372/1000 [00:47<01:18,  7.99it/s]
 37%|██████████████▉                         | 373/1000 [00:47<01:18,  7.97it/s]
 37%|██████████████▉                         | 374/1000 [00:47<01:18,  7.99it/s]
 38%|███████████████                         | 375/1000 [00:47<01:18,  7.96it/s]
 38%|███████████████                         | 376/1000 [00:47<01:18,  8.00it/s]
 38%|███████████████                         | 377/1000 [00:47<01:18,  7.97it/s]
 38%|███████████████                         | 378/1000 [00:47<01:18,  7.96it/s]
 38%|███████████████▏                        | 379/1000 [00:48<01:18,  7.92it/s]
 38%|███████████████▏                        | 380/1000 [00:48<01:18,  7.94it/s]
 38%|███████████████▏                        | 381/1000 [00:48<01:17,  7.97it/s]
 38%|███████████████▎                        | 382/1000 [00:48<01:17,  7.97it/s]
 38%|███████████████▎                        | 383/1000 [00:48<01:17,  7.97it/s]
 38%|███████████████▎                        | 384/1000 [00:48<01:17,  7.94it/s]
 38%|███████████████▍                        | 385/1000 [00:48<01:17,  7.90it/s]
 39%|███████████████▍                        | 386/1000 [00:48<01:18,  7.86it/s]
 39%|███████████████▍                        | 387/1000 [00:49<01:17,  7.86it/s]
 39%|███████████████▌                        | 388/1000 [00:49<01:18,  7.84it/s]
 39%|███████████████▌                        | 389/1000 [00:49<01:17,  7.84it/s]
 39%|███████████████▌                        | 390/1000 [00:49<01:17,  7.85it/s]
 39%|███████████████▋                        | 391/1000 [00:49<01:17,  7.90it/s]
 39%|███████████████▋                        | 392/1000 [00:49<01:16,  7.91it/s]
 39%|███████████████▋                        | 393/1000 [00:49<01:17,  7.79it/s]
 39%|███████████████▊                        | 394/1000 [00:50<01:17,  7.85it/s]
 40%|███████████████▊                        | 395/1000 [00:50<01:22,  7.31it/s]
 40%|███████████████▊                        | 396/1000 [00:50<01:20,  7.46it/s]
 40%|███████████████▉                        | 397/1000 [00:50<01:19,  7.55it/s]
 40%|███████████████▉                        | 398/1000 [00:50<01:18,  7.69it/s]
 40%|███████████████▉                        | 399/1000 [00:50<01:17,  7.79it/s]
 40%|████████████████                        | 400/1000 [00:50<01:16,  7.84it/s]
 40%|████████████████                        | 401/1000 [00:50<01:16,  7.88it/s]
 40%|████████████████                        | 402/1000 [00:51<01:15,  7.92it/s]
 40%|████████████████                        | 403/1000 [00:51<01:15,  7.93it/s]
 40%|████████████████▏                       | 404/1000 [00:51<01:15,  7.90it/s]
 40%|████████████████▏                       | 405/1000 [00:51<01:16,  7.78it/s]
 41%|████████████████▏                       | 406/1000 [00:51<01:16,  7.74it/s]
 41%|████████████████▎                       | 407/1000 [00:51<01:16,  7.76it/s]
 41%|████████████████▎                       | 408/1000 [00:51<01:15,  7.80it/s]
 41%|████████████████▎                       | 409/1000 [00:51<01:15,  7.88it/s]
 41%|████████████████▍                       | 410/1000 [00:52<01:14,  7.88it/s]
 41%|████████████████▍                       | 411/1000 [00:52<01:16,  7.65it/s]
 41%|████████████████▍                       | 412/1000 [00:52<01:16,  7.68it/s]
 41%|████████████████▌                       | 413/1000 [00:52<01:15,  7.76it/s]
 41%|████████████████▌                       | 414/1000 [00:52<01:14,  7.84it/s]
 42%|████████████████▌                       | 415/1000 [00:52<01:14,  7.87it/s]
 42%|████████████████▋                       | 416/1000 [00:52<01:13,  7.92it/s]
 42%|████████████████▋                       | 417/1000 [00:52<01:13,  7.94it/s]
 42%|████████████████▋                       | 418/1000 [00:53<01:13,  7.97it/s]
 42%|████████████████▊                       | 419/1000 [00:53<01:12,  7.96it/s]
 42%|████████████████▊                       | 420/1000 [00:53<01:12,  7.99it/s]
 42%|████████████████▊                       | 421/1000 [00:53<01:12,  8.02it/s]
 42%|████████████████▉                       | 422/1000 [00:53<01:11,  8.05it/s]
 42%|████████████████▉                       | 423/1000 [00:53<01:11,  8.04it/s]
 42%|████████████████▉                       | 424/1000 [00:53<01:11,  8.04it/s]
 42%|█████████████████                       | 425/1000 [00:53<01:11,  8.03it/s]
 43%|█████████████████                       | 426/1000 [00:54<01:11,  8.02it/s]
 43%|█████████████████                       | 427/1000 [00:54<01:11,  8.02it/s]
 43%|█████████████████                       | 428/1000 [00:54<01:11,  8.00it/s]
 43%|█████████████████▏                      | 429/1000 [00:54<01:10,  8.04it/s]
 43%|█████████████████▏                      | 430/1000 [00:54<01:10,  8.07it/s]
 43%|█████████████████▏                      | 431/1000 [00:54<01:10,  8.08it/s]
 43%|█████████████████▎                      | 432/1000 [00:54<01:10,  8.08it/s]
 43%|█████████████████▎                      | 433/1000 [00:54<01:10,  8.05it/s]
 43%|█████████████████▎                      | 434/1000 [00:55<01:10,  8.05it/s]
 44%|█████████████████▍                      | 435/1000 [00:55<01:10,  8.02it/s]
 44%|█████████████████▍                      | 436/1000 [00:55<01:10,  8.03it/s]
 44%|█████████████████▍                      | 437/1000 [00:55<01:10,  8.02it/s]
 44%|█████████████████▌                      | 438/1000 [00:55<01:10,  8.02it/s]
 44%|█████████████████▌                      | 439/1000 [00:55<01:09,  8.04it/s]
 44%|█████████████████▌                      | 440/1000 [00:55<01:09,  8.04it/s]
 44%|█████████████████▋                      | 441/1000 [00:55<01:09,  8.02it/s]
 44%|█████████████████▋                      | 442/1000 [00:56<01:09,  7.99it/s]
 44%|█████████████████▋                      | 443/1000 [00:56<01:09,  7.99it/s]
 44%|█████████████████▊                      | 444/1000 [00:56<01:09,  8.01it/s]
 44%|█████████████████▊                      | 445/1000 [00:56<01:09,  8.02it/s]
 45%|█████████████████▊                      | 446/1000 [00:56<01:09,  8.03it/s]
 45%|█████████████████▉                      | 447/1000 [00:56<01:08,  8.04it/s]
 45%|█████████████████▉                      | 448/1000 [00:56<01:09,  7.99it/s]
 45%|█████████████████▉                      | 449/1000 [00:56<01:09,  7.94it/s]
 45%|██████████████████                      | 450/1000 [00:57<01:08,  7.98it/s]
 45%|██████████████████                      | 451/1000 [00:57<01:08,  7.98it/s]
 45%|██████████████████                      | 452/1000 [00:57<01:08,  7.95it/s]
 45%|██████████████████                      | 453/1000 [00:57<01:10,  7.80it/s]
 45%|██████████████████▏                     | 454/1000 [00:57<01:10,  7.78it/s]
 46%|██████████████████▏                     | 455/1000 [00:57<01:09,  7.79it/s]
 46%|██████████████████▏                     | 456/1000 [00:57<01:09,  7.87it/s]
 46%|██████████████████▎                     | 457/1000 [00:57<01:08,  7.87it/s]
 46%|██████████████████▎                     | 458/1000 [00:58<01:08,  7.88it/s]
 46%|██████████████████▎                     | 459/1000 [00:58<01:08,  7.85it/s]
 46%|██████████████████▍                     | 460/1000 [00:58<01:08,  7.89it/s]
 46%|██████████████████▍                     | 461/1000 [00:58<01:07,  7.93it/s]
 46%|██████████████████▍                     | 462/1000 [00:58<01:07,  7.95it/s]
 46%|██████████████████▌                     | 463/1000 [00:58<01:08,  7.87it/s]
 46%|██████████████████▌                     | 464/1000 [00:58<01:08,  7.77it/s]
 46%|██████████████████▌                     | 465/1000 [00:58<01:08,  7.81it/s]
 47%|██████████████████▋                     | 466/1000 [00:59<01:08,  7.85it/s]
 47%|██████████████████▋                     | 467/1000 [00:59<01:07,  7.87it/s]
 47%|██████████████████▋                     | 468/1000 [00:59<01:07,  7.90it/s]
 47%|██████████████████▊                     | 469/1000 [00:59<01:06,  7.96it/s]
 47%|██████████████████▊                     | 470/1000 [00:59<01:06,  7.97it/s]
 47%|██████████████████▊                     | 471/1000 [00:59<01:07,  7.88it/s]
 47%|██████████████████▉                     | 472/1000 [00:59<01:09,  7.56it/s]
 47%|██████████████████▉                     | 473/1000 [01:00<01:09,  7.62it/s]
 47%|██████████████████▉                     | 474/1000 [01:00<01:12,  7.25it/s]
 48%|███████████████████                     | 475/1000 [01:00<01:10,  7.46it/s]
 48%|███████████████████                     | 476/1000 [01:00<01:08,  7.62it/s]
 48%|███████████████████                     | 477/1000 [01:00<01:08,  7.67it/s]
 48%|███████████████████                     | 478/1000 [01:00<01:07,  7.75it/s]
 48%|███████████████████▏                    | 479/1000 [01:00<01:06,  7.84it/s]
 48%|███████████████████▏                    | 480/1000 [01:00<01:05,  7.90it/s]
 48%|███████████████████▏                    | 481/1000 [01:01<01:05,  7.94it/s]
 48%|███████████████████▎                    | 482/1000 [01:01<01:05,  7.88it/s]
 48%|███████████████████▎                    | 483/1000 [01:01<01:05,  7.92it/s]
 48%|███████████████████▎                    | 484/1000 [01:01<01:04,  7.96it/s]
 48%|███████████████████▍                    | 485/1000 [01:01<01:04,  7.98it/s]
 49%|███████████████████▍                    | 486/1000 [01:01<01:04,  7.98it/s]
 49%|███████████████████▍                    | 487/1000 [01:01<01:04,  8.00it/s]
 49%|███████████████████▌                    | 488/1000 [01:01<01:04,  8.00it/s]
 49%|███████████████████▌                    | 489/1000 [01:02<01:03,  7.98it/s]
 49%|███████████████████▌                    | 490/1000 [01:02<01:04,  7.91it/s]
 49%|███████████████████▋                    | 491/1000 [01:02<01:04,  7.90it/s]
 49%|███████████████████▋                    | 492/1000 [01:02<01:04,  7.92it/s]
 49%|███████████████████▋                    | 493/1000 [01:02<01:03,  7.95it/s]
 49%|███████████████████▊                    | 494/1000 [01:02<01:03,  7.97it/s]
 50%|███████████████████▊                    | 495/1000 [01:02<01:03,  7.97it/s]
 50%|███████████████████▊                    | 496/1000 [01:02<01:03,  7.97it/s]
 50%|███████████████████▉                    | 497/1000 [01:03<01:02,  7.99it/s]
 50%|███████████████████▉                    | 498/1000 [01:03<01:02,  7.97it/s]
 50%|███████████████████▉                    | 499/1000 [01:03<01:02,  8.00it/s]
 50%|████████████████████                    | 500/1000 [01:03<01:02,  8.02it/s]
 50%|████████████████████                    | 501/1000 [01:03<01:01,  8.06it/s]
 50%|████████████████████                    | 502/1000 [01:03<01:01,  8.05it/s]
 50%|████████████████████                    | 503/1000 [01:03<01:01,  8.07it/s]
 50%|████████████████████▏                   | 504/1000 [01:03<01:01,  8.06it/s]
 50%|████████████████████▏                   | 505/1000 [01:04<01:01,  8.08it/s]
 51%|████████████████████▏                   | 506/1000 [01:04<01:01,  8.01it/s]
 51%|████████████████████▎                   | 507/1000 [01:04<01:01,  7.98it/s]
 51%|████████████████████▎                   | 508/1000 [01:04<01:01,  7.97it/s]
 51%|████████████████████▎                   | 509/1000 [01:04<01:01,  8.01it/s]
 51%|████████████████████▍                   | 510/1000 [01:04<01:01,  8.02it/s]
 51%|████████████████████▍                   | 511/1000 [01:04<01:00,  8.05it/s]
 51%|████████████████████▍                   | 512/1000 [01:04<01:00,  8.03it/s]
 51%|████████████████████▌                   | 513/1000 [01:05<01:00,  8.04it/s]
 51%|████████████████████▌                   | 514/1000 [01:05<01:00,  7.98it/s]
 52%|████████████████████▌                   | 515/1000 [01:05<01:00,  8.00it/s]
 52%|████████████████████▋                   | 516/1000 [01:05<01:00,  8.04it/s]
 52%|████████████████████▋                   | 517/1000 [01:05<01:00,  8.04it/s]
 52%|████████████████████▋                   | 518/1000 [01:05<01:00,  8.01it/s]
 52%|████████████████████▊                   | 519/1000 [01:05<00:59,  8.04it/s]
 52%|████████████████████▊                   | 520/1000 [01:05<00:59,  8.04it/s]
 52%|████████████████████▊                   | 521/1000 [01:06<01:00,  7.97it/s]
 52%|████████████████████▉                   | 522/1000 [01:06<01:00,  7.90it/s]
 52%|████████████████████▉                   | 523/1000 [01:06<01:00,  7.92it/s]
 52%|████████████████████▉                   | 524/1000 [01:06<00:59,  7.94it/s]
 52%|█████████████████████                   | 525/1000 [01:06<00:59,  7.98it/s]
 53%|█████████████████████                   | 526/1000 [01:06<00:59,  8.00it/s]
 53%|█████████████████████                   | 527/1000 [01:06<00:59,  8.01it/s]
 53%|█████████████████████                   | 528/1000 [01:06<00:58,  8.04it/s]
 53%|█████████████████████▏                  | 529/1000 [01:07<00:58,  8.07it/s]
 53%|█████████████████████▏                  | 530/1000 [01:07<00:58,  8.01it/s]
 53%|█████████████████████▏                  | 531/1000 [01:07<00:58,  8.01it/s]
 53%|█████████████████████▎                  | 532/1000 [01:07<00:58,  8.02it/s]
 53%|█████████████████████▎                  | 533/1000 [01:07<00:58,  8.04it/s]
 53%|█████████████████████▎                  | 534/1000 [01:07<00:57,  8.05it/s]
 54%|█████████████████████▍                  | 535/1000 [01:07<00:57,  8.04it/s]
 54%|█████████████████████▍                  | 536/1000 [01:07<00:57,  8.05it/s]
 54%|█████████████████████▍                  | 537/1000 [01:08<00:57,  8.07it/s]
 54%|█████████████████████▌                  | 538/1000 [01:08<00:57,  8.03it/s]
 54%|█████████████████████▌                  | 539/1000 [01:08<00:57,  8.04it/s]
 54%|█████████████████████▌                  | 540/1000 [01:08<00:57,  8.06it/s]
 54%|█████████████████████▋                  | 541/1000 [01:08<00:56,  8.08it/s]
 54%|█████████████████████▋                  | 542/1000 [01:08<00:56,  8.09it/s]
 54%|█████████████████████▋                  | 543/1000 [01:08<00:56,  8.03it/s]
 54%|█████████████████████▊                  | 544/1000 [01:08<00:57,  7.98it/s]
 55%|█████████████████████▊                  | 545/1000 [01:09<00:57,  7.97it/s]
 55%|█████████████████████▊                  | 546/1000 [01:09<00:57,  7.94it/s]
 55%|█████████████████████▉                  | 547/1000 [01:09<00:56,  7.95it/s]
 55%|█████████████████████▉                  | 548/1000 [01:09<00:56,  7.99it/s]
 55%|█████████████████████▉                  | 549/1000 [01:09<00:56,  8.01it/s]
 55%|██████████████████████                  | 550/1000 [01:09<00:55,  8.04it/s]
 55%|██████████████████████                  | 551/1000 [01:09<00:56,  7.92it/s]
 55%|██████████████████████                  | 552/1000 [01:09<00:56,  7.96it/s]
 55%|██████████████████████                  | 553/1000 [01:10<00:58,  7.67it/s]
 55%|██████████████████████▏                 | 554/1000 [01:10<00:59,  7.46it/s]
 56%|██████████████████████▏                 | 555/1000 [01:10<00:58,  7.59it/s]
 56%|██████████████████████▏                 | 556/1000 [01:10<00:57,  7.70it/s]
 56%|██████████████████████▎                 | 557/1000 [01:10<00:56,  7.78it/s]
 56%|██████████████████████▎                 | 558/1000 [01:10<00:56,  7.88it/s]
 56%|██████████████████████▎                 | 559/1000 [01:10<00:55,  7.91it/s]
 56%|██████████████████████▍                 | 560/1000 [01:10<00:55,  7.96it/s]
 56%|██████████████████████▍                 | 561/1000 [01:11<00:54,  8.00it/s]
 56%|██████████████████████▍                 | 562/1000 [01:11<00:55,  7.95it/s]
 56%|██████████████████████▌                 | 563/1000 [01:11<00:54,  7.96it/s]
 56%|██████████████████████▌                 | 564/1000 [01:11<00:54,  7.94it/s]
 56%|██████████████████████▌                 | 565/1000 [01:11<00:54,  8.01it/s]
 57%|██████████████████████▋                 | 566/1000 [01:11<00:54,  8.03it/s]
 57%|██████████████████████▋                 | 567/1000 [01:11<00:53,  8.06it/s]
 57%|██████████████████████▋                 | 568/1000 [01:11<00:53,  8.06it/s]
 57%|██████████████████████▊                 | 569/1000 [01:12<00:53,  8.08it/s]
 57%|██████████████████████▊                 | 570/1000 [01:12<00:53,  8.00it/s]
 57%|██████████████████████▊                 | 571/1000 [01:12<00:53,  8.00it/s]
 57%|██████████████████████▉                 | 572/1000 [01:12<00:53,  7.99it/s]
 57%|██████████████████████▉                 | 573/1000 [01:12<00:53,  8.02it/s]
 57%|██████████████████████▉                 | 574/1000 [01:12<00:52,  8.06it/s]
 57%|███████████████████████                 | 575/1000 [01:12<00:52,  8.05it/s]
 58%|███████████████████████                 | 576/1000 [01:12<00:52,  8.07it/s]
 58%|███████████████████████                 | 577/1000 [01:13<00:52,  8.02it/s]
 58%|███████████████████████                 | 578/1000 [01:13<00:52,  7.99it/s]
 58%|███████████████████████▏                | 579/1000 [01:13<00:52,  7.98it/s]
 58%|███████████████████████▏                | 580/1000 [01:13<00:52,  8.01it/s]
 58%|███████████████████████▏                | 581/1000 [01:13<00:52,  8.03it/s]
 58%|███████████████████████▎                | 582/1000 [01:13<00:51,  8.07it/s]
 58%|███████████████████████▎                | 583/1000 [01:13<00:51,  8.07it/s]
 58%|███████████████████████▎                | 584/1000 [01:13<00:51,  8.08it/s]
 58%|███████████████████████▍                | 585/1000 [01:14<00:51,  8.08it/s]
 59%|███████████████████████▍                | 586/1000 [01:14<00:51,  8.00it/s]
 59%|███████████████████████▍                | 587/1000 [01:14<00:51,  8.04it/s]
 59%|███████████████████████▌                | 588/1000 [01:14<00:51,  8.06it/s]
 59%|███████████████████████▌                | 589/1000 [01:14<00:50,  8.09it/s]
 59%|███████████████████████▌                | 590/1000 [01:14<00:50,  8.10it/s]
 59%|███████████████████████▋                | 591/1000 [01:14<00:50,  8.10it/s]
 59%|███████████████████████▋                | 592/1000 [01:14<00:50,  8.10it/s]
 59%|███████████████████████▋                | 593/1000 [01:15<00:50,  8.12it/s]
 59%|███████████████████████▊                | 594/1000 [01:15<00:50,  8.06it/s]
 60%|███████████████████████▊                | 595/1000 [01:15<00:50,  8.06it/s]
 60%|███████████████████████▊                | 596/1000 [01:15<00:51,  7.80it/s]
 60%|███████████████████████▉                | 597/1000 [01:15<00:51,  7.89it/s]
 60%|███████████████████████▉                | 598/1000 [01:15<00:51,  7.79it/s]
 60%|███████████████████████▉                | 599/1000 [01:15<00:51,  7.79it/s]
 60%|████████████████████████                | 600/1000 [01:15<00:50,  7.87it/s]
 60%|████████████████████████                | 601/1000 [01:16<00:50,  7.96it/s]
 60%|████████████████████████                | 602/1000 [01:16<00:50,  7.93it/s]
 60%|████████████████████████                | 603/1000 [01:16<00:49,  7.98it/s]
 60%|████████████████████████▏               | 604/1000 [01:16<00:49,  7.99it/s]
 60%|████████████████████████▏               | 605/1000 [01:16<00:49,  8.01it/s]
 61%|████████████████████████▏               | 606/1000 [01:16<00:49,  8.03it/s]
 61%|████████████████████████▎               | 607/1000 [01:16<00:49,  8.00it/s]
 61%|████████████████████████▎               | 608/1000 [01:16<00:48,  8.04it/s]
 61%|████████████████████████▎               | 609/1000 [01:17<00:48,  8.07it/s]
 61%|████████████████████████▍               | 610/1000 [01:17<00:48,  8.05it/s]
 61%|████████████████████████▍               | 611/1000 [01:17<00:48,  8.04it/s]
 61%|████████████████████████▍               | 612/1000 [01:17<00:48,  8.03it/s]
 61%|████████████████████████▌               | 613/1000 [01:17<00:48,  8.06it/s]
 61%|████████████████████████▌               | 614/1000 [01:17<00:47,  8.09it/s]
 62%|████████████████████████▌               | 615/1000 [01:17<00:47,  8.09it/s]
 62%|████████████████████████▋               | 616/1000 [01:17<00:47,  8.09it/s]
 62%|████████████████████████▋               | 617/1000 [01:18<00:47,  8.07it/s]
 62%|████████████████████████▋               | 618/1000 [01:18<00:47,  8.05it/s]
 62%|████████████████████████▊               | 619/1000 [01:18<00:47,  8.02it/s]
 62%|████████████████████████▊               | 620/1000 [01:18<00:47,  7.98it/s]
 62%|████████████████████████▊               | 621/1000 [01:18<00:47,  7.97it/s]
 62%|████████████████████████▉               | 622/1000 [01:18<00:47,  7.94it/s]
 62%|████████████████████████▉               | 623/1000 [01:18<00:47,  7.92it/s]
 62%|████████████████████████▉               | 624/1000 [01:18<00:47,  7.87it/s]
 62%|█████████████████████████               | 625/1000 [01:19<00:47,  7.93it/s]
 63%|█████████████████████████               | 626/1000 [01:19<00:47,  7.91it/s]
 63%|█████████████████████████               | 627/1000 [01:19<00:46,  7.95it/s]
 63%|█████████████████████████               | 628/1000 [01:19<00:46,  7.97it/s]
 63%|█████████████████████████▏              | 629/1000 [01:19<00:46,  8.02it/s]
 63%|█████████████████████████▏              | 630/1000 [01:19<00:45,  8.05it/s]
 63%|█████████████████████████▏              | 631/1000 [01:19<00:46,  8.00it/s]
 63%|█████████████████████████▎              | 632/1000 [01:19<00:46,  7.99it/s]
 63%|█████████████████████████▎              | 633/1000 [01:20<00:48,  7.56it/s]
 63%|█████████████████████████▎              | 634/1000 [01:20<00:47,  7.63it/s]
 64%|█████████████████████████▍              | 635/1000 [01:20<00:47,  7.70it/s]
 64%|█████████████████████████▍              | 636/1000 [01:20<00:46,  7.84it/s]
 64%|█████████████████████████▍              | 637/1000 [01:20<00:45,  7.91it/s]
 64%|█████████████████████████▌              | 638/1000 [01:20<00:45,  7.98it/s]
 64%|█████████████████████████▌              | 639/1000 [01:20<00:45,  7.99it/s]
 64%|█████████████████████████▌              | 640/1000 [01:20<00:44,  8.00it/s]
 64%|█████████████████████████▋              | 641/1000 [01:21<00:44,  8.02it/s]
 64%|█████████████████████████▋              | 642/1000 [01:21<00:44,  7.99it/s]
 64%|█████████████████████████▋              | 643/1000 [01:21<00:44,  8.01it/s]
 64%|█████████████████████████▊              | 644/1000 [01:21<00:44,  8.04it/s]
 64%|█████████████████████████▊              | 645/1000 [01:21<00:44,  8.07it/s]
 65%|█████████████████████████▊              | 646/1000 [01:21<00:44,  8.01it/s]
 65%|█████████████████████████▉              | 647/1000 [01:21<00:44,  8.01it/s]
 65%|█████████████████████████▉              | 648/1000 [01:21<00:43,  8.04it/s]
 65%|█████████████████████████▉              | 649/1000 [01:22<00:43,  8.06it/s]
 65%|██████████████████████████              | 650/1000 [01:22<00:43,  8.04it/s]
 65%|██████████████████████████              | 651/1000 [01:22<00:44,  7.83it/s]
 65%|██████████████████████████              | 652/1000 [01:22<00:44,  7.78it/s]
 65%|██████████████████████████              | 653/1000 [01:22<00:44,  7.87it/s]
 65%|██████████████████████████▏             | 654/1000 [01:22<00:43,  7.94it/s]
 66%|██████████████████████████▏             | 655/1000 [01:22<00:43,  7.97it/s]
 66%|██████████████████████████▏             | 656/1000 [01:22<00:42,  8.02it/s]
 66%|██████████████████████████▎             | 657/1000 [01:23<00:42,  8.04it/s]
 66%|██████████████████████████▎             | 658/1000 [01:23<00:42,  8.01it/s]
 66%|██████████████████████████▎             | 659/1000 [01:23<00:43,  7.93it/s]
 66%|██████████████████████████▍             | 660/1000 [01:23<00:42,  7.98it/s]
 66%|██████████████████████████▍             | 661/1000 [01:23<00:42,  8.01it/s]
 66%|██████████████████████████▍             | 662/1000 [01:23<00:42,  8.03it/s]
 66%|██████████████████████████▌             | 663/1000 [01:23<00:42,  7.86it/s]
 66%|██████████████████████████▌             | 664/1000 [01:23<00:42,  7.88it/s]
 66%|██████████████████████████▌             | 665/1000 [01:24<00:42,  7.93it/s]
 67%|██████████████████████████▋             | 666/1000 [01:24<00:42,  7.93it/s]
 67%|██████████████████████████▋             | 667/1000 [01:24<00:41,  7.93it/s]
 67%|██████████████████████████▋             | 668/1000 [01:24<00:41,  7.98it/s]
 67%|██████████████████████████▊             | 669/1000 [01:24<00:41,  7.99it/s]
 67%|██████████████████████████▊             | 670/1000 [01:24<00:41,  8.02it/s]
 67%|██████████████████████████▊             | 671/1000 [01:24<00:41,  7.94it/s]
 67%|██████████████████████████▉             | 672/1000 [01:24<00:41,  7.98it/s]
 67%|██████████████████████████▉             | 673/1000 [01:25<00:41,  7.92it/s]
 67%|██████████████████████████▉             | 674/1000 [01:25<00:41,  7.83it/s]
 68%|███████████████████████████             | 675/1000 [01:25<00:41,  7.75it/s]
 68%|███████████████████████████             | 676/1000 [01:25<00:41,  7.78it/s]
 68%|███████████████████████████             | 677/1000 [01:25<00:40,  7.88it/s]
 68%|███████████████████████████             | 678/1000 [01:25<00:40,  7.94it/s]
 68%|███████████████████████████▏            | 679/1000 [01:25<00:40,  7.89it/s]
 68%|███████████████████████████▏            | 680/1000 [01:26<00:40,  7.92it/s]
 68%|███████████████████████████▏            | 681/1000 [01:26<00:40,  7.95it/s]
 68%|███████████████████████████▎            | 682/1000 [01:26<00:39,  7.96it/s]
 68%|███████████████████████████▎            | 683/1000 [01:26<00:39,  7.96it/s]
 68%|███████████████████████████▎            | 684/1000 [01:26<00:39,  7.98it/s]
 68%|███████████████████████████▍            | 685/1000 [01:26<00:39,  7.99it/s]
 69%|███████████████████████████▍            | 686/1000 [01:26<00:39,  7.99it/s]
 69%|███████████████████████████▍            | 687/1000 [01:26<00:39,  7.92it/s]
 69%|███████████████████████████▌            | 688/1000 [01:27<00:39,  7.92it/s]
 69%|███████████████████████████▌            | 689/1000 [01:27<00:39,  7.97it/s]
 69%|███████████████████████████▌            | 690/1000 [01:27<00:39,  7.93it/s]
 69%|███████████████████████████▋            | 691/1000 [01:27<00:38,  7.96it/s]
 69%|███████████████████████████▋            | 692/1000 [01:27<00:38,  7.96it/s]
 69%|███████████████████████████▋            | 693/1000 [01:27<00:38,  7.99it/s]
 69%|███████████████████████████▊            | 694/1000 [01:27<00:38,  7.97it/s]
 70%|███████████████████████████▊            | 695/1000 [01:27<00:38,  7.95it/s]
 70%|███████████████████████████▊            | 696/1000 [01:28<00:38,  7.98it/s]
 70%|███████████████████████████▉            | 697/1000 [01:28<00:37,  8.01it/s]
 70%|███████████████████████████▉            | 698/1000 [01:28<00:37,  7.99it/s]
 70%|███████████████████████████▉            | 699/1000 [01:28<00:37,  8.00it/s]
 70%|████████████████████████████            | 700/1000 [01:28<00:37,  8.01it/s]
 70%|████████████████████████████            | 701/1000 [01:28<00:37,  8.04it/s]
 70%|████████████████████████████            | 702/1000 [01:28<00:37,  8.05it/s]
 70%|████████████████████████████            | 703/1000 [01:28<00:37,  7.98it/s]
 70%|████████████████████████████▏           | 704/1000 [01:29<00:37,  7.98it/s]
 70%|████████████████████████████▏           | 705/1000 [01:29<00:36,  7.99it/s]
 71%|████████████████████████████▏           | 706/1000 [01:29<00:36,  7.98it/s]
 71%|████████████████████████████▎           | 707/1000 [01:29<00:36,  7.98it/s]
 71%|████████████████████████████▎           | 708/1000 [01:29<00:36,  8.01it/s]
 71%|████████████████████████████▎           | 709/1000 [01:29<00:36,  8.02it/s]
 71%|████████████████████████████▍           | 710/1000 [01:29<00:36,  8.01it/s]
 71%|████████████████████████████▍           | 711/1000 [01:29<00:36,  7.85it/s]
 71%|████████████████████████████▍           | 712/1000 [01:30<00:36,  7.89it/s]
 71%|████████████████████████████▌           | 713/1000 [01:30<00:38,  7.44it/s]
 71%|████████████████████████████▌           | 714/1000 [01:30<00:37,  7.59it/s]
 72%|████████████████████████████▌           | 715/1000 [01:30<00:37,  7.64it/s]
 72%|████████████████████████████▋           | 716/1000 [01:30<00:36,  7.75it/s]
 72%|████████████████████████████▋           | 717/1000 [01:30<00:36,  7.83it/s]
 72%|████████████████████████████▋           | 718/1000 [01:30<00:35,  7.88it/s]
 72%|████████████████████████████▊           | 719/1000 [01:30<00:35,  7.90it/s]
 72%|████████████████████████████▊           | 720/1000 [01:31<00:35,  7.94it/s]
 72%|████████████████████████████▊           | 721/1000 [01:31<00:34,  7.98it/s]
 72%|████████████████████████████▉           | 722/1000 [01:31<00:34,  7.95it/s]
 72%|████████████████████████████▉           | 723/1000 [01:31<00:34,  7.96it/s]
 72%|████████████████████████████▉           | 724/1000 [01:31<00:34,  7.97it/s]
 72%|█████████████████████████████           | 725/1000 [01:31<00:34,  7.99it/s]
 73%|█████████████████████████████           | 726/1000 [01:31<00:34,  7.99it/s]
 73%|█████████████████████████████           | 727/1000 [01:31<00:34,  7.96it/s]
 73%|█████████████████████████████           | 728/1000 [01:32<00:34,  8.00it/s]
 73%|█████████████████████████████▏          | 729/1000 [01:32<00:33,  7.99it/s]
 73%|█████████████████████████████▏          | 730/1000 [01:32<00:33,  7.95it/s]
 73%|█████████████████████████████▏          | 731/1000 [01:32<00:33,  7.95it/s]
 73%|█████████████████████████████▎          | 732/1000 [01:32<00:33,  7.99it/s]
 73%|█████████████████████████████▎          | 733/1000 [01:32<00:33,  8.02it/s]
 73%|█████████████████████████████▎          | 734/1000 [01:32<00:33,  8.03it/s]
 74%|█████████████████████████████▍          | 735/1000 [01:32<00:33,  7.93it/s]
 74%|█████████████████████████████▍          | 736/1000 [01:33<00:33,  7.97it/s]
 74%|█████████████████████████████▍          | 737/1000 [01:33<00:32,  7.98it/s]
 74%|█████████████████████████████▌          | 738/1000 [01:33<00:32,  7.99it/s]
 74%|█████████████████████████████▌          | 739/1000 [01:33<00:32,  7.98it/s]
 74%|█████████████████████████████▌          | 740/1000 [01:33<00:32,  8.01it/s]
 74%|█████████████████████████████▋          | 741/1000 [01:33<00:33,  7.74it/s]
 74%|█████████████████████████████▋          | 742/1000 [01:33<00:33,  7.77it/s]
 74%|█████████████████████████████▋          | 743/1000 [01:33<00:34,  7.53it/s]
 74%|█████████████████████████████▊          | 744/1000 [01:34<00:35,  7.31it/s]
 74%|█████████████████████████████▊          | 745/1000 [01:34<00:34,  7.37it/s]
 75%|█████████████████████████████▊          | 746/1000 [01:34<00:33,  7.55it/s]
 75%|█████████████████████████████▉          | 747/1000 [01:34<00:33,  7.67it/s]
 75%|█████████████████████████████▉          | 748/1000 [01:34<00:32,  7.79it/s]
 75%|█████████████████████████████▉          | 749/1000 [01:34<00:31,  7.88it/s]
 75%|██████████████████████████████          | 750/1000 [01:34<00:31,  7.89it/s]
 75%|██████████████████████████████          | 751/1000 [01:35<00:31,  7.83it/s]
 75%|██████████████████████████████          | 752/1000 [01:35<00:32,  7.74it/s]
 75%|██████████████████████████████          | 753/1000 [01:35<00:31,  7.77it/s]
 75%|██████████████████████████████▏         | 754/1000 [01:35<00:31,  7.84it/s]
 76%|██████████████████████████████▏         | 755/1000 [01:35<00:30,  7.90it/s]
 76%|██████████████████████████████▏         | 756/1000 [01:35<00:30,  7.94it/s]
 76%|██████████████████████████████▎         | 757/1000 [01:35<00:30,  7.98it/s]
 76%|██████████████████████████████▎         | 758/1000 [01:35<00:30,  7.99it/s]
 76%|██████████████████████████████▎         | 759/1000 [01:36<00:30,  8.01it/s]
 76%|██████████████████████████████▍         | 760/1000 [01:36<00:29,  8.02it/s]
 76%|██████████████████████████████▍         | 761/1000 [01:36<00:29,  7.98it/s]
 76%|██████████████████████████████▍         | 762/1000 [01:36<00:29,  7.96it/s]
 76%|██████████████████████████████▌         | 763/1000 [01:36<00:29,  7.99it/s]
 76%|██████████████████████████████▌         | 764/1000 [01:36<00:29,  8.02it/s]
 76%|██████████████████████████████▌         | 765/1000 [01:36<00:29,  8.04it/s]
 77%|██████████████████████████████▋         | 766/1000 [01:36<00:29,  8.02it/s]
 77%|██████████████████████████████▋         | 767/1000 [01:37<00:29,  8.03it/s]
 77%|██████████████████████████████▋         | 768/1000 [01:37<00:28,  8.04it/s]
 77%|██████████████████████████████▊         | 769/1000 [01:37<00:28,  8.03it/s]
 77%|██████████████████████████████▊         | 770/1000 [01:37<00:28,  8.03it/s]
 77%|██████████████████████████████▊         | 771/1000 [01:37<00:28,  7.99it/s]
 77%|██████████████████████████████▉         | 772/1000 [01:37<00:28,  7.98it/s]
 77%|██████████████████████████████▉         | 773/1000 [01:37<00:28,  7.97it/s]
 77%|██████████████████████████████▉         | 774/1000 [01:37<00:28,  7.98it/s]
 78%|███████████████████████████████         | 775/1000 [01:38<00:28,  8.00it/s]
 78%|███████████████████████████████         | 776/1000 [01:38<00:27,  8.02it/s]
 78%|███████████████████████████████         | 777/1000 [01:38<00:27,  8.01it/s]
 78%|███████████████████████████████         | 778/1000 [01:38<00:27,  8.02it/s]
 78%|███████████████████████████████▏        | 779/1000 [01:38<00:27,  7.97it/s]
 78%|███████████████████████████████▏        | 780/1000 [01:38<00:27,  7.98it/s]
 78%|███████████████████████████████▏        | 781/1000 [01:38<00:27,  7.97it/s]
 78%|███████████████████████████████▎        | 782/1000 [01:38<00:27,  7.93it/s]
 78%|███████████████████████████████▎        | 783/1000 [01:39<00:27,  7.87it/s]
 78%|███████████████████████████████▎        | 784/1000 [01:39<00:28,  7.64it/s]
 78%|███████████████████████████████▍        | 785/1000 [01:39<00:28,  7.68it/s]
 79%|███████████████████████████████▍        | 786/1000 [01:39<00:27,  7.76it/s]
 79%|███████████████████████████████▍        | 787/1000 [01:39<00:27,  7.82it/s]
 79%|███████████████████████████████▌        | 788/1000 [01:39<00:26,  7.87it/s]
 79%|███████████████████████████████▌        | 789/1000 [01:39<00:26,  7.90it/s]
 79%|███████████████████████████████▌        | 790/1000 [01:39<00:26,  7.81it/s]
 79%|███████████████████████████████▋        | 791/1000 [01:40<00:27,  7.70it/s]
 79%|███████████████████████████████▋        | 792/1000 [01:40<00:28,  7.42it/s]
 79%|███████████████████████████████▋        | 793/1000 [01:40<00:27,  7.54it/s]
 79%|███████████████████████████████▊        | 794/1000 [01:40<00:26,  7.65it/s]
 80%|███████████████████████████████▊        | 795/1000 [01:40<00:26,  7.71it/s]
 80%|███████████████████████████████▊        | 796/1000 [01:40<00:26,  7.81it/s]
 80%|███████████████████████████████▉        | 797/1000 [01:40<00:25,  7.87it/s]
 80%|███████████████████████████████▉        | 798/1000 [01:40<00:25,  7.88it/s]
 80%|███████████████████████████████▉        | 799/1000 [01:41<00:25,  7.91it/s]
 80%|████████████████████████████████        | 800/1000 [01:41<00:25,  7.93it/s]
 80%|████████████████████████████████        | 801/1000 [01:41<00:25,  7.92it/s]
 80%|████████████████████████████████        | 802/1000 [01:41<00:25,  7.92it/s]
 80%|████████████████████████████████        | 803/1000 [01:41<00:24,  7.93it/s]
 80%|████████████████████████████████▏       | 804/1000 [01:41<00:24,  7.97it/s]
 80%|████████████████████████████████▏       | 805/1000 [01:41<00:24,  8.00it/s]
 81%|████████████████████████████████▏       | 806/1000 [01:41<00:24,  7.95it/s]
 81%|████████████████████████████████▎       | 807/1000 [01:42<00:24,  7.97it/s]
 81%|████████████████████████████████▎       | 808/1000 [01:42<00:24,  7.95it/s]
 81%|████████████████████████████████▎       | 809/1000 [01:42<00:24,  7.94it/s]
 81%|████████████████████████████████▍       | 810/1000 [01:42<00:23,  7.96it/s]
 81%|████████████████████████████████▍       | 811/1000 [01:42<00:23,  7.96it/s]
 81%|████████████████████████████████▍       | 812/1000 [01:42<00:23,  7.96it/s]
 81%|████████████████████████████████▌       | 813/1000 [01:42<00:23,  7.98it/s]
 81%|████████████████████████████████▌       | 814/1000 [01:42<00:23,  7.94it/s]
 82%|████████████████████████████████▌       | 815/1000 [01:43<00:23,  7.96it/s]
 82%|████████████████████████████████▋       | 816/1000 [01:43<00:23,  7.92it/s]
 82%|████████████████████████████████▋       | 817/1000 [01:43<00:23,  7.91it/s]
 82%|████████████████████████████████▋       | 818/1000 [01:43<00:22,  7.93it/s]
 82%|████████████████████████████████▊       | 819/1000 [01:43<00:22,  7.94it/s]
 82%|████████████████████████████████▊       | 820/1000 [01:43<00:22,  7.89it/s]
 82%|████████████████████████████████▊       | 821/1000 [01:43<00:22,  7.91it/s]
 82%|████████████████████████████████▉       | 822/1000 [01:43<00:22,  7.88it/s]
 82%|████████████████████████████████▉       | 823/1000 [01:44<00:22,  7.93it/s]
 82%|████████████████████████████████▉       | 824/1000 [01:44<00:22,  7.93it/s]
 82%|█████████████████████████████████       | 825/1000 [01:44<00:22,  7.88it/s]
 83%|█████████████████████████████████       | 826/1000 [01:44<00:23,  7.56it/s]
 83%|█████████████████████████████████       | 827/1000 [01:44<00:23,  7.37it/s]
 83%|█████████████████████████████████       | 828/1000 [01:44<00:23,  7.44it/s]
 83%|█████████████████████████████████▏      | 829/1000 [01:44<00:22,  7.55it/s]
 83%|█████████████████████████████████▏      | 830/1000 [01:45<00:22,  7.66it/s]
 83%|█████████████████████████████████▏      | 831/1000 [01:45<00:21,  7.75it/s]
 83%|█████████████████████████████████▎      | 832/1000 [01:45<00:21,  7.69it/s]
 83%|█████████████████████████████████▎      | 833/1000 [01:45<00:21,  7.77it/s]
 83%|█████████████████████████████████▎      | 834/1000 [01:45<00:21,  7.55it/s]
 84%|█████████████████████████████████▍      | 835/1000 [01:45<00:21,  7.71it/s]
 84%|█████████████████████████████████▍      | 836/1000 [01:45<00:21,  7.79it/s]
 84%|█████████████████████████████████▍      | 837/1000 [01:45<00:20,  7.88it/s]
 84%|█████████████████████████████████▌      | 838/1000 [01:46<00:20,  7.94it/s]
 84%|█████████████████████████████████▌      | 839/1000 [01:46<00:20,  8.00it/s]
 84%|█████████████████████████████████▌      | 840/1000 [01:46<00:20,  7.96it/s]
 84%|█████████████████████████████████▋      | 841/1000 [01:46<00:19,  7.97it/s]
 84%|█████████████████████████████████▋      | 842/1000 [01:46<00:19,  7.98it/s]
 84%|█████████████████████████████████▋      | 843/1000 [01:46<00:19,  7.98it/s]
 84%|█████████████████████████████████▊      | 844/1000 [01:46<00:19,  8.00it/s]
 84%|█████████████████████████████████▊      | 845/1000 [01:46<00:19,  8.00it/s]
 85%|█████████████████████████████████▊      | 846/1000 [01:47<00:19,  8.04it/s]
 85%|█████████████████████████████████▉      | 847/1000 [01:47<00:19,  8.04it/s]
 85%|█████████████████████████████████▉      | 848/1000 [01:47<00:18,  8.02it/s]
 85%|█████████████████████████████████▉      | 849/1000 [01:47<00:18,  7.98it/s]
 85%|██████████████████████████████████      | 850/1000 [01:47<00:18,  7.98it/s]
 85%|██████████████████████████████████      | 851/1000 [01:47<00:18,  7.97it/s]
 85%|██████████████████████████████████      | 852/1000 [01:47<00:18,  7.97it/s]
 85%|██████████████████████████████████      | 853/1000 [01:47<00:18,  7.96it/s]
 85%|██████████████████████████████████▏     | 854/1000 [01:48<00:18,  7.95it/s]
 86%|██████████████████████████████████▏     | 855/1000 [01:48<00:18,  7.93it/s]
 86%|██████████████████████████████████▏     | 856/1000 [01:48<00:18,  7.93it/s]
 86%|██████████████████████████████████▎     | 857/1000 [01:48<00:17,  7.95it/s]
 86%|██████████████████████████████████▎     | 858/1000 [01:48<00:17,  7.96it/s]
 86%|██████████████████████████████████▎     | 859/1000 [01:48<00:17,  7.95it/s]
 86%|██████████████████████████████████▍     | 860/1000 [01:48<00:17,  7.94it/s]
 86%|██████████████████████████████████▍     | 861/1000 [01:48<00:17,  7.94it/s]
 86%|██████████████████████████████████▍     | 862/1000 [01:49<00:17,  7.88it/s]
 86%|██████████████████████████████████▌     | 863/1000 [01:49<00:17,  7.84it/s]
 86%|██████████████████████████████████▌     | 864/1000 [01:49<00:17,  7.78it/s]
 86%|██████████████████████████████████▌     | 865/1000 [01:49<00:17,  7.81it/s]
 87%|██████████████████████████████████▋     | 866/1000 [01:49<00:17,  7.83it/s]
 87%|██████████████████████████████████▋     | 867/1000 [01:49<00:16,  7.88it/s]
 87%|██████████████████████████████████▋     | 868/1000 [01:49<00:16,  7.78it/s]
 87%|██████████████████████████████████▊     | 869/1000 [01:49<00:16,  7.75it/s]
 87%|██████████████████████████████████▊     | 870/1000 [01:50<00:18,  7.21it/s]
 87%|██████████████████████████████████▊     | 871/1000 [01:50<00:17,  7.43it/s]
 87%|██████████████████████████████████▉     | 872/1000 [01:50<00:16,  7.56it/s]
 87%|██████████████████████████████████▉     | 873/1000 [01:50<00:16,  7.68it/s]
 87%|██████████████████████████████████▉     | 874/1000 [01:50<00:16,  7.74it/s]
 88%|███████████████████████████████████     | 875/1000 [01:50<00:16,  7.79it/s]
 88%|███████████████████████████████████     | 876/1000 [01:50<00:15,
172.0s 107 7.84it/s]
 88%|███████████████████████████████████     | 877/1000 [01:51<00:15,  7.87it/s]
 88%|███████████████████████████████████     | 878/1000 [01:51<00:15,  7.88it/s]
 88%|███████████████████████████████████▏    | 879/1000 [01:51<00:15,  7.93it/s]
 88%|███████████████████████████████████▏    | 880/1000 [01:51<00:15,  7.86it/s]
 88%|███████████████████████████████████▏    | 881/1000 [01:51<00:15,  7.88it/s]
 88%|███████████████████████████████████▎    | 882/1000 [01:51<00:15,  7.87it/s]
 88%|███████████████████████████████████▎    | 883/1000 [01:51<00:15,  7.78it/s]
 88%|███████████████████████████████████▎    | 884/1000 [01:51<00:14,  7.82it/s]
 88%|███████████████████████████████████▍    | 885/1000 [01:52<00:14,  7.84it/s]
 89%|███████████████████████████████████▍    | 886/1000 [01:52<00:14,  7.86it/s]
 89%|███████████████████████████████████▍    | 887/1000 [01:52<00:14,  7.76it/s]
 89%|███████████████████████████████████▌    | 888/1000 [01:52<00:14,  7.79it/s]
 89%|███████████████████████████████████▌    | 889/1000 [01:52<00:14,  7.83it/s]
 89%|███████████████████████████████████▌    | 890/1000 [01:52<00:13,  7.89it/s]
 89%|███████████████████████████████████▋    | 891/1000 [01:52<00:13,  7.85it/s]
 89%|███████████████████████████████████▋    | 892/1000 [01:52<00:13,  7.89it/s]
 89%|███████████████████████████████████▋    | 893/1000 [01:53<00:13,  7.89it/s]
 89%|███████████████████████████████████▊    | 894/1000 [01:53<00:13,  7.91it/s]
 90%|███████████████████████████████████▊    | 895/1000 [01:53<00:13,  7.86it/s]
 90%|███████████████████████████████████▊    | 896/1000 [01:53<00:13,  7.89it/s]
 90%|███████████████████████████████████▉    | 897/1000 [01:53<00:13,  7.89it/s]
 90%|███████████████████████████████████▉    | 898/1000 [01:53<00:12,  7.93it/s]
 90%|███████████████████████████████████▉    | 899/1000 [01:53<00:13,  7.65it/s]
 90%|████████████████████████████████████    | 900/1000 [01:53<00:12,  7.73it/s]
 90%|████████████████████████████████████    | 901/1000 [01:54<00:12,  7.79it/s]
 90%|████████████████████████████████████    | 902/1000 [01:54<00:12,  7.87it/s]
 90%|████████████████████████████████████    | 903/1000 [01:54<00:12,  7.79it/s]
 90%|████████████████████████████████████▏   | 904/1000 [01:54<00:12,  7.86it/s]
 90%|████████████████████████████████████▏   | 905/1000 [01:54<00:12,  7.89it/s]
 91%|████████████████████████████████████▏   | 906/1000 [01:54<00:11,  7.94it/s]
 91%|████████████████████████████████████▎   | 907/1000 [01:54<00:11,  7.95it/s]
 91%|████████████████████████████████████▎   | 908/1000 [01:54<00:11,  7.95it/s]
 91%|████████████████████████████████████▎   | 909/1000 [01:55<00:11,  7.96it/s]
 91%|████████████████████████████████████▍   | 910/1000 [01:55<00:11,  7.98it/s]
 91%|████████████████████████████████████▍   | 911/1000 [01:55<00:11,  7.84it/s]
 91%|████████████████████████████████████▍   | 912/1000 [01:55<00:11,  7.69it/s]
 91%|████████████████████████████████████▌   | 913/1000 [01:55<00:11,  7.71it/s]
 91%|████████████████████████████████████▌   | 914/1000 [01:55<00:11,  7.74it/s]
 92%|████████████████████████████████████▌   | 915/1000 [01:55<00:10,  7.81it/s]
 92%|████████████████████████████████████▋   | 916/1000 [01:55<00:10,  7.85it/s]
 92%|████████████████████████████████████▋   | 917/1000 [01:56<00:10,  7.90it/s]
 92%|████████████████████████████████████▋   | 918/1000 [01:56<00:10,  7.91it/s]
 92%|████████████████████████████████████▊   | 919/1000 [01:56<00:10,  7.91it/s]
 92%|████████████████████████████████████▊   | 920/1000 [01:56<00:10,  7.93it/s]
 92%|████████████████████████████████████▊   | 921/1000 [01:56<00:09,  7.98it/s]
 92%|████████████████████████████████████▉   | 922/1000 [01:56<00:09,  7.98it/s]
 92%|████████████████████████████████████▉   | 923/1000 [01:56<00:09,  7.94it/s]
 92%|████████████████████████████████████▉   | 924/1000 [01:56<00:09,  7.92it/s]
 92%|█████████████████████████████████████   | 925/1000 [01:57<00:09,  7.92it/s]
 93%|█████████████████████████████████████   | 926/1000 [01:57<00:09,  7.94it/s]
 93%|█████████████████████████████████████   | 927/1000 [01:57<00:09,  7.91it/s]
 93%|█████████████████████████████████████   | 928/1000 [01:57<00:09,  7.92it/s]
 93%|█████████████████████████████████████▏  | 929/1000 [01:57<00:08,  7.93it/s]
 93%|█████████████████████████████████████▏  | 930/1000 [01:57<00:08,  7.91it/s]
 93%|█████████████████████████████████████▏  | 931/1000 [01:57<00:08,  7.92it/s]
 93%|█████████████████████████████████████▎  | 932/1000 [01:57<00:08,  7.93it/s]
 93%|█████████████████████████████████████▎  | 933/1000 [01:58<00:08,  7.88it/s]
 93%|█████████████████████████████████████▎  | 934/1000 [01:58<00:08,  7.84it/s]
 94%|█████████████████████████████████████▍  | 935/1000 [01:58<00:08,  7.83it/s]
 94%|█████████████████████████████████████▍  | 936/1000 [01:58<00:08,  7.84it/s]
 94%|█████████████████████████████████████▍  | 937/1000 [01:58<00:07,  7.89it/s]
 94%|█████████████████████████████████████▌  | 938/1000 [01:58<00:08,  7.65it/s]
 94%|█████████████████████████████████████▌  | 939/1000 [01:58<00:07,  7.80it/s]
 94%|█████████████████████████████████████▌  | 940/1000 [01:59<00:07,  7.87it/s]
 94%|█████████████████████████████████████▋  | 941/1000 [01:59<00:07,  7.94it/s]
 94%|█████████████████████████████████████▋  | 942/1000 [01:59<00:07,  8.01it/s]
 94%|█████████████████████████████████████▋  | 943/1000 [01:59<00:07,  8.00it/s]
 94%|█████████████████████████████████████▊  | 944/1000 [01:59<00:06,  8.04it/s]
 94%|█████████████████████████████████████▊  | 945/1000 [01:59<00:06,  8.03it/s]
 95%|█████████████████████████████████████▊  | 946/1000 [01:59<00:06,  8.02it/s]
 95%|█████████████████████████████████████▉  | 947/1000 [01:59<00:06,  7.72it/s]
 95%|█████████████████████████████████████▉  | 948/1000 [02:00<00:06,  7.78it/s]
 95%|█████████████████████████████████████▉  | 949/1000 [02:00<00:06,  7.35it/s]
 95%|██████████████████████████████████████  | 950/1000 [02:00<00:06,  7.48it/s]
 95%|██████████████████████████████████████  | 951/1000 [02:00<00:06,  7.55it/s]
 95%|██████████████████████████████████████  | 952/1000 [02:00<00:06,  7.55it/s]
 95%|██████████████████████████████████████  | 953/1000 [02:00<00:06,  7.65it/s]
 95%|██████████████████████████████████████▏ | 954/1000 [02:00<00:05,  7.77it/s]
 96%|██████████████████████████████████████▏ | 955/1000 [02:00<00:05,  7.82it/s]
 96%|██████████████████████████████████████▏ | 956/1000 [02:01<00:05,  7.84it/s]
 96%|██████████████████████████████████████▎ | 957/1000 [02:01<00:05,  7.87it/s]
 96%|██████████████████████████████████████▎ | 958/1000 [02:01<00:05,  7.88it/s]
 96%|██████████████████████████████████████▎ | 959/1000 [02:01<00:05,  7.89it/s]
 96%|██████████████████████████████████████▍ | 960/1000 [02:01<00:05,  7.94it/s]
 96%|██████████████████████████████████████▍ | 961/1000 [02:01<00:04,  7.96it/s]
 96%|██████████████████████████████████████▍ | 962/1000 [02:01<00:04,  8.00it/s]
 96%|██████████████████████████████████████▌ | 963/1000 [02:01<00:04,  8.00it/s]
 96%|██████████████████████████████████████▌ | 964/1000 [02:02<00:04,  7.99it/s]
 96%|██████████████████████████████████████▌ | 965/1000 [02:02<00:04,  8.00it/s]
 97%|██████████████████████████████████████▋ | 966/1000 [02:02<00:04,  7.98it/s]
 97%|██████████████████████████████████████▋ | 967/1000 [02:02<00:04,  7.99it/s]
 97%|██████████████████████████████████████▋ | 968/1000 [02:02<00:03,  8.00it/s]
 97%|██████████████████████████████████████▊ | 969/1000 [02:02<00:03,  8.01it/s]
 97%|██████████████████████████████████████▊ | 970/1000 [02:02<00:03,  8.01it/s]
 97%|██████████████████████████████████████▊ | 971/1000 [02:02<00:03,  8.01it/s]
 97%|██████████████████████████████████████▉ | 972/1000 [02:03<00:03,  7.96it/s]
 97%|██████████████████████████████████████▉ | 973/1000 [02:03<00:03,  7.99it/s]
 97%|██████████████████████████████████████▉ | 974/1000 [02:03<00:03,  7.95it/s]
 98%|███████████████████████████████████████ | 975/1000 [02:03<00:03,  7.93it/s]
 98%|███████████████████████████████████████ | 976/1000 [02:03<00:03,  7.93it/s]
 98%|███████████████████████████████████████ | 977/1000 [02:03<00:02,  7.95it/s]
 98%|███████████████████████████████████████ | 978/1000 [02:03<00:02,  7.70it/s]
 98%|███████████████████████████████████████▏| 979/1000 [02:03<00:02,  7.71it/s]
 98%|███████████████████████████████████████▏| 980/1000 [02:04<00:02,  7.77it/s]
 98%|███████████████████████████████████████▏| 981/1000 [02:04<00:02,  7.84it/s]
 98%|███████████████████████████████████████▎| 982/1000 [02:04<00:02,  7.83it/s]
 98%|███████████████████████████████████████▎| 983/1000 [02:04<00:02,  7.88it/s]
 98%|███████████████████████████████████████▎| 984/1000 [02:04<00:02,  7.89it/s]
 98%|███████████████████████████████████████▍| 985/1000 [02:04<00:01,  7.93it/s]
 99%|███████████████████████████████████████▍| 986/1000 [02:04<00:01,  7.94it/s]
 99%|███████████████████████████████████████▍| 987/1000 [02:04<00:01,  7.96it/s]
 99%|███████████████████████████████████████▌| 988/1000 [02:05<00:01,  7.94it/s]
 99%|███████████████████████████████████████▌| 989/1000 [02:05<00:01,  7.96it/s]
 99%|███████████████████████████████████████▌| 990/1000 [02:05<00:01,  7.80it/s]
 99%|███████████████████████████████████████▋| 991/1000 [02:05<00:01,  7.68it/s]
 99%|███████████████████████████████████████▋| 992/1000 [02:05<00:01,  7.75it/s]
 99%|███████████████████████████████████████▋| 993/1000 [02:05<00:00,  7.84it/s]
 99%|███████████████████████████████████████▊| 994/1000 [02:05<00:00,  7.90it/s]
100%|███████████████████████████████████████▊| 995/1000 [02:06<00:00,  7.93it/s]
100%|███████████████████████████████████████▊| 996/1000 [02:06<00:00,  7.95it/s]
100%|███████████████████████████████████████▉| 997/1000 [02:06<00:00,  7.98it/s]
100%|███████████████████████████████████████▉| 998/1000 [02:06<00:00,  7.94it/s]
100%|███████████████████████████████████████▉| 999/1000 [02:06<00:00,  7.97it/s]
100%|███████████████████████████████████████| 1000/1000 [02:06<00:00,  7.97it/s]
100%|███████████████████████████████████████| 1000/1000 [02:06<00:00,  7.90it/s]
191.5s 108 Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.
194.7s 109 test.csv:   0%|                                     | 0.00/79.4M [00:00<?, ?B/s]
test.csv:   0%|                                     | 0.00/79.4M [00:00<?, ?B/s]
test.csv:   0%|                                     | 0.00/79.4M [00:00<?, ?B/s]
test.csv:   0%|                                     | 0.00/79.4M [00:00<?, ?B/s]
test.csv:   0%|                                     | 0.00/79.4M [00:00<?, ?B/s]
test.csv:   0%|                                     | 0.00/79.4M [00:01<?, ?B/s]
test.csv:   0%|                                     | 0.00/79.4M [00:01<?, ?B/s]
test.csv:   0%|                                     | 0.00/79.4M [00:01<?, ?B/s]
test.csv:   0%|                                     | 0.00/79.4M [00:01<?, ?B/s]
test.csv:   0%|                                     | 0.00/79.4M [00:01<?, ?B/s]
test.csv:   0%|                                     | 0.00/79.4M [00:02<?, ?B/s]
test.csv:   0%|                                     | 0.00/79.4M [00:02<?, ?B/s]
test.csv:   0%|                                     | 0.00/79.4M [00:02<?, ?B/s]
test.csv:  84%|████████████████████████▍    | 67.0M/79.4M [00:02<00:00, 336MB/s]
test.csv: 100%|████████████████████████████| 79.4M/79.4M [00:03<00:00, 80.1MB/s]
test.csv: 100%|████████████████████████████| 79.4M/79.4M [00:03<00:00, 24.7MB/s]
198.3s 110 0%|                                                | 0/422786 [00:00<?, ?it/s]
 42%|████████████▌                 | 176839/422786 [00:00<00:00, 1768332.66it/s]
 90%|███████████████████████████   | 381304/422786 [00:00<00:00, 1930833.28it/s]
100%|██████████████████████████████| 422786/422786 [00:00<00:00, 1920711.46it/s]
200.1s 111 Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.
200.1s 112 
200.1s 113 Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.
202.8s 114 Writing finetune_lora.py
226.5s 115 Loading checkpoint from checkpoints/HRM-checkpoint-sudoku-extreme/checkpoint...
226.6s 116 Checkpoint loaded.
226.6s 117 Applying LoRA...
226.6s 118 LoRA applied. Trainable parameters: 558082
226.9s 119 0%|                                                   | 0/250 [00:00<?, ?it/s][Rank 0, World Size 1]: Epoch 0
273.7s 120 W0713 06:46:28.572000 101 torch/_inductor/utils.py:1679] [0/0] Not enough SMs to use max_autotune_gemm mode
280.7s 121 /usr/local/lib/python3.12/dist-packages/torch/_inductor/compile_fx.py:2900: UserWarning: Tesla T4 does not support bfloat16 compilation natively, skipping
280.7s 122 warnings.warn(
325.4s 123 /usr/local/lib/python3.12/dist-packages/torch/_inductor/compile_fx.py:2900: UserWarning: Tesla T4 does not support bfloat16 compilation natively, skipping
325.4s 124 warnings.warn(
325.5s 125 /usr/local/lib/python3.12/dist-packages/torch/_inductor/compile_fx.py:2900: UserWarning: Tesla T4 does not support bfloat16 compilation natively, skipping
325.5s 126 warnings.warn(
353.6s 127 /usr/local/lib/python3.12/dist-packages/torch/_inductor/compile_fx.py:2900: UserWarning: Tesla T4 does not support bfloat16 compilation natively, skipping
353.6s 128 warnings.warn(
353.8s 129 /usr/local/lib/python3.12/dist-packages/torch/_inductor/compile_fx.py:2900: UserWarning: Tesla T4 does not support bfloat16 compilation natively, skipping
353.8s 130 warnings.warn(
353.8s 131 /usr/local/lib/python3.12/dist-packages/torch/_inductor/compile_fx.py:2900: UserWarning: Tesla T4 does not support bfloat16 compilation natively, skipping
353.8s 132 warnings.warn(
354.0s 133 /usr/local/lib/python3.12/dist-packages/torch/_inductor/compile_fx.py:2900: UserWarning: Tesla T4 does not support bfloat16 compilation natively, skipping
354.0s 134 warnings.warn(
357.7s 135 /usr/local/lib/python3.12/dist-packages/torch/_inductor/compile_fx.py:2900: UserWarning: Tesla T4 does not support bfloat16 compilation natively, skipping
357.7s 136 warnings.warn(
358.5s 137 /usr/local/lib/python3.12/dist-packages/torch/_inductor/compile_fx.py:2900: UserWarning: Tesla T4 does not support bfloat16 compilation natively, skipping
358.5s 138 warnings.warn(
359.0s 139 /usr/local/lib/python3.12/dist-packages/torch/_inductor/compile_fx.py:2900: UserWarning: Tesla T4 does not support bfloat16 compilation natively, skipping
359.0s 140 warnings.warn(
359.2s 141 /usr/local/lib/python3.12/dist-packages/torch/_inductor/compile_fx.py:2900: UserWarning: Tesla T4 does not support bfloat16 compilation natively, skipping
359.2s 142 warnings.warn(
359.3s 143 /usr/local/lib/python3.12/dist-packages/torch/_inductor/compile_fx.py:2900: UserWarning: Tesla T4 does not support bfloat16 compilation natively, skipping
359.3s 144 warnings.warn(
359.4s 145 /usr/local/lib/python3.12/dist-packages/torch/_inductor/compile_fx.py:2900: UserWarning: Tesla T4 does not support bfloat16 compilation natively, skipping
359.4s 146 warnings.warn(
360.5s 147 /usr/local/lib/python3.12/dist-packages/torch/_inductor/compile_fx.py:2900: UserWarning: Tesla T4 does not support bfloat16 compilation natively, skipping
360.5s 148 warnings.warn(
361.1s 149 /usr/local/lib/python3.12/dist-packages/torch/_inductor/compile_fx.py:2900: UserWarning: Tesla T4 does not support bfloat16 compilation natively, skipping
361.1s 150 warnings.warn(
361.5s 151 /usr/local/lib/python3.12/dist-packages/torch/_inductor/compile_fx.py:2900: UserWarning: Tesla T4 does not support bfloat16 compilation natively, skipping
361.5s 152 warnings.warn(
362.4s 153 /usr/local/lib/python3.12/dist-packages/torch/_inductor/compile_fx.py:2900: UserWarning: Tesla T4 does not support bfloat16 compilation natively, skipping
362.4s 154 warnings.warn(
43212.4s 155 0%|▏                                       | 1/250 [02:20<9:41:04, 140.02s/it]
  1%|▎                                        | 2/250 [02:20<3:59:00, 57.82s/it]
  1%|▍                                        | 3/250 [02:20<2:09:39, 31.50s/it]
  2%|▋                                        | 4/250 [02:20<1:18:25, 19.13s/it]
  2%|▊                                          | 5/250 [02:20<50:11, 12.29s/it]
  2%|█                                          | 6/250 [02:20<33:13,  8.17s/it]
  3%|█▏                                         | 7/250 [02:21<22:29,  5.55s/it]
  3%|█▍                                         | 8/250 [02:21<15:29,  3.84s/it]
  4%|█▌                                         | 9/250 [02:21<10:48,  2.69s/it]
  4%|█▋                                        | 10/250 [02:21<07:39,  1.91s/it]
  4%|█▊                                        | 11/250 [02:21<05:29,  1.38s/it]
  5%|██                                        | 12/250 [02:21<04:00,  1.01s/it]
  5%|██▏                                       | 13/250 [02:22<02:59,  1.32it/s]
  6%|██▎                                       | 14/250 [02:22<02:16,  1.73it/s]
  6%|██▌                                       | 15/250 [02:22<01:46,  2.20it/s]
  6%|██▋                                       | 16/250 [02:22<01:26,  2.71it/s]
  7%|██▊                                       | 17/250 [02:22<01:11,  3.24it/s]
  7%|███                                       | 18/250 [02:23<01:01,  3.75it/s]
  8%|███▏                                      | 19/250 [02:23<00:54,  4.21it/s]
  8%|███▎                                      | 20/250 [02:23<00:49,  4.61it/s]
  8%|███▌                                      | 21/250 [02:23<00:46,  4.93it/s]
  9%|███▋                                      | 22/250 [02:23<00:43,  5.19it/s]
  9%|███▊                                      | 23/250 [02:23<00:42,  5.39it/s]
 10%|████                                      | 24/250 [02:24<00:40,  5.54it/s]
 10%|████▏                                     | 25/250 [02:24<00:39,  5.66it/s]
 10%|████▎                                     | 26/250 [02:24<00:39,  5.72it/s]
 11%|████▌                                     | 27/250 [02:24<00:38,  5.77it/s]
 11%|████▋                                     | 28/250 [02:24<00:38,  5.76it/s]
 12%|████▊                                     | 29/250 [02:24<00:38,  5.80it/s]
 12%|█████                                     | 30/250 [02:25<00:37,  5.82it/s]
 12%|█████▏                                    | 31/250 [02:25<00:37,  5.84it/s]
 13%|█████▍                                    | 32/250 [02:25<00:37,  5.86it/s]
 13%|█████▌                                    | 33/250 [02:25<00:36,  5.87it/s]
 14%|█████▋                                    | 34/250 [02:25<00:36,  5.88it/s]
 14%|█████▉                                    | 35/250 [02:25<00:36,  5.88it/s]
 14%|██████                                    | 36/250 [02:26<00:36,  5.88it/s]
 15%|██████▏                                   | 37/250 [02:26<00:36,  5.88it/s]
 15%|██████▍                                   | 38/250 [02:26<00:36,  5.89it/s]
 16%|██████▌                                   | 39/250 [02:26<00:35,  5.89it/s]
 16%|██████▋                                   | 40/250 [02:26<00:35,  5.89it/s]
 16%|██████▉                                   | 41/250 [02:26<00:35,  5.89it/s]
 17%|███████                                   | 42/250 [02:27<00:35,  5.90it/s]
 17%|███████▏                                  | 43/250 [02:27<00:35,  5.90it/s]
 18%|███████▍                                  | 44/250 [02:27<00:34,  5.89it/s]
 18%|███████▌                                  | 45/250 [02:27<00:34,  5.89it/s]
 18%|███████▋                                  | 46/250 [02:27<00:34,  5.88it/s]
 19%|███████▉                                  | 47/250 [02:27<00:34,  5.88it/s]
 19%|████████                                  | 48/250 [02:28<00:34,  5.89it/s]
 20%|████████▏                                 | 49/250 [02:28<00:34,  5.88it/s]
 20%|████████▍                                 | 50/250 [02:28<00:34,  5.88it/s]
 20%|████████▌                                 | 51/250 [02:28<00:33,  5.88it/s]
 21%|████████▋                                 | 52/250 [02:28<00:33,  5.87it/s]
 21%|████████▉                                 | 53/250 [02:28<00:33,  5.87it/s]
 22%|█████████                                 | 54/250 [02:29<00:33,  5.87it/s]
 22%|█████████▏                                | 55/250 [02:29<00:33,  5.87it/s]
 22%|█████████▍                                | 56/250 [02:29<00:33,  5.88it/s]
 23%|█████████▌                                | 57/250 [02:29<00:32,  5.88it/s]
 23%|█████████▋                                | 58/250 [02:29<00:32,  5.87it/s]
 24%|█████████▉                                | 59/250 [02:29<00:32,  5.87it/s]
 24%|██████████                                | 60/250 [02:30<00:32,  5.86it/s]
 24%|██████████▏                               | 61/250 [02:30<00:32,  5.86it/s]
 25%|██████████▍                               | 62/250 [02:30<00:32,  5.87it/s]
 25%|██████████▌                               | 63/250 [02:30<00:31,  5.86it/s]
 26%|██████████▊                               | 64/250 [02:30<00:31,  5.85it/s]
 26%|██████████▉                               | 65/250 [02:31<00:31,  5.84it/s]
 26%|███████████                               | 66/250 [02:31<00:31,  5.84it/s]
 27%|███████████▎                              | 67/250 [02:31<00:31,  5.84it/s]
 27%|███████████▍                              | 68/250 [02:31<00:30,  5.87it/s]
 28%|███████████▌                              | 69/250 [02:31<00:30,  5.87it/s]
 28%|███████████▊                              | 70/250 [02:31<00:30,  5.85it/s]
 28%|███████████▉                              | 71/250 [02:32<00:30,  5.87it/s]
 29%|████████████                              | 72/250 [02:32<00:30,  5.85it/s]
 29%|████████████▎                             | 73/250 [02:32<00:30,  5.85it/s]
 30%|████████████▍                             | 74/250 [02:32<00:30,  5.84it/s]
 30%|████████████▌                             | 75/250 [02:32<00:29,  5.84it/s]
 30%|████████████▊                             | 76/250 [02:32<00:29,  5.84it/s]
 31%|████████████▉                             | 77/250 [02:33<00:29,  5.83it/s]
 31%|█████████████                             | 78/250 [02:33<00:29,  5.83it/s]
 32%|█████████████▎                            | 79/250 [02:33<00:29,  5.83it/s]
 32%|█████████████▍                            | 80/250 [02:33<00:29,  5.83it/s]
 32%|█████████████▌                            | 81/250 [02:33<00:28,  5.83it/s]
 33%|█████████████▊                            | 82/250 [02:33<00:28,  5.84it/s]
 33%|█████████████▉                            | 83/250 [02:34<00:28,  5.84it/s]
 34%|██████████████                            | 84/250 [02:34<00:28,  5.84it/s]
 34%|██████████████▎                           | 85/250 [02:34<00:28,  5.83it/s]
 34%|██████████████▍                           | 86/250 [02:34<00:28,  5.81it/s]
 35%|██████████████▌                           | 87/250 [02:34<00:28,  5.78it/s]
 35%|██████████████▊                           | 88/250 [02:34<00:27,  5.79it/s]
 36%|██████████████▉                           | 89/250 [02:35<00:27,  5.80it/s]
 36%|███████████████                           | 90/250 [02:35<00:27,  5.80it/s]
 36%|███████████████▎                          | 91/250 [02:35<00:27,  5.82it/s]
 37%|███████████████▍                          | 92/250 [02:35<00:27,  5.82it/s]
 37%|███████████████▌                          | 93/250 [02:35<00:26,  5.82it/s]
 38%|███████████████▊                          | 94/250 [02:35<00:26,  5.82it/s]
 38%|███████████████▉                          | 95/250 [02:36<00:26,  5.82it/s]
 38%|████████████████▏                         | 96/250 [02:36<00:26,  5.81it/s]
 39%|████████████████▎                         | 97/250 [02:36<00:26,  5.81it/s]
 39%|████████████████▍                         | 98/250 [02:36<00:26,  5.81it/s]
 40%|████████████████▋                         | 99/250 [02:36<00:25,  5.82it/s]
 40%|████████████████▍                        | 100/250 [02:37<00:25,  5.81it/s]
 40%|████████████████▌                        | 101/250 [02:37<00:25,  5.81it/s]
 41%|████████████████▋                        | 102/250 [02:37<00:25,  5.81it/s]
 41%|████████████████▉                        | 103/250 [02:37<00:25,  5.82it/s]
 42%|█████████████████                        | 104/250 [02:37<00:25,  5.81it/s]
 42%|█████████████████▏                       | 105/250 [02:37<00:24,  5.82it/s]
 42%|█████████████████▍                       | 106/250 [02:38<00:24,  5.82it/s]
 43%|█████████████████▌                       | 107/250 [02:38<00:24,  5.82it/s]
 43%|█████████████████▋                       | 108/250 [02:38<00:24,  5.81it/s]
 44%|█████████████████▉                       | 109/250 [02:38<00:24,  5.81it/s]
 44%|██████████████████                       | 110/250 [02:38<00:24,  5.79it/s]
 44%|██████████████████▏                      | 111/250 [02:38<00:23,  5.79it/s]
 45%|██████████████████▎                      | 112/250 [02:39<00:23,  5.78it/s]
 45%|██████████████████▌                      | 113/250 [02:39<00:23,  5.79it/s]
 46%|██████████████████▋                      | 114/250 [02:39<00:23,  5.78it/s]
 46%|██████████████████▊                      | 115/250 [02:39<00:23,  5.78it/s]
 46%|███████████████████                      | 116/250 [02:39<00:23,  5.79it/s]
 47%|███████████████████▏                     | 117/250 [02:39<00:22,  5.78it/s]
 47%|███████████████████▎                     | 118/250 [02:40<00:22,  5.78it/s]
 48%|███████████████████▌                     | 119/250 [02:40<00:22,  5.80it/s]
 48%|███████████████████▋                     | 120/250 [02:40<00:22,  5.81it/s]
 48%|███████████████████▊                     | 121/250 [02:40<00:22,  5.82it/s]
 49%|████████████████████                     | 122/250 [02:40<00:22,  5.82it/s]
 49%|████████████████████▏                    | 123/250 [02:40<00:21,  5.82it/s]
 50%|████████████████████▎                    | 124/250 [02:41<00:21,  5.82it/s]
 50%|████████████████████▌                    | 125/250 [02:41<00:21,  5.83it/s]


In [ ]:
%%writefile evaluate.py
from typing import List
import yaml
import os

import torch
torch.backends.cuda.enable_flash_sdp(False)
torch.backends.cuda.enable_mem_efficient_sdp(False)
import torch.distributed as dist
torch.backends.cuda.enable_flash_sdp(False)
torch.backends.cuda.enable_mem_efficient_sdp(False)

import pydantic
from omegaconf import OmegaConf
from pretrain import PretrainConfig, init_train_state, evaluate, create_dataloader


class EvalConfig(pydantic.BaseModel):
    checkpoint: str
    
    save_outputs: List[str] = ["inputs", "labels", "puzzle_identifiers", "logits", "q_halt_logits", "q_continue_logits"]


def launch():
    eval_cfg = EvalConfig(**OmegaConf.to_container(OmegaConf.from_cli()))  # type: ignore
    
    RANK = 0
    WORLD_SIZE = 1
    # Initialize distributed training if in distributed environment (e.g. torchrun)
    if "LOCAL_RANK" in os.environ:
        # Initialize distributed, default device and dtype
        dist.init_process_group(backend="nccl")

        RANK = dist.get_rank()
        WORLD_SIZE = dist.get_world_size()

        torch.cuda.set_device(int(os.environ["LOCAL_RANK"]))

    with open(os.path.join(os.path.dirname(eval_cfg.checkpoint), "all_config.yaml"), "r") as f:
        config = PretrainConfig(**yaml.safe_load(f))

        config.eval_save_outputs = eval_cfg.save_outputs
        config.checkpoint_path = os.path.dirname(eval_cfg.checkpoint)

    # Dataloader
    train_loader, train_metadata = create_dataloader(config, "train", test_set_mode=False, epochs_per_iter=1, global_batch_size=config.global_batch_size, rank=RANK, world_size=WORLD_SIZE)
    eval_loader,  eval_metadata  = create_dataloader(config, "test", test_set_mode=True, epochs_per_iter=1, global_batch_size=config.global_batch_size, rank=RANK, world_size=WORLD_SIZE)

    # Models
    train_state = init_train_state(config, train_metadata, world_size=WORLD_SIZE)
    # Try unwrap torch.compile
    try:
        train_state.model.load_state_dict(torch.load(eval_cfg.checkpoint, map_location="cuda"), assign=True)
    except:
        train_state.model.load_state_dict({k.removeprefix("_orig_mod."): v for k, v in torch.load(eval_cfg.checkpoint, map_location="cuda").items()}, assign=True)
    
    train_state.step = 0
    ckpt_filename = os.path.basename(eval_cfg.checkpoint)
    if ckpt_filename.startswith("step_"):
        train_state.step = int(ckpt_filename.removeprefix("step_"))

    # Evaluate
    print ("Starting evaluation")
    
    train_state.model.eval()
    metrics = evaluate(config, train_state, eval_loader, eval_metadata, rank=RANK, world_size=WORLD_SIZE)

    if metrics is not None:
        print (metrics)


if __name__ == "__main__":
    launch()



In [ ]:
%%writefile eval_cpu.py
import os, glob, yaml
import torch
import numpy as np

os.environ["DISABLE_COMPILE"] = "1"

from pretrain import PretrainConfig, init_train_state, create_dataloader
from models.losses import IGNORE_LABEL_ID

# Find the latest checkpoint
latest_step = -1
best_path = None
for root, dirs, files in os.walk("checkpoints"):
    for f in files:
        if f.startswith("step_") and "all_preds" not in f:
            try:
                step = int(f.split("_")[1])
                if step > latest_step:
                    latest_step = step
                    best_path = os.path.join(root, f)
            except: pass

if not best_path:
    print("ERROR: No checkpoint found!")
    exit(1)

print(f"Checkpoint: {best_path} (step {latest_step})")

# Load config from checkpoint
with open(os.path.join(os.path.dirname(best_path), "all_config.yaml"), "r") as f:
    config = PretrainConfig(**yaml.safe_load(f))
config.eval_save_outputs = ["logits"]
config.checkpoint_path = os.path.dirname(best_path)

# Create dataloader
eval_loader, eval_metadata = create_dataloader(config, "test", test_set_mode=True, 
    epochs_per_iter=1, global_batch_size=config.global_batch_size, rank=0, world_size=1)

# Create model on CPU
train_state = init_train_state(config, eval_metadata, world_size=1)

# Load checkpoint weights to CPU
try:
    train_state.model.load_state_dict(torch.load(best_path, map_location="cpu", weights_only=True), assign=True)
except:
    train_state.model.load_state_dict(
        {k.removeprefix("_orig_mod."): v for k, v in torch.load(best_path, map_location="cpu", weights_only=True).items()}, 
        assign=True)

train_state.model = train_state.model.float().cpu()
train_state.model.eval()

total_correct = 0
total_cells = 0
total_exact = 0
total_puzzles = 0
max_iters = 16

print("Running evaluation on CPU...")

with torch.inference_mode():
    for set_name, batch, global_batch_size in eval_loader:
        batch = {k: v.cpu().float() if v.is_floating_point() else v.cpu() for k, v in batch.items()}
        
        with torch.device("cpu"):
            carry = train_state.model.initial_carry(batch)
        
        for step_i in range(max_iters):
            try:
                carry, _, metrics, preds, all_finish = train_state.model(
                    carry=carry, batch=batch, return_keys=["logits"])
                if all_finish:
                    break
            except Exception as e:
                print(f"  Iteration {step_i} failed with error: {e}")
                import traceback
                traceback.print_exc()
                exit(1)
        
        labels = batch["labels"]
        logits = preds.get("logits", None)
        
        if logits is not None:
            predictions = torch.argmax(logits, dim=-1)
            mask = labels != IGNORE_LABEL_ID
            
            correct = (predictions == labels) & mask
            total_correct += correct.sum().item()
            total_cells += mask.sum().item()
            
            for i in range(labels.shape[0]):
                puzzle_mask = mask[i]
                if puzzle_mask.sum() > 0:
                    total_puzzles += 1
                    if correct[i].sum() == puzzle_mask.sum():
                        total_exact += 1
        
        break # Just do one batch for debugging

print("=" * 60)
print(f"RESULTS (step {latest_step})")
print("=" * 60)
print(f"Cell Accuracy:    {total_correct}/{total_cells} = {100*total_correct/max(total_cells,1):.2f}%")
print(f"Puzzle Accuracy:  {total_exact}/{total_puzzles} = {100*total_exact/max(total_puzzles,1):.2f}%")



In [ ]:
%%writefile finetune_lora.py
from typing import Optional, Any, Sequence, List
from dataclasses import dataclass
import os
import math
import yaml
import shutil

import torch
torch.backends.cuda.enable_flash_sdp(False)
torch.backends.cuda.enable_mem_efficient_sdp(False)
import torch.distributed as dist
torch.backends.cuda.enable_flash_sdp(False)
torch.backends.cuda.enable_mem_efficient_sdp(False)
from torch import nn
from torch.utils.data import DataLoader

import tqdm
import wandb
import coolname
import hydra
import pydantic
from omegaconf import DictConfig
from torch.optim import AdamW

from puzzle_dataset import PuzzleDataset, PuzzleDatasetConfig, PuzzleDatasetMetadata
from utils.functions import load_model_class, get_model_source_path
from models.sparse_embedding import CastedSparseEmbeddingSignSGD_Distributed
from models.layers import CastedLinear
import torch.nn.functional as F
torch.backends.cuda.enable_flash_sdp(False)
torch.backends.cuda.enable_mem_efficient_sdp(False)

original_casted_linear_forward = CastedLinear.forward

def lora_forward(self, input: torch.Tensor) -> torch.Tensor:
    base_out = original_casted_linear_forward(self, input)
    if hasattr(self, 'lora_A') and hasattr(self, 'lora_B'):
        lora_A_casted = self.lora_A.to(input.dtype)
        lora_B_casted = self.lora_B.to(input.dtype)
        lora_out = F.linear(F.linear(input, lora_A_casted), lora_B_casted) * self.scaling
        return base_out + lora_out.to(base_out.dtype)
    return base_out

CastedLinear.forward = lora_forward

def apply_lora_and_freeze(model, r=8, alpha=16):
    for param in model.parameters():
        param.requires_grad = False
    
    for name, module in model.named_modules():
        if isinstance(module, CastedLinear):
            if "qkv_proj" in name or "o_proj" in name or "gate_up_proj" in name or "down_proj" in name:
                module.lora_A = nn.Parameter(torch.zeros((r, module.weight.shape[1]), device=module.weight.device, dtype=module.weight.dtype))
                module.lora_B = nn.Parameter(torch.zeros((module.weight.shape[0], r), device=module.weight.device, dtype=module.weight.dtype))
                nn.init.kaiming_uniform_(module.lora_A, a=math.sqrt(5))
                nn.init.zeros_(module.lora_B)
                module.scaling = alpha / r
                module.lora_A.requires_grad = True
                module.lora_B.requires_grad = True
                
    for name, param in model.named_parameters():
        if "q_head" in name:
            param.requires_grad = True


class LossConfig(pydantic.BaseModel):
    model_config = pydantic.ConfigDict(extra='allow')
    
    name: str


class ArchConfig(pydantic.BaseModel):
    model_config = pydantic.ConfigDict(extra='allow')

    name: str
    loss: LossConfig


class PretrainConfig(pydantic.BaseModel):
    # Config
    arch: ArchConfig
    load_checkpoint: Optional[str] = None
    # Data
    data_path: str

    # Hyperparams
    global_batch_size: int
    epochs: int

    lr: float
    lr_min_ratio: float
    lr_warmup_steps: int

    weight_decay: float
    beta1: float
    beta2: float

    # Puzzle embedding
    puzzle_emb_lr: float
    puzzle_emb_weight_decay: float

    # Names
    project_name: Optional[str] = None
    run_name: Optional[str] = None
    checkpoint_path: Optional[str] = None

    # Extras
    seed: int = 0
    checkpoint_every_eval: bool = False
    eval_interval: Optional[int] = None
    eval_save_outputs: List[str] = []


@dataclass
class TrainState:
    model: nn.Module
    optimizers: Sequence[torch.optim.Optimizer]
    optimizer_lrs: Sequence[float]
    carry: Any

    step: int
    total_steps: int


def create_dataloader(config: PretrainConfig, split: str, rank: int, world_size: int, **kwargs):
    dataset = PuzzleDataset(PuzzleDatasetConfig(
        seed=config.seed,

        dataset_path=config.data_path,

        rank=rank,
        num_replicas=world_size,
        
        **kwargs
    ), split=split)
    dataloader = DataLoader(
        dataset,
        batch_size=None,

        num_workers=1,
        prefetch_factor=8,

        pin_memory=True,
        persistent_workers=True
    )
    return dataloader, dataset.metadata


def create_model(config: PretrainConfig, train_metadata: PuzzleDatasetMetadata, world_size: int):
    model_cfg = dict(
        **config.arch.__pydantic_extra__,  # type: ignore

        batch_size=config.global_batch_size // world_size,

        vocab_size=train_metadata.vocab_size,
        seq_len=train_metadata.seq_len,
        num_puzzle_identifiers=train_metadata.num_puzzle_identifiers,
        causal=False  # Non-autoregressive
    )

    # Instantiate model with loss head
    model_cls = load_model_class(config.arch.name)
    loss_head_cls = load_model_class(config.arch.loss.name)

    with torch.device("cpu"):
        model: nn.Module = model_cls(model_cfg)
        model = loss_head_cls(model, **config.arch.loss.__pydantic_extra__)  # type: ignore
        if "DISABLE_COMPILE" not in os.environ:
            model = torch.compile(model, dynamic=False)  # type: ignore

        # Broadcast parameters from rank 0
        if world_size > 1:
            with torch.no_grad():
                for param in list(model.parameters()) + list(model.buffers()):
                    dist.broadcast(param, src=0)

    if config.load_checkpoint is not None:
        print(f"Loading checkpoint from {config.load_checkpoint}...")
        state_dict = torch.load(config.load_checkpoint, map_location="cpu")
        clean_state_dict = {k.replace("_orig_mod.", ""): v for k, v in state_dict.items()}
        model.load_state_dict(clean_state_dict, strict=False)
        print("Checkpoint loaded.")

    print("Applying LoRA...")
    apply_lora_and_freeze(model)
    print("LoRA applied. Trainable parameters:", sum(p.numel() for p in model.parameters() if p.requires_grad))

    # Optimizers and lr
    optimizers = [
        CastedSparseEmbeddingSignSGD_Distributed(
            model.model.puzzle_emb.buffers(),  # type: ignore
            
            lr=0,  # Needs to be set by scheduler
            weight_decay=config.puzzle_emb_weight_decay,

            world_size=world_size
        ),
        AdamW(
            model.parameters(),
            lr=0,  # Needs to be set by scheduler
            weight_decay=config.weight_decay,
            betas=(config.beta1, config.beta2)
        )
    ]
    optimizer_lrs = [
        config.puzzle_emb_lr,
        config.lr
    ]

    return model, optimizers, optimizer_lrs


def cosine_schedule_with_warmup_lr_lambda(
    current_step: int, *, base_lr: float, num_warmup_steps: int, num_training_steps: int, min_ratio: float = 0.0, num_cycles: float = 0.5
):
    if current_step < num_warmup_steps:
        return base_lr * float(current_step) / float(max(1, num_warmup_steps))

    progress = float(current_step - num_warmup_steps) / float(max(1, num_training_steps - num_warmup_steps))
    return base_lr * (min_ratio + max(0.0, (1 - min_ratio) * 0.5 * (1.0 + math.cos(math.pi * float(num_cycles) * 2.0 * progress))))


def init_train_state(config: PretrainConfig, train_metadata: PuzzleDatasetMetadata, world_size: int):
    # Estimated total training steps
    total_steps = int(config.epochs * train_metadata.total_groups * train_metadata.mean_puzzle_examples / config.global_batch_size)

    # Model
    model, optimizers, optimizer_lrs = create_model(config, train_metadata, world_size=world_size)

    return TrainState(
        step=0,
        total_steps=total_steps,

        model=model,
        optimizers=optimizers,
        optimizer_lrs=optimizer_lrs,
        carry=None
    )


def save_train_state(config: PretrainConfig, train_state: TrainState):
    # FIXME: Only saved model.
    if config.checkpoint_path is None:
        return

    os.makedirs(config.checkpoint_path, exist_ok=True)
    torch.save(train_state.model.state_dict(), os.path.join(config.checkpoint_path, f"step_{train_state.step}"))


def compute_lr(base_lr: float, config: PretrainConfig, train_state: TrainState):
    return cosine_schedule_with_warmup_lr_lambda(
        current_step=train_state.step,
        base_lr=base_lr,
        num_warmup_steps=round(config.lr_warmup_steps),
        num_training_steps=train_state.total_steps,
        min_ratio=config.lr_min_ratio
    )


def train_batch(config: PretrainConfig, train_state: TrainState, batch: Any, global_batch_size: int, rank: int, world_size: int):
    train_state.step += 1
    if train_state.step > train_state.total_steps:  # At most train_total_steps
        return

    # To device
    batch = {k: v.cpu() for k, v in batch.items()}

    # Init carry if it is None
    if train_state.carry is None:
        with torch.device("cpu"):
            train_state.carry = train_state.model.initial_carry(batch)  # type: ignore

    # Forward
    train_state.carry, loss, metrics, _, _ = train_state.model(carry=train_state.carry, batch=batch, return_keys=[])

    ((1 / global_batch_size) * loss).backward()

    # Allreduce
    if world_size > 1:
        for param in train_state.model.parameters():
            if param.grad is not None:
                dist.all_reduce(param.grad)
            
    # Apply optimizer
    lr_this_step = None    
    for optim, base_lr in zip(train_state.optimizers, train_state.optimizer_lrs):
        lr_this_step = compute_lr(base_lr, config, train_state)

        for param_group in optim.param_groups:
            param_group['lr'] = lr_this_step
            
        optim.step()
        optim.zero_grad()

    # Reduce metrics
    if len(metrics):
        assert not any(v.requires_grad for v in metrics.values())

        metric_keys = list(sorted(metrics.keys()))  # Sort keys to guarantee all processes use the same order.
        # Reduce and reconstruct
        metric_values = torch.stack([metrics[k] for k in metric_keys])
        if world_size > 1:
            dist.reduce(metric_values, dst=0)

        if rank == 0:
            metric_values = metric_values.cpu().numpy()
            reduced_metrics = {k: metric_values[i] for i, k in enumerate(metric_keys)}
            
            # Postprocess
            count = max(reduced_metrics["count"], 1)  # Avoid NaNs
            reduced_metrics = {f"train/{k}": v / (global_batch_size if k.endswith("loss") else count) for k, v in reduced_metrics.items()}

            reduced_metrics["train/lr"] = lr_this_step
            return reduced_metrics


def evaluate(config: PretrainConfig, train_state: TrainState, eval_loader: torch.utils.data.DataLoader, eval_metadata: PuzzleDatasetMetadata, rank: int, world_size: int):
    with torch.no_grad():
        set_ids = {k: idx for idx, k in enumerate(eval_metadata.sets)}
        
        all_preds = {}

        metric_keys = []
        metric_values = None
        metric_global_batch_size = [0 for _ in range(len(set_ids))]
        
        carry = None
        for set_name, batch, global_batch_size in eval_loader:
            # To device
            batch = {k: v.cpu() for k, v in batch.items()}
            with torch.device("cpu"):
                carry = train_state.model.initial_carry(batch)  # type: ignore

            # Forward
            while True:
                carry, _, metrics, preds, all_finish = train_state.model(carry=carry, batch=batch, return_keys=config.eval_save_outputs)
                
                if all_finish:
                    break

            for collection in (batch, preds):
                for k, v in collection.items():
                    if k in config.eval_save_outputs:
                        all_preds.setdefault(k, [])
                        all_preds[k].append(v.cpu())  # Move to CPU for saving GPU memory
                        
            del carry, preds, batch, all_finish

            # Aggregate
            set_id = set_ids[set_name]
            
            if metric_values is None:
                metric_keys = list(sorted(metrics.keys()))  # Sort keys to guarantee all processes use the same order.
                metric_values = torch.zeros((len(set_ids), len(metrics.values())), dtype=torch.float32, device="cpu")
                
            metric_values[set_id] += torch.stack([metrics[k] for k in metric_keys])
            metric_global_batch_size[set_id] += global_batch_size

        if len(all_preds) and config.checkpoint_path is not None:
            all_preds = {k: torch.cat(v, dim=0) for k, v in all_preds.items()}

            os.makedirs(config.checkpoint_path, exist_ok=True)
            torch.save(all_preds, os.path.join(config.checkpoint_path, f"step_{train_state.step}_all_preds.{rank}"))

        # Logging
        # Reduce to rank 0
        if metric_values is not None:
            if world_size > 1:
                dist.reduce(metric_values, dst=0)
            
            if rank == 0:
                reduced_metrics = metric_values.cpu().numpy()
                reduced_metrics = {set_name: {metric_name: reduced_metrics[set_id, metric_id] for metric_id, metric_name in enumerate(metric_keys)}
                                   for set_id, set_name in enumerate(set_ids)}
                
                # Postprocess
                for set_name, metrics in reduced_metrics.items():
                    count = metrics.pop("count")
                    reduced_metrics[set_name] = {k: v / count for k, v in metrics.items()}

                return reduced_metrics


def save_code_and_config(config: PretrainConfig):
    if config.checkpoint_path is None or wandb.run is None:
        return

    os.makedirs(config.checkpoint_path, exist_ok=True)

    # Copy code
    code_list = [
        get_model_source_path(config.arch.name),
        get_model_source_path(config.arch.loss.name)
    ]
    for code_file in code_list:
        if code_file is not None:
            code_name = os.path.basename(code_file)

            shutil.copy(code_file, os.path.join(config.checkpoint_path, code_name))

    # Dump config as yaml
    config_file = os.path.join(config.checkpoint_path, "all_config.yaml")
    with open(config_file, "wt") as f:
        yaml.dump(config.model_dump(), f)

    # Log code
    wandb.run.log_code(config.checkpoint_path)


def load_synced_config(hydra_config: DictConfig, rank: int, world_size: int) -> PretrainConfig:
    objects = [None]
    if rank == 0:
        config = PretrainConfig(**hydra_config)  # type: ignore

        # Naming
        if config.project_name is None:
            config.project_name = f"{os.path.basename(config.data_path).capitalize()} ACT-torch"
        if config.run_name is None:
            config.run_name = f"{config.arch.name.split('@')[-1]} {coolname.generate_slug(2)}"
        if config.checkpoint_path is None:
            config.checkpoint_path = os.path.join("checkpoints", config.project_name, config.run_name)

        objects = [config]

    if world_size > 1:
        dist.broadcast_object_list(objects, src=0)

    return objects[0]  # type: ignore


@hydra.main(config_path="config", config_name="cfg_pretrain", version_base=None)
def launch(hydra_config: DictConfig):
    RANK = 0
    WORLD_SIZE = 1

    # Initialize distributed training if in distributed environment (e.g. torchrun)
    if "LOCAL_RANK" in os.environ:
        # Initialize distributed, default device and dtype
        dist.init_process_group(backend="nccl")

        RANK = dist.get_rank()
        WORLD_SIZE = dist.get_world_size()

        #torch.cuda.set_device(int(os.environ["LOCAL_RANK"]))
        
    # Load sync'ed config
    config = load_synced_config(hydra_config, rank=RANK, world_size=WORLD_SIZE)

    # Seed RNGs to ensure consistency
    torch.random.manual_seed(config.seed + RANK)

    # Dataset
    train_epochs_per_iter = config.eval_interval if config.eval_interval is not None else config.epochs
    total_iters = config.epochs // train_epochs_per_iter

    assert config.epochs % train_epochs_per_iter == 0, "Eval interval must be a divisor of total epochs."

    train_loader, train_metadata = create_dataloader(config, "train", test_set_mode=False, epochs_per_iter=train_epochs_per_iter, global_batch_size=config.global_batch_size, rank=RANK, world_size=WORLD_SIZE)
    eval_loader,  eval_metadata  = create_dataloader(config, "test", test_set_mode=True, epochs_per_iter=1, global_batch_size=config.global_batch_size, rank=RANK, world_size=WORLD_SIZE)

    # Train state
    train_state = init_train_state(config, train_metadata, world_size=WORLD_SIZE)

    # Progress bar and logger
    progress_bar = None
    if RANK == 0:
        progress_bar = tqdm.tqdm(total=train_state.total_steps)

        wandb.init(project=config.project_name, name=config.run_name, config=config.model_dump(), settings=wandb.Settings(_disable_stats=True))  # type: ignore
        wandb.log({"num_params": sum(x.numel() for x in train_state.model.parameters())}, step=0)
        save_code_and_config(config)

    # Training Loop
    for _iter_id in range(total_iters):
        print (f"[Rank {RANK}, World Size {WORLD_SIZE}]: Epoch {_iter_id * train_epochs_per_iter}")

        ############ Train Iter
        train_state.model.train()
        for set_name, batch, global_batch_size in train_loader:
            metrics = train_batch(config, train_state, batch, global_batch_size, rank=RANK, world_size=WORLD_SIZE)

            if RANK == 0 and metrics is not None:
                wandb.log(metrics, step=train_state.step)
                progress_bar.update(train_state.step - progress_bar.n)  # type: ignore

        ############ Checkpointing (save BEFORE evaluation to prevent data loss)
        if RANK == 0 and (config.checkpoint_every_eval or (_iter_id == total_iters - 1)):
            save_train_state(config, train_state)

        ############ Evaluation (wrapped in try/except for T4 GPU stability)
        try:
            train_state.model.eval()
            metrics = evaluate(config, train_state, eval_loader, eval_metadata, rank=RANK, world_size=WORLD_SIZE)

            if RANK == 0 and metrics is not None:
                wandb.log(metrics, step=train_state.step)
                print(f"[Eval at step {train_state.step}] {metrics}")
        except Exception as e:
            print(f"WARNING: Evaluation failed at step {train_state.step}: {e}")
            print("Checkpoint was already saved. Training will continue.")

    # finalize
    if dist.is_initialized():
        dist.destroy_process_group()
    wandb.finish()


if __name__ == "__main__":
    launch()



In [ ]:
%%writefile fix_9x9_notebook.py
"""
Fix the 9x9 fine-tuning notebook with the same bug fixes that solved the 6x6 CUDA crash.

Root cause: During evaluation, the last test batch gets padded with blank_identifier_id=1,
but CastedSparseEmbedding only has num_puzzle_identifiers entries (could be 1).
Index 1 is out of bounds, triggering a CUDA device-side assert.

This script applies the fix to kaggle_finetune_9x9.ipynb.
"""
import json

with open('kaggle_finetune_9x9.ipynb', 'r', encoding='utf-8') as f:
    nb = json.load(f)

for cell in nb['cells']:
    if cell['cell_type'] != 'code':
        continue
    source = ''.join(cell['source'])
    
    # Fix 1: num_puzzle_identifiers OOB bug in create_model
    if 'num_puzzle_identifiers=train_metadata.num_puzzle_identifiers,' in source:
        cell['source'] = [
            line.replace(
                'num_puzzle_identifiers=train_metadata.num_puzzle_identifiers,',
                'num_puzzle_identifiers=max(train_metadata.num_puzzle_identifiers, train_metadata.blank_identifier_id + 1),'
            ) for line in cell['source']
        ]
        print("Fixed: num_puzzle_identifiers OOB bug")

with open('kaggle_finetune_9x9.ipynb', 'w', encoding='utf-8') as f:
    json.dump(nb, f, indent=1)

print("Done! kaggle_finetune_9x9.ipynb has been patched.")



In [ ]:
%%writefile fix_cuda.py
import os

with open('finetune_lora.py', 'r', encoding='utf-8') as f:
    content = f.read()

content = content.replace('"cuda"', '"cpu"')
content = content.replace('.cuda()', '.cpu()')
content = content.replace('torch.cuda.set_device', '#torch.cuda.set_device')

with open('finetune_lora.py', 'w', encoding='utf-8') as f:
    f.write(content)



In [ ]:
%%writefile fix_nbs.py
import json

def fix_6x6():
    with open('kaggle_finetune_6x6.ipynb', 'r', encoding='utf-8') as f:
        nb = json.load(f)
    
    for cell in nb['cells']:
        if cell['cell_type'] == 'code':
            # Fix pip install
            new_source = []
            for line in cell['source']:
                if '!pip install -r requirements.txt' in line:
                    new_source.append("!pip install setuptools wheel\n")
                    new_source.append("!pip install einops tqdm pydantic wandb omegaconf hydra-core huggingface_hub peft argdantic coolname --quiet\n")
                elif 'from argdantic import ArgParser' in line:
                    new_source.append("import argparse\n")
                elif 'from pydantic import BaseModel' in line:
                    pass
                elif 'cli = ArgParser()' in line:
                    pass
                elif 'class DataProcessConfig' in line:
                    pass
                elif 'output_dir: str' in line or 'num_train: int' in line or 'num_test: int' in line or 'holes: int' in line:
                    pass
                elif 'def convert_subset(set_name, config):' in line:
                    new_source.append("def convert_subset(set_name, output_dir, num_train, num_test, holes):\n")
                elif 'config.num_train' in line:
                    new_source.append(line.replace('config.num_train', 'num_train').replace('config.num_test', 'num_test'))
                elif 'config.holes' in line:
                    new_source.append(line.replace('config.holes', 'holes'))
                elif 'config.output_dir' in line:
                    new_source.append(line.replace('config.output_dir', 'output_dir'))
                elif '@cli.command(singleton=True)' in line:
                    pass
                elif 'def preprocess_data(config: DataProcessConfig):' in line:
                    pass
                elif 'convert_subset("train", config)' in line:
                    new_source.append("    convert_subset(\"train\", args.output_dir, args.num_train, args.num_test, args.holes)\n")
                elif 'convert_subset("test", config)' in line:
                    new_source.append("    convert_subset(\"test\", args.output_dir, args.num_train, args.num_test, args.holes)\n")
                elif 'cli()' in line:
                    new_source.append("    parser = argparse.ArgumentParser()\n")
                    new_source.append("    parser.add_argument(\"--output-dir\", type=str, default=\"data/sudoku-6x6\")\n")
                    new_source.append("    parser.add_argument(\"--num-train\", type=int, default=1000)\n")
                    new_source.append("    parser.add_argument(\"--num-test\", type=int, default=100)\n")
                    new_source.append("    parser.add_argument(\"--holes\", type=int, default=20)\n")
                    new_source.append("    args = parser.parse_args()\n")
                    new_source.append("    convert_subset(\"train\", args.output_dir, args.num_train, args.num_test, args.holes)\n")
                    new_source.append("    convert_subset(\"test\", args.output_dir, args.num_train, args.num_test, args.holes)\n")
                else:
                    new_source.append(line)
            cell['source'] = new_source

    with open('kaggle_finetune_6x6.ipynb', 'w', encoding='utf-8') as f:
        json.dump(nb, f, indent=1)

def fix_9x9():
    with open('kaggle_finetune_9x9.ipynb', 'r', encoding='utf-8') as f:
        nb = json.load(f)
    
    for cell in nb['cells']:
        if cell['cell_type'] == 'code':
            new_source = []
            for line in cell['source']:
                if '!pip install -r requirements.txt' in line:
                    new_source.append("!pip install setuptools wheel\n")
                    new_source.append("!pip install einops tqdm pydantic wandb omegaconf hydra-core huggingface_hub peft argdantic coolname --quiet\n")
                else:
                    new_source.append(line)
            cell['source'] = new_source

    with open('kaggle_finetune_9x9.ipynb', 'w', encoding='utf-8') as f:
        json.dump(nb, f, indent=1)

fix_6x6()
fix_9x9()



In [ ]:
%%writefile fix_nbs2.py
import json

def fix_6x6():
    with open('kaggle_finetune_6x6.ipynb', 'r', encoding='utf-8') as f:
        nb = json.load(f)
    
    for cell in nb['cells']:
        if cell['cell_type'] == 'code':
            source = ''.join(cell['source'])
            
            # 1. Change git clone url
            if 'https://github.com/sapientinc/HRM.git' in source:
                source = source.replace('https://github.com/sapientinc/HRM.git', 'https://github.com/jagan25-mj/NHRT.git')
            
            # 2. Fix the dataset builder recursion bug
            if 'dataset/build_6x6_sudoku_dataset.py' in source:
                # We will entirely rewrite the source of this cell to be perfectly correct
                new_source = '''%%writefile dataset/build_6x6_sudoku_dataset.py
import os
import json
import random
import numpy as np
from tqdm import tqdm
import argparse

try:
    from dataset.common import PuzzleDatasetMetadata
except ImportError:
    from common import PuzzleDatasetMetadata

def is_valid(board, r, c, val):
    if val in board[r]: return False
    if val in board[:, c]: return False
    br, bc = 2 * (r // 2), 3 * (c // 3)
    if val in board[br:br+2, bc:bc+3]: return False
    return True

def solve(board):
    for r in range(6):
        for c in range(6):
            if board[r, c] == 0:
                nums = list(range(1, 7))
                random.shuffle(nums)
                for val in nums:
                    if is_valid(board, r, c, val):
                        board[r, c] = val
                        if solve(board): return True
                        board[r, c] = 0
                return False
    return True

def generate_6x6_board():
    board = np.zeros((6, 6), dtype=int)
    solve(board)
    return board

def generate_puzzles(num_puzzles, num_holes):
    inputs, labels = [], []
    for _ in tqdm(range(num_puzzles), desc="Generating 6x6 puzzles"):
        solution = generate_6x6_board()
        puzzle = solution.copy()
        cells = [(r, c) for r in range(6) for c in range(6)]
        random.shuffle(cells)
        for r, c in cells[:num_holes]:
            puzzle[r, c] = 0
        inputs.append(puzzle)
        labels.append(solution)
    return inputs, labels

def convert_subset(set_name, output_dir, num_train, num_test, holes):
    num_puzzles = num_train if set_name == "train" else num_test
    inputs, labels = generate_puzzles(num_puzzles, holes)
    results = {k: [] for k in ["inputs", "labels", "puzzle_identifiers", "puzzle_indices", "group_indices"]}
    results["puzzle_indices"].append(0)
    results["group_indices"].append(0)
    for i, (inp, out) in enumerate(zip(inputs, labels)):
        results["inputs"].append(inp.flatten() + 1)
        results["labels"].append(out.flatten() + 1)
        results["puzzle_indices"].append(i + 1)
        results["puzzle_identifiers"].append(0)
        results["group_indices"].append(i + 1)
    results = {
        "inputs": np.array(results["inputs"], dtype=np.int32),
        "labels": np.array(results["labels"], dtype=np.int32),
        "group_indices": np.array(results["group_indices"], dtype=np.int32),
        "puzzle_indices": np.array(results["puzzle_indices"], dtype=np.int32),
        "puzzle_identifiers": np.array(results["puzzle_identifiers"], dtype=np.int32),
    }
    metadata = PuzzleDatasetMetadata(
        seq_len=36, vocab_size=8, pad_id=0, ignore_label_id=0,
        blank_identifier_id=1, num_puzzle_identifiers=1,
        total_groups=len(results["group_indices"]) - 1,
        mean_puzzle_examples=1, sets=["all"]
    )
    save_dir = os.path.join(output_dir, set_name)
    os.makedirs(save_dir, exist_ok=True)
    with open(os.path.join(save_dir, "dataset.json"), "w") as f:
        json.dump(metadata.model_dump(), f)
    for k, v in results.items():
        np.save(os.path.join(save_dir, f"all__{k}.npy"), v)
    with open(os.path.join(output_dir, "identifiers.json"), "w") as f:
        json.dump(["<blank>"], f)

if __name__ == "__main__":
    parser = argparse.ArgumentParser()
    parser.add_argument("--output-dir", type=str, default="data/sudoku-6x6")
    parser.add_argument("--num-train", type=int, default=1000)
    parser.add_argument("--num-test", type=int, default=100)
    parser.add_argument("--holes", type=int, default=20)
    args = parser.parse_args()
    
    convert_subset("train", args.output_dir, args.num_train, args.num_test, args.holes)
    convert_subset("test", args.output_dir, args.num_train, args.num_test, args.holes)
'''
                source = new_source
                
            cell['source'] = [line + '\n' for line in source.split('\n')]
            # Fix double newlines at the end of lines
            cell['source'] = [line.replace('\n\n', '\n') for line in cell['source']]
            if cell['source'] and cell['source'][-1].endswith('\n'):
                cell['source'][-1] = cell['source'][-1][:-1]
                
    with open('kaggle_finetune_6x6.ipynb', 'w', encoding='utf-8') as f:
        json.dump(nb, f, indent=1)

def fix_9x9():
    with open('kaggle_finetune_9x9.ipynb', 'r', encoding='utf-8') as f:
        nb = json.load(f)
    
    for cell in nb['cells']:
        if cell['cell_type'] == 'code':
            source = ''.join(cell['source'])
            if 'https://github.com/sapientinc/HRM.git' in source:
                source = source.replace('https://github.com/sapientinc/HRM.git', 'https://github.com/jagan25-mj/NHRT.git')
                
            cell['source'] = [line + '\n' for line in source.split('\n')]
            # Fix double newlines
            cell['source'] = [line.replace('\n\n', '\n') for line in cell['source']]
            if cell['source'] and cell['source'][-1].endswith('\n'):
                cell['source'][-1] = cell['source'][-1][:-1]

    with open('kaggle_finetune_9x9.ipynb', 'w', encoding='utf-8') as f:
        json.dump(nb, f, indent=1)

fix_6x6()
fix_9x9()



In [ ]:
%%writefile fix_nbs3.py
import json
with open('kaggle_finetune_6x6.ipynb', 'r', encoding='utf-8') as f:
    nb = json.load(f)

for i, cell in enumerate(nb['cells']):
    if i == 6:
        # Restore the dataset generation cell
        cell['source'] = [
            '!python dataset/build_6x6_sudoku_dataset.py --output-dir data/sudoku-6x6 --num-train 1000 --num-test 100 --holes 20\n',
            '\n',
            'import os\n',
            'print("\\nDataset created:")\n',
            'for split in ["train", "test"]:\n',
            '    split_dir = f"data/sudoku-6x6/{split}"\n',
            '    if os.path.exists(split_dir):\n',
            '        files = os.listdir(split_dir)\n',
            '        print(f"  {split}: {files}\\n")\n'
        ]

with open('kaggle_finetune_6x6.ipynb', 'w', encoding='utf-8') as f:
    json.dump(nb, f, indent=1)



In [ ]:
%%writefile generate_6x6_sudoku_dataset_chatgpt.py

import random
import pandas as pd

BASE = [
    [1,2,3,4,5,6],
    [4,5,6,1,2,3],
    [2,3,4,5,6,1],
    [5,6,1,2,3,4],
    [3,4,5,6,1,2],
    [6,1,2,3,4,5],
]

def permute(grid):
    nums=list(range(1,7))
    random.shuffle(nums)
    mp={i+1:nums[i] for i in range(6)}
    return [[mp[v] for v in row] for row in grid]

def make_puzzle(sol, clues=18):
    cells=[(r,c) for r in range(6) for c in range(6)]
    random.shuffle(cells)
    keep=set(cells[:clues])
    puz=[]
    for r in range(6):
        row=[]
        for c in range(6):
            row.append(sol[r][c] if (r,c) in keep else 0)
        puz.append(row)
    return puz

def encode(g):
    return "".join(str(x) for row in g for x in row)

def generate_dataset(n=10000, clues=18):
    rows=[]
    for i in range(n):
        sol=permute(BASE)
        puz=make_puzzle(sol, clues)
        rows.append({
            "id": i,
            "puzzle": encode(puz),
            "solution": encode(sol),
            "clues": clues
        })
    return pd.DataFrame(rows)

if __name__ == "__main__":
    df=generate_dataset(100000,18)
    df.to_csv("sudoku6x6_train.csv", index=False)
    print("saved", len(df), "samples")



In [ ]:
%%writefile make_nb.py
import json

with open("finetune_lora.py", "r", encoding="utf-8") as f:
    script_content = f.read()

# Revert cpu back to cuda for Kaggle T4 GPUs
script_content = script_content.replace('"cpu"', '"cuda"').replace('.cpu()', '.cuda()').replace('.cuda().numpy()', '.cpu().numpy()')
script_lines = [line + "\n" for line in script_content.split("\n")]

notebook = {
    "cells": [
        {
            "cell_type": "markdown",
            "metadata": {},
            "source": [
                "# HRM LoRA Fine-Tuning on Kaggle (T4 GPU)\n",
                "This notebook sets up the environment, downloads the pre-trained HRM checkpoint, builds the Sudoku Extreme dataset, and runs a custom parameter-efficient LoRA fine-tuning script on a T4 GPU."
            ]
        },
        {
            "cell_type": "code",
            "execution_count": None,
            "metadata": {},
            "outputs": [],
            "source": [
                "!git clone https://github.com/sapientinc/HRM.git\n",
                "%cd HRM\n",
                "!sed -i '/adam-atan2/d' requirements.txt\n",
                "!pip install -r requirements.txt peft huggingface_hub wandb"
            ]
        },
        {
            "cell_type": "code",
            "execution_count": None,
            "metadata": {},
            "outputs": [],
            "source": [
                "# Patch models/layers.py to bypass flash_attn requirement and use PyTorch native SDPA\n",
                "patch_code = '''\\\n",
                "\n",
                "import torch.nn.functional as F\n",
                "def flash_attn_func(q, k, v, causal=False):\n",
                "    q = q.transpose(1, 2)\n",
                "    k = k.transpose(1, 2)\n",
                "    v = v.transpose(1, 2)\n",
                "    out = F.scaled_dot_product_attention(q.contiguous(), k.contiguous(), v.contiguous(), is_causal=causal)\n",
                "    return out.transpose(1, 2).contiguous()\n",
                "'''\n",
                "with open('models/layers.py', 'r') as f:\n",
                "    content = f.read()\n",
                "content = content.replace('from flash_attn_interface import flash_attn_func', 'pass')\n",
                "content = content.replace('from flash_attn import flash_attn_func', 'pass')\n",
                "content += patch_code\n",
                "with open('models/layers.py', 'w') as f:\n",
                "    f.write(content)\n"
            ]
        },
        {
            "cell_type": "code",
            "execution_count": None,
            "metadata": {},
            "outputs": [],
            "source": [
                "# 1. Build the Sudoku Extreme Dataset\n",
                "!sed -i 's/if set_name == \"train\" and config.subsample_size is not None:/if config.subsample_size is not None:/g' dataset/build_sudoku_dataset.py\n",
                "!python dataset/build_sudoku_dataset.py --output-dir data/sudoku-extreme-1k-aug-1000 --subsample-size 1000 --num-aug 1000"
            ]
        },
        {
            "cell_type": "code",
            "execution_count": None,
            "metadata": {},
            "outputs": [],
            "source": [
                "# 2. Download the pre-trained HRM checkpoint\n",
                "import os\n",
                "from huggingface_hub import snapshot_download\n",
                "os.makedirs('checkpoints/HRM-checkpoint-sudoku-extreme', exist_ok=True)\n",
                "snapshot_download(repo_id='sapientinc/HRM-checkpoint-sudoku-extreme', local_dir='checkpoints/HRM-checkpoint-sudoku-extreme')"
            ]
        },
        {
            "cell_type": "code",
            "execution_count": None,
            "metadata": {},
            "outputs": [],
            "source": ["%%writefile finetune_lora.py\n"] + script_lines
        },
        {
            "cell_type": "code",
            "execution_count": None,
            "metadata": {},
            "outputs": [],
            "source": [
                "# 4. Run the Fine-Tuning on T4 GPU\n",
                "!WANDB_MODE=disabled python finetune_lora.py data_path=data/sudoku-extreme-1k-aug-1000 epochs=2 eval_interval=1 lr=1e-5 puzzle_emb_lr=1e-5 weight_decay=1.0 puzzle_emb_weight_decay=1.0 global_batch_size=8 +load_checkpoint=checkpoints/HRM-checkpoint-sudoku-extreme/checkpoint"
            ]
        },
        {
            "cell_type": "markdown",
            "metadata": {},
            "source": [
                "### Expected Results\n",
                "- **Trainable Parameters**: ~558,082 (only ~2% of the base 27M parameters are trainable due to LoRA injection).\n",
                "- **VRAM Usage**: Fits comfortably within the 16GB VRAM of a Kaggle T4 GPU thanks to frozen base weights and a batch size of 8.\n",
                "- **Performance**: The script uses `torch.compile` and mixed precision on the T4 GPU, leading to fast iteration times.\n",
                "- **Outputs**: Checkpoints will be saved in the `checkpoints/` directory at the end of each evaluation epoch, with the ACT (Adaptive Computation Time) halting logic dynamically fine-tuned for the augmented dataset."
            ]
        }
    ],
    "metadata": {
        "accelerator": "GPU",
        "language_info": {
            "name": "python"
        }
    },
    "nbformat": 4,
    "nbformat_minor": 4
}

with open("kaggle_finetune_lora.ipynb", "w", encoding="utf-8") as f:
    json.dump(notebook, f, indent=2)

print("Created kaggle_finetune_lora.ipynb")



In [ ]:
%%writefile pretrain.py
from typing import Optional, Any, Sequence, List
from dataclasses import dataclass
import os
import math
import yaml
import shutil

import torch
torch.backends.cuda.enable_flash_sdp(False)
torch.backends.cuda.enable_mem_efficient_sdp(False)
import torch.distributed as dist
torch.backends.cuda.enable_flash_sdp(False)
torch.backends.cuda.enable_mem_efficient_sdp(False)
from torch import nn
from torch.utils.data import DataLoader

import tqdm
import wandb
import coolname
import hydra
import pydantic
from omegaconf import DictConfig
from torch.optim import AdamW

from puzzle_dataset import PuzzleDataset, PuzzleDatasetConfig, PuzzleDatasetMetadata
from utils.functions import load_model_class, get_model_source_path
from models.sparse_embedding import CastedSparseEmbeddingSignSGD_Distributed


class LossConfig(pydantic.BaseModel):
    model_config = pydantic.ConfigDict(extra='allow')
    
    name: str


class ArchConfig(pydantic.BaseModel):
    model_config = pydantic.ConfigDict(extra='allow')

    name: str
    loss: LossConfig


class PretrainConfig(pydantic.BaseModel):
    # Config
    arch: ArchConfig
    # Data
    data_path: str

    # Hyperparams
    global_batch_size: int
    epochs: int

    lr: float
    lr_min_ratio: float
    lr_warmup_steps: int

    weight_decay: float
    beta1: float
    beta2: float

    # Puzzle embedding
    puzzle_emb_lr: float
    puzzle_emb_weight_decay: float

    # Names
    project_name: Optional[str] = None
    run_name: Optional[str] = None
    checkpoint_path: Optional[str] = None

    # Extras
    seed: int = 0
    checkpoint_every_eval: bool = False
    eval_interval: Optional[int] = None
    eval_save_outputs: List[str] = []


@dataclass
class TrainState:
    model: nn.Module
    optimizers: Sequence[torch.optim.Optimizer]
    optimizer_lrs: Sequence[float]
    carry: Any

    step: int
    total_steps: int


def create_dataloader(config: PretrainConfig, split: str, rank: int, world_size: int, **kwargs):
    dataset = PuzzleDataset(PuzzleDatasetConfig(
        seed=config.seed,

        dataset_path=config.data_path,

        rank=rank,
        num_replicas=world_size,
        
        **kwargs
    ), split=split)
    dataloader = DataLoader(
        dataset,
        batch_size=None,

        num_workers=1,
        prefetch_factor=8,

        pin_memory=True,
        persistent_workers=True
    )
    return dataloader, dataset.metadata


def create_model(config: PretrainConfig, train_metadata: PuzzleDatasetMetadata, world_size: int):
    model_cfg = dict(
        **config.arch.__pydantic_extra__,  # type: ignore

        batch_size=config.global_batch_size // world_size,

        vocab_size=train_metadata.vocab_size,
        seq_len=train_metadata.seq_len,
        num_puzzle_identifiers=max(train_metadata.num_puzzle_identifiers, train_metadata.blank_identifier_id + 1),
        causal=False  # Non-autoregressive
    )

    # Instantiate model with loss head
    model_cls = load_model_class(config.arch.name)
    loss_head_cls = load_model_class(config.arch.loss.name)

    with torch.device("cuda"):
        model: nn.Module = model_cls(model_cfg)
        model = loss_head_cls(model, **config.arch.loss.__pydantic_extra__)  # type: ignore
        if "DISABLE_COMPILE" not in os.environ:
            model = torch.compile(model, dynamic=False)  # type: ignore

        # Broadcast parameters from rank 0
        if world_size > 1:
            with torch.no_grad():
                for param in list(model.parameters()) + list(model.buffers()):
                    dist.broadcast(param, src=0)

    # Optimizers and lr
    optimizers = [
        CastedSparseEmbeddingSignSGD_Distributed(
            model.model.puzzle_emb.buffers(),  # type: ignore
            
            lr=0,  # Needs to be set by scheduler
            weight_decay=config.puzzle_emb_weight_decay,

            world_size=world_size
        ),
        AdamW(
            model.parameters(),
            lr=0,  # Needs to be set by scheduler
            weight_decay=config.weight_decay,
            betas=(config.beta1, config.beta2)
        )
    ]
    optimizer_lrs = [
        config.puzzle_emb_lr,
        config.lr
    ]

    return model, optimizers, optimizer_lrs


def cosine_schedule_with_warmup_lr_lambda(
    current_step: int, *, base_lr: float, num_warmup_steps: int, num_training_steps: int, min_ratio: float = 0.0, num_cycles: float = 0.5
):
    if current_step < num_warmup_steps:
        return base_lr * float(current_step) / float(max(1, num_warmup_steps))

    progress = float(current_step - num_warmup_steps) / float(max(1, num_training_steps - num_warmup_steps))
    return base_lr * (min_ratio + max(0.0, (1 - min_ratio) * 0.5 * (1.0 + math.cos(math.pi * float(num_cycles) * 2.0 * progress))))


def init_train_state(config: PretrainConfig, train_metadata: PuzzleDatasetMetadata, world_size: int):
    # Estimated total training steps
    total_steps = int(config.epochs * train_metadata.total_groups * train_metadata.mean_puzzle_examples / config.global_batch_size)

    # Model
    model, optimizers, optimizer_lrs = create_model(config, train_metadata, world_size=world_size)

    return TrainState(
        step=0,
        total_steps=total_steps,

        model=model,
        optimizers=optimizers,
        optimizer_lrs=optimizer_lrs,
        carry=None
    )


def save_train_state(config: PretrainConfig, train_state: TrainState):
    # FIXME: Only saved model.
    if config.checkpoint_path is None:
        return

    os.makedirs(config.checkpoint_path, exist_ok=True)
    torch.save(train_state.model.state_dict(), os.path.join(config.checkpoint_path, f"step_{train_state.step}"))


def compute_lr(base_lr: float, config: PretrainConfig, train_state: TrainState):
    return cosine_schedule_with_warmup_lr_lambda(
        current_step=train_state.step,
        base_lr=base_lr,
        num_warmup_steps=round(config.lr_warmup_steps),
        num_training_steps=train_state.total_steps,
        min_ratio=config.lr_min_ratio
    )


def train_batch(config: PretrainConfig, train_state: TrainState, batch: Any, global_batch_size: int, rank: int, world_size: int):
    train_state.step += 1
    if train_state.step > train_state.total_steps:  # At most train_total_steps
        return

    # To device
    batch = {k: v.cuda() for k, v in batch.items()}

    # Init carry if it is None
    if train_state.carry is None:
        with torch.device("cuda"):
            train_state.carry = train_state.model.initial_carry(batch)  # type: ignore

    # Forward
    train_state.carry, loss, metrics, _, _ = train_state.model(carry=train_state.carry, batch=batch, return_keys=[])

    ((1 / global_batch_size) * loss).backward()

    # Allreduce
    if world_size > 1:
        for param in train_state.model.parameters():
            if param.grad is not None:
                dist.all_reduce(param.grad)
            
    # Apply optimizer
    lr_this_step = None    
    for optim, base_lr in zip(train_state.optimizers, train_state.optimizer_lrs):
        lr_this_step = compute_lr(base_lr, config, train_state)

        for param_group in optim.param_groups:
            param_group['lr'] = lr_this_step
            
        optim.step()
        optim.zero_grad()

    # Reduce metrics
    if len(metrics):
        assert not any(v.requires_grad for v in metrics.values())

        metric_keys = list(sorted(metrics.keys()))  # Sort keys to guarantee all processes use the same order.
        # Reduce and reconstruct
        metric_values = torch.stack([metrics[k] for k in metric_keys])
        if world_size > 1:
            dist.reduce(metric_values, dst=0)

        if rank == 0:
            metric_values = metric_values.cpu().numpy()
            reduced_metrics = {k: metric_values[i] for i, k in enumerate(metric_keys)}
            
            # Postprocess
            count = max(reduced_metrics["count"], 1)  # Avoid NaNs
            reduced_metrics = {f"train/{k}": v / (global_batch_size if k.endswith("loss") else count) for k, v in reduced_metrics.items()}

            reduced_metrics["train/lr"] = lr_this_step
            return reduced_metrics


def evaluate(config: PretrainConfig, train_state: TrainState, eval_loader: torch.utils.data.DataLoader, eval_metadata: PuzzleDatasetMetadata, rank: int, world_size: int):
    with torch.no_grad():
        set_ids = {k: idx for idx, k in enumerate(eval_metadata.sets)}
        
        all_preds = {}

        metric_keys = []
        metric_values = None
        metric_global_batch_size = [0 for _ in range(len(set_ids))]
        
        carry = None
        for set_name, batch, global_batch_size in eval_loader:
            # To device
            batch = {k: v.cuda() for k, v in batch.items()}
            with torch.device("cuda"):
                carry = train_state.model.initial_carry(batch)  # type: ignore

            # Forward
            while True:
                carry, _, metrics, preds, all_finish = train_state.model(carry=carry, batch=batch, return_keys=config.eval_save_outputs)
                
                if all_finish:
                    break

            for collection in (batch, preds):
                for k, v in collection.items():
                    if k in config.eval_save_outputs:
                        all_preds.setdefault(k, [])
                        all_preds[k].append(v.cpu())  # Move to CPU for saving GPU memory
                        
            del carry, preds, batch, all_finish

            # Aggregate
            set_id = set_ids[set_name]
            
            if metric_values is None:
                metric_keys = list(sorted(metrics.keys()))  # Sort keys to guarantee all processes use the same order.
                metric_values = torch.zeros((len(set_ids), len(metrics.values())), dtype=torch.float32, device="cuda")
                
            metric_values[set_id] += torch.stack([metrics[k] for k in metric_keys])
            metric_global_batch_size[set_id] += global_batch_size

        if len(all_preds) and config.checkpoint_path is not None:
            all_preds = {k: torch.cat(v, dim=0) for k, v in all_preds.items()}

            os.makedirs(config.checkpoint_path, exist_ok=True)
            torch.save(all_preds, os.path.join(config.checkpoint_path, f"step_{train_state.step}_all_preds.{rank}"))

        # Logging
        # Reduce to rank 0
        if metric_values is not None:
            if world_size > 1:
                dist.reduce(metric_values, dst=0)
            
            if rank == 0:
                reduced_metrics = metric_values.cpu().numpy()
                reduced_metrics = {set_name: {metric_name: reduced_metrics[set_id, metric_id] for metric_id, metric_name in enumerate(metric_keys)}
                                   for set_id, set_name in enumerate(set_ids)}
                
                # Postprocess
                for set_name, metrics in reduced_metrics.items():
                    count = metrics.pop("count")
                    reduced_metrics[set_name] = {k: v / count for k, v in metrics.items()}

                return reduced_metrics


def save_code_and_config(config: PretrainConfig):
    if config.checkpoint_path is None or wandb.run is None:
        return

    os.makedirs(config.checkpoint_path, exist_ok=True)

    # Copy code
    code_list = [
        get_model_source_path(config.arch.name),
        get_model_source_path(config.arch.loss.name)
    ]
    for code_file in code_list:
        if code_file is not None:
            code_name = os.path.basename(code_file)

            shutil.copy(code_file, os.path.join(config.checkpoint_path, code_name))

    # Dump config as yaml
    config_file = os.path.join(config.checkpoint_path, "all_config.yaml")
    with open(config_file, "wt") as f:
        yaml.dump(config.model_dump(), f)

    # Log code
    wandb.run.log_code(config.checkpoint_path)


def load_synced_config(hydra_config: DictConfig, rank: int, world_size: int) -> PretrainConfig:
    objects = [None]
    if rank == 0:
        config = PretrainConfig(**hydra_config)  # type: ignore

        # Naming
        if config.project_name is None:
            config.project_name = f"{os.path.basename(config.data_path).capitalize()} ACT-torch"
        if config.run_name is None:
            config.run_name = f"{config.arch.name.split('@')[-1]} {coolname.generate_slug(2)}"
        if config.checkpoint_path is None:
            config.checkpoint_path = os.path.join("checkpoints", config.project_name, config.run_name)

        objects = [config]

    if world_size > 1:
        dist.broadcast_object_list(objects, src=0)

    return objects[0]  # type: ignore


@hydra.main(config_path="config", config_name="cfg_pretrain", version_base=None)
def launch(hydra_config: DictConfig):
    RANK = 0
    WORLD_SIZE = 1

    # Initialize distributed training if in distributed environment (e.g. torchrun)
    if "LOCAL_RANK" in os.environ:
        # Initialize distributed, default device and dtype
        dist.init_process_group(backend="nccl")

        RANK = dist.get_rank()
        WORLD_SIZE = dist.get_world_size()

        torch.cuda.set_device(int(os.environ["LOCAL_RANK"]))
        
    # Load sync'ed config
    config = load_synced_config(hydra_config, rank=RANK, world_size=WORLD_SIZE)

    # Seed RNGs to ensure consistency
    torch.random.manual_seed(config.seed + RANK)

    # Dataset
    train_epochs_per_iter = config.eval_interval if config.eval_interval is not None else config.epochs
    total_iters = config.epochs // train_epochs_per_iter

    assert config.epochs % train_epochs_per_iter == 0, "Eval interval must be a divisor of total epochs."

    train_loader, train_metadata = create_dataloader(config, "train", test_set_mode=False, epochs_per_iter=train_epochs_per_iter, global_batch_size=config.global_batch_size, rank=RANK, world_size=WORLD_SIZE)
    eval_loader,  eval_metadata  = create_dataloader(config, "test", test_set_mode=True, epochs_per_iter=1, global_batch_size=config.global_batch_size, rank=RANK, world_size=WORLD_SIZE)

    # Train state
    train_state = init_train_state(config, train_metadata, world_size=WORLD_SIZE)

    # Progress bar and logger
    progress_bar = None
    if RANK == 0:
        progress_bar = tqdm.tqdm(total=train_state.total_steps)

        wandb.init(project=config.project_name, name=config.run_name, config=config.model_dump(), settings=wandb.Settings(_disable_stats=True))  # type: ignore
        wandb.log({"num_params": sum(x.numel() for x in train_state.model.parameters())}, step=0)
        save_code_and_config(config)

    # Training Loop
    for _iter_id in range(total_iters):
        print (f"[Rank {RANK}, World Size {WORLD_SIZE}]: Epoch {_iter_id * train_epochs_per_iter}")

        ############ Train Iter
        train_state.model.train()
        for set_name, batch, global_batch_size in train_loader:
            metrics = train_batch(config, train_state, batch, global_batch_size, rank=RANK, world_size=WORLD_SIZE)

            if RANK == 0 and metrics is not None:
                wandb.log(metrics, step=train_state.step)
                progress_bar.update(train_state.step - progress_bar.n)  # type: ignore

        ############ Checkpointing (save BEFORE evaluation to prevent data loss)
        if RANK == 0 and (config.checkpoint_every_eval or (_iter_id == total_iters - 1)):
            save_train_state(config, train_state)

        ############ Evaluation (wrapped in try/except for T4 GPU stability)
        try:
            train_state.model.eval()
            metrics = evaluate(config, train_state, eval_loader, eval_metadata, rank=RANK, world_size=WORLD_SIZE)

            if RANK == 0 and metrics is not None:
                wandb.log(metrics, step=train_state.step)
                print(f"[Eval at step {train_state.step}] {metrics}")
        except Exception as e:
            print(f"WARNING: Evaluation failed at step {train_state.step}: {e}")
            print("Checkpoint was already saved. Training will continue.")

    # finalize
    if dist.is_initialized():
        dist.destroy_process_group()
    wandb.finish()


if __name__ == "__main__":
    launch()



In [ ]:
%%writefile puzzle_dataset.py
import os
import json

import numpy as np
import pydantic

import torch
from torch.utils.data import IterableDataset, get_worker_info

from models.losses import IGNORE_LABEL_ID
from dataset.common import PuzzleDatasetMetadata


def _sample_batch(rng: np.random.Generator, group_order: np.ndarray, puzzle_indices: np.ndarray, group_indices: np.ndarray, start_index: int, global_batch_size: int):
    # Pack examples into a full batch
    batch = []
    batch_puzzle_indices = []
    current_size = 0

    while (start_index < group_order.size) and (current_size < global_batch_size):
        # Pick a group and a puzzle from that group
        group_id = group_order[start_index]
        puzzle_id = rng.integers(group_indices[group_id], group_indices[group_id + 1])
        start_index += 1

        # Get range of the puzzle
        puzzle_start = puzzle_indices[puzzle_id]
        puzzle_size = int(puzzle_indices[puzzle_id + 1] - puzzle_start)

        append_size = min(puzzle_size, global_batch_size - current_size)

        # Put into batch
        batch_puzzle_indices.append(np.full(append_size, puzzle_id, dtype=np.int32))
        batch.append(puzzle_start + np.random.choice(puzzle_size, append_size, replace=False))

        current_size += append_size

    return start_index, np.concatenate(batch), np.concatenate(batch_puzzle_indices)


class PuzzleDatasetConfig(pydantic.BaseModel):
    seed: int
    dataset_path: str
    global_batch_size: int
    test_set_mode: bool

    epochs_per_iter: int  # Batch X epochs in an iteration to reduce overhead.

    rank: int
    num_replicas: int


class PuzzleDataset(IterableDataset):
    def __init__(self, config: PuzzleDatasetConfig, split: str = "train"):
        super().__init__()
        self.config = config
        self.split = split
        self.metadata = self._load_metadata()
        
        # Checks
        assert self.config.global_batch_size % self.config.num_replicas == 0, f"Global batch size {self.config.global_batch_size} must be multiples of nodes {self.config.num_replicas}."
        self.local_batch_size = self.config.global_batch_size // self.config.num_replicas

        # State
        self._data = None
        self._iters = 0

    def _load_metadata(self) -> PuzzleDatasetMetadata:
        with open(os.path.join(self.config.dataset_path, self.split, "dataset.json"), "r") as f:
            return PuzzleDatasetMetadata(**json.load(f))

    def _lazy_load_dataset(self):
        if self._data is not None:
            return

        field_mmap_modes = {
            "inputs": "r",
            "labels": "r",

            # Keep indices in memory
            "puzzle_identifiers": None,
            "puzzle_indices": None,
            "group_indices": None
        }

        # Load data
        self._data = {}
        for set_name in self.metadata.sets:
            # Load subset
            self._data[set_name] = {
                field_name: np.load(os.path.join(self.config.dataset_path, self.split, f"{set_name}__{field_name}.npy"), mmap_mode=mmap_mode)
                for field_name, mmap_mode in field_mmap_modes.items()
            }

    def _collate_batch(self, batch):
        # Convert dtype
        batch = {k: v.astype(np.int32) for k, v in batch.items()}

        # Convert ignore label IDs
        if self.metadata.ignore_label_id is not None:
            batch["labels"][batch["labels"] == self.metadata.ignore_label_id] = IGNORE_LABEL_ID

        # Pad
        if batch["puzzle_identifiers"].size < self.local_batch_size:
            pad_size = self.local_batch_size - batch["puzzle_identifiers"].size

            pad_values = {
                "inputs": self.metadata.pad_id,
                "labels": IGNORE_LABEL_ID,

                "puzzle_identifiers": self.metadata.blank_identifier_id
            }
            batch = {k: np.pad(v, ((0, pad_size), ) + ((0, 0), ) * (v.ndim - 1), constant_values=pad_values[k]) for k, v in batch.items()}

        # To tensor
        return {k: torch.from_numpy(v) for k, v in batch.items()}
    
    def _iter_test(self):
        for set_name, dataset in self._data.items():  # type: ignore
            total_examples = len(dataset["inputs"])

            # Load examples one by one
            start_index = 0
            while start_index < total_examples:
                # Compute indices
                end_index = min(total_examples, start_index + self.config.global_batch_size)
                
                local_start = start_index + self.config.rank * self.local_batch_size
                local_end   = min(start_index + (self.config.rank + 1) * self.local_batch_size, end_index)
                
                # Get batch of examples, and also puzzle IDs
                puzzle_indices = []
                puzzle_index = np.searchsorted(dataset["puzzle_indices"], local_start, side="right") - 1
                for i in range(local_start, local_end):
                    while puzzle_index + 1 < len(dataset["puzzle_indices"]) and i >= dataset["puzzle_indices"][puzzle_index + 1]:
                        puzzle_index += 1

                    puzzle_indices.append(puzzle_index)
                
                batch = self._collate_batch({
                    "inputs": dataset["inputs"][local_start: local_end],
                    "labels": dataset["labels"][local_start: local_end],
                    "puzzle_identifiers": dataset["puzzle_identifiers"][puzzle_indices]
                })

                yield set_name, batch, end_index - start_index
                
                # Advance to next batch
                start_index += self.config.global_batch_size

    def _iter_train(self):
        for set_name, dataset in self._data.items():  # type: ignore
            # Increase epoch count
            self._iters += 1

            # Randomly shuffle groups
            rng = np.random.Generator(np.random.Philox(seed=self.config.seed + self._iters))

            group_order = np.concatenate([rng.permutation(dataset["group_indices"].size - 1) for _i in range(self.config.epochs_per_iter)])
            start_index = 0
            
            while start_index < group_order.size:
                start_index, batch_indices, batch_puzzle_indices = _sample_batch(
                    rng,
                    group_order=group_order,
                    puzzle_indices=dataset["puzzle_indices"],
                    group_indices=dataset["group_indices"],
                    start_index=start_index,
                    global_batch_size=self.config.global_batch_size,
                )

                # Select current rank and collate
                global_effective_batch_size = batch_puzzle_indices.size  # Global effective batch size, excluding pads

                # Drop last batch
                if global_effective_batch_size < self.config.global_batch_size:
                    break

                batch_indices        = batch_indices       [self.config.rank * self.local_batch_size: (self.config.rank + 1) * self.local_batch_size]
                batch_puzzle_indices = batch_puzzle_indices[self.config.rank * self.local_batch_size: (self.config.rank + 1) * self.local_batch_size]
                batch = self._collate_batch({
                    "inputs": dataset["inputs"][batch_indices],
                    "labels": dataset["labels"][batch_indices],
                    "puzzle_identifiers": dataset["puzzle_identifiers"][batch_puzzle_indices]
                })

                yield set_name, batch, global_effective_batch_size
                
    def __iter__(self):
        worker_info = get_worker_info()
        assert worker_info is None or worker_info.num_workers == 1, "Multithreaded data loading is not currently supported."
        
        self._lazy_load_dataset()
        
        # Iterate using specified mode
        if self.config.test_set_mode:
            yield from self._iter_test()
        else:
            yield from self._iter_train()



In [ ]:
%%writefile requirements.txt
torch
einops
tqdm
coolname
pydantic
argdantic
wandb
omegaconf
hydra-core
huggingface_hub



In [ ]:
%%writefile update_eval_nb.py
import json
import os

with open('kaggle_evaluate_6x6.ipynb', 'r') as f:
    nb = json.load(f)

# Cell 0: add pip install
nb['cells'][0]['source'] = [
    "!git clone https://github.com/jagan25-mj/NHRT.git\n",
    "%cd NHRT\n",
    "!pip install -r requirements.txt"
]

# Cell 2: add debug info
new_source = [
    "import os\n",
    "import glob\n",
    "\n",
    "# Let's see what is inside /kaggle/input/ to debug!\n",
    "print('Contents of /kaggle/input:')\n",
    "os.system('find /kaggle/input -maxdepth 5')\n",
    "\n",
    "checkpoint_files = glob.glob('/kaggle/input/**/step_*', recursive=True)\n",
    "checkpoint_files = [f for f in checkpoint_files if 'all_preds' not in f]\n",
    "\n",
    "if not checkpoint_files:\n",
    "    print('\\nNo checkpoints found! Make sure you attached the output of the previous notebook.')\n",
    "    print('If it was an interactive session, you might need to run the training in a \"Save Version -> Run All\" commit to save the outputs permanently.')\n",
    "else:\n",
    "    def extract_step(f):\n",
    "        basename = os.path.basename(f)\n",
    "        return int(basename.replace('step_', ''))\n",
    "    \n",
    "    latest_checkpoint = max(checkpoint_files, key=extract_step)\n",
    "    print(f'\\nFound checkpoint: {latest_checkpoint}')\n",
    "    \n",
    "    cmd = f'python evaluate.py checkpoint=\"{latest_checkpoint}\"'\n",
    "    print(f'Running: {cmd}')\n",
    "    os.system(cmd)\n"
]
nb['cells'][2]['source'] = new_source

with open('kaggle_evaluate_6x6.ipynb', 'w') as f:
    json.dump(nb, f, indent=1)



In [ ]:
%%writefile checkpoints/HRM-checkpoint-sudoku-extreme/all_config.yaml
arch:
  H_cycles: 2
  H_layers: 4
  L_cycles: 2
  L_layers: 4
  expansion: 4
  halt_exploration_prob: 0.1
  halt_max_steps: 16
  hidden_size: 512
  loss:
    loss_type: stablemax_cross_entropy
    name: losses@ACTLossHead
  name: hrm.hrm_act_v1@HierarchicalReasoningModel_ACTV1
  num_heads: 8
  pos_encodings: rope
  puzzle_emb_ndim: 512
beta1: 0.9
beta2: 0.95
checkpoint_every_eval: true
checkpoint_path: checkpoints/Sudoku-extreme-1k-aug-1000/HierarchicalReasoningModel_ACTV1 manipulative-peacock
data_path: data/sudoku-extreme-1k-aug-1000
epochs: 40000
eval_interval: 4000
eval_save_outputs: []
global_batch_size: 768
lr: 0.0001
lr_min_ratio: 1.0
lr_warmup_steps: 2000
project_name: Sudoku-extreme-1k-aug-1000
puzzle_emb_lr: 0.0001
puzzle_emb_weight_decay: 1.0
run_name: HierarchicalReasoningModel_ACTV1 manipulative-peacock
seed: 0
weight_decay: 1.0



In [ ]:
%%writefile checkpoints/Sudoku-extreme-1k-aug-1000 ACT-torch/HierarchicalReasoningModel_ACTV1 precise-cricket/all_config.yaml
arch:
  H_cycles: 2
  H_layers: 4
  L_cycles: 2
  L_layers: 4
  expansion: 4
  halt_exploration_prob: 0.1
  halt_max_steps: 16
  hidden_size: 512
  loss:
    loss_type: stablemax_cross_entropy
    name: losses@ACTLossHead
  name: hrm.hrm_act_v1@HierarchicalReasoningModel_ACTV1
  num_heads: 8
  pos_encodings: rope
  puzzle_emb_ndim: 512
beta1: 0.9
beta2: 0.95
checkpoint_every_eval: true
checkpoint_path: checkpoints\Sudoku-extreme-1k-aug-1000 ACT-torch\HierarchicalReasoningModel_ACTV1
  precise-cricket
data_path: data/sudoku-extreme-1k-aug-1000
epochs: 2
eval_interval: 1
eval_save_outputs: []
global_batch_size: 8
load_checkpoint: checkpoints/HRM-checkpoint-sudoku-extreme/checkpoint
lr: 1.0e-05
lr_min_ratio: 1.0
lr_warmup_steps: 2000
project_name: Sudoku-extreme-1k-aug-1000 ACT-torch
puzzle_emb_lr: 1.0e-05
puzzle_emb_weight_decay: 1.0
run_name: HierarchicalReasoningModel_ACTV1 precise-cricket
seed: 0
weight_decay: 1.0



In [ ]:
%%writefile checkpoints/Sudoku-extreme-1k-aug-1000 ACT-torch/HierarchicalReasoningModel_ACTV1 precise-cricket/hrm_act_v1.py
from typing import Tuple, List, Dict, Optional
from dataclasses import dataclass
import math

import torch
import torch.nn.functional as F
from torch import nn
from pydantic import BaseModel

from models.common import trunc_normal_init_
from models.layers import rms_norm, SwiGLU, Attention, RotaryEmbedding, CosSin, CastedEmbedding, CastedLinear
from models.sparse_embedding import CastedSparseEmbedding


@dataclass
class HierarchicalReasoningModel_ACTV1InnerCarry:
    z_H: torch.Tensor
    z_L: torch.Tensor


@dataclass
class HierarchicalReasoningModel_ACTV1Carry:
    inner_carry: HierarchicalReasoningModel_ACTV1InnerCarry
    
    steps: torch.Tensor
    halted: torch.Tensor
    
    current_data: Dict[str, torch.Tensor]


class HierarchicalReasoningModel_ACTV1Config(BaseModel):
    batch_size: int
    seq_len: int
    puzzle_emb_ndim: int = 0
    num_puzzle_identifiers: int
    vocab_size: int

    H_cycles: int
    L_cycles: int

    H_layers: int
    L_layers: int

    # Transformer config
    hidden_size: int
    expansion: float
    num_heads: int
    pos_encodings: str

    rms_norm_eps: float = 1e-5
    rope_theta: float = 10000.0
    
    # Halting Q-learning config
    halt_max_steps: int
    halt_exploration_prob: float

    forward_dtype: str = "bfloat16"


class HierarchicalReasoningModel_ACTV1Block(nn.Module):
    def __init__(self, config: HierarchicalReasoningModel_ACTV1Config) -> None:
        super().__init__()

        self.self_attn = Attention(
            hidden_size=config.hidden_size,
            head_dim=config.hidden_size // config.num_heads,
            num_heads=config.num_heads,
            num_key_value_heads=config.num_heads,
            causal=False
        )
        self.mlp = SwiGLU(
            hidden_size=config.hidden_size,
            expansion=config.expansion,
        )
        self.norm_eps = config.rms_norm_eps

    def forward(self, cos_sin: CosSin, hidden_states: torch.Tensor) -> torch.Tensor:
        # Post Norm
        # Self Attention
        hidden_states = rms_norm(hidden_states + self.self_attn(cos_sin=cos_sin, hidden_states=hidden_states), variance_epsilon=self.norm_eps)
        # Fully Connected
        hidden_states = rms_norm(hidden_states + self.mlp(hidden_states), variance_epsilon=self.norm_eps)
        return hidden_states


class HierarchicalReasoningModel_ACTV1ReasoningModule(nn.Module):
    def __init__(self, layers: List[HierarchicalReasoningModel_ACTV1Block]):
        super().__init__()

        self.layers = torch.nn.ModuleList(layers)

    def forward(self, hidden_states: torch.Tensor, input_injection: torch.Tensor, **kwargs) -> torch.Tensor:
        # Input injection (add)
        hidden_states = hidden_states + input_injection
        # Layers
        for layer in self.layers:
            hidden_states = layer(hidden_states=hidden_states, **kwargs)

        return hidden_states


class HierarchicalReasoningModel_ACTV1_Inner(nn.Module):
    def __init__(self, config: HierarchicalReasoningModel_ACTV1Config) -> None:
        super().__init__()
        self.config = config
        self.forward_dtype = getattr(torch, self.config.forward_dtype)

        # I/O
        self.embed_scale  = math.sqrt(self.config.hidden_size)
        embed_init_std = 1.0 / self.embed_scale

        self.embed_tokens = CastedEmbedding(self.config.vocab_size, self.config.hidden_size, init_std=embed_init_std, cast_to=self.forward_dtype)
        self.lm_head      = CastedLinear(self.config.hidden_size, self.config.vocab_size, bias=False)
        self.q_head       = CastedLinear(self.config.hidden_size, 2, bias=True)

        self.puzzle_emb_len = -(self.config.puzzle_emb_ndim // -self.config.hidden_size)  # ceil div
        if self.config.puzzle_emb_ndim > 0:
            # Zero init puzzle embeddings
            self.puzzle_emb = CastedSparseEmbedding(self.config.num_puzzle_identifiers, self.config.puzzle_emb_ndim,
                                                    batch_size=self.config.batch_size, init_std=0, cast_to=self.forward_dtype)

        # LM Blocks
        if self.config.pos_encodings == "rope":
            self.rotary_emb = RotaryEmbedding(dim=self.config.hidden_size // self.config.num_heads,
                                              max_position_embeddings=self.config.seq_len + self.puzzle_emb_len,
                                              base=self.config.rope_theta)
        elif self.config.pos_encodings == "learned":
            self.embed_pos = CastedEmbedding(self.config.seq_len + self.puzzle_emb_len, self.config.hidden_size, init_std=embed_init_std, cast_to=self.forward_dtype)
        else:
            raise NotImplementedError()

        # Reasoning Layers
        self.H_level = HierarchicalReasoningModel_ACTV1ReasoningModule(layers=[HierarchicalReasoningModel_ACTV1Block(self.config) for _i in range(self.config.H_layers)])
        self.L_level = HierarchicalReasoningModel_ACTV1ReasoningModule(layers=[HierarchicalReasoningModel_ACTV1Block(self.config) for _i in range(self.config.L_layers)])
        
        # Initial states
        self.H_init = nn.Buffer(trunc_normal_init_(torch.empty(self.config.hidden_size, dtype=self.forward_dtype), std=1), persistent=True)
        self.L_init = nn.Buffer(trunc_normal_init_(torch.empty(self.config.hidden_size, dtype=self.forward_dtype), std=1), persistent=True)

        # Q head special init
        # Init Q to (almost) zero for faster learning during bootstrapping
        with torch.no_grad():
            self.q_head.weight.zero_()
            self.q_head.bias.fill_(-5)  # type: ignore

    def _input_embeddings(self, input: torch.Tensor, puzzle_identifiers: torch.Tensor):
        # Token embedding
        embedding = self.embed_tokens(input.to(torch.int32))

        # Puzzle embeddings
        if self.config.puzzle_emb_ndim > 0:
            puzzle_embedding = self.puzzle_emb(puzzle_identifiers)
            
            pad_count = self.puzzle_emb_len * self.config.hidden_size - puzzle_embedding.shape[-1]
            if pad_count > 0:
                puzzle_embedding = F.pad(puzzle_embedding, (0, pad_count))

            embedding = torch.cat((puzzle_embedding.view(-1, self.puzzle_emb_len, self.config.hidden_size), embedding), dim=-2)

        # Position embeddings
        if self.config.pos_encodings == "learned":
            # scale by 1/sqrt(2) to maintain forward variance
            embedding = 0.707106781 * (embedding + self.embed_pos.embedding_weight.to(self.forward_dtype))

        # Scale
        return self.embed_scale * embedding

    def empty_carry(self, batch_size: int):
        return HierarchicalReasoningModel_ACTV1InnerCarry(
            z_H=torch.empty(batch_size, self.config.seq_len + self.puzzle_emb_len, self.config.hidden_size, dtype=self.forward_dtype),
            z_L=torch.empty(batch_size, self.config.seq_len + self.puzzle_emb_len, self.config.hidden_size, dtype=self.forward_dtype),
        )
        
    def reset_carry(self, reset_flag: torch.Tensor, carry: HierarchicalReasoningModel_ACTV1InnerCarry):
        return HierarchicalReasoningModel_ACTV1InnerCarry(
            z_H=torch.where(reset_flag.view(-1, 1, 1), self.H_init, carry.z_H),
            z_L=torch.where(reset_flag.view(-1, 1, 1), self.L_init, carry.z_L),
        )

    def forward(self, carry: HierarchicalReasoningModel_ACTV1InnerCarry, batch: Dict[str, torch.Tensor]) -> Tuple[HierarchicalReasoningModel_ACTV1InnerCarry, torch.Tensor, Tuple[torch.Tensor, torch.Tensor]]:
        seq_info = dict(
            cos_sin=self.rotary_emb() if hasattr(self, "rotary_emb") else None,
        )

        # Input encoding
        input_embeddings = self._input_embeddings(batch["inputs"], batch["puzzle_identifiers"])

        # Forward iterations
        with torch.no_grad():
            z_H, z_L = carry.z_H, carry.z_L

            for _H_step in range(self.config.H_cycles):
                for _L_step in range(self.config.L_cycles):
                    if not ((_H_step == self.config.H_cycles - 1) and (_L_step == self.config.L_cycles - 1)):
                        z_L = self.L_level(z_L, z_H + input_embeddings, **seq_info)

                if not (_H_step == self.config.H_cycles - 1):
                    z_H = self.H_level(z_H, z_L, **seq_info)

        assert not z_H.requires_grad and not z_L.requires_grad

        # 1-step grad
        z_L = self.L_level(z_L, z_H + input_embeddings, **seq_info)
        z_H = self.H_level(z_H, z_L, **seq_info)

        # LM Outputs
        new_carry = HierarchicalReasoningModel_ACTV1InnerCarry(z_H=z_H.detach(), z_L=z_L.detach())  # New carry no grad
        output = self.lm_head(z_H)[:, self.puzzle_emb_len:]

        # Q head
        q_logits = self.q_head(z_H[:, 0]).to(torch.float32)
        
        return new_carry, output, (q_logits[..., 0], q_logits[..., 1])


class HierarchicalReasoningModel_ACTV1(nn.Module):
    """ACT wrapper."""

    def __init__(self, config_dict: dict):
        super().__init__()
        self.config = HierarchicalReasoningModel_ACTV1Config(**config_dict)
        self.inner = HierarchicalReasoningModel_ACTV1_Inner(self.config)

    @property
    def puzzle_emb(self):
        return self.inner.puzzle_emb

    def initial_carry(self, batch: Dict[str, torch.Tensor]):
        batch_size = batch["inputs"].shape[0]

        return HierarchicalReasoningModel_ACTV1Carry(
            inner_carry=self.inner.empty_carry(batch_size),  # Empty is expected, it will be reseted in first pass as all sequences are halted.
            
            steps=torch.zeros((batch_size, ), dtype=torch.int32),
            halted=torch.ones((batch_size, ), dtype=torch.bool),  # Default to halted
            
            current_data={k: torch.empty_like(v) for k, v in batch.items()}
        )
        
    def forward(self, carry: HierarchicalReasoningModel_ACTV1Carry, batch: Dict[str, torch.Tensor]) -> Tuple[HierarchicalReasoningModel_ACTV1Carry, Dict[str, torch.Tensor]]:
        # Update data, carry (removing halted sequences)
        new_inner_carry = self.inner.reset_carry(carry.halted, carry.inner_carry)
        
        new_steps = torch.where(carry.halted, 0, carry.steps)

        new_current_data = {k: torch.where(carry.halted.view((-1, ) + (1, ) * (batch[k].ndim - 1)), batch[k], v) for k, v in carry.current_data.items()}

        # Forward inner model
        new_inner_carry, logits, (q_halt_logits, q_continue_logits) = self.inner(new_inner_carry, new_current_data)

        outputs = {
            "logits": logits,
            "q_halt_logits": q_halt_logits,
            "q_continue_logits": q_continue_logits
        }
        
        with torch.no_grad():
            # Step
            new_steps = new_steps + 1
            is_last_step = new_steps >= self.config.halt_max_steps
            
            halted = is_last_step

            # if training, and ACT is enabled
            if self.training and (self.config.halt_max_steps > 1):
                # Halt signal
                # NOTE: During evaluation, always use max steps, this is to guarantee the same halting steps inside a batch for batching purposes
                halted = halted | (q_halt_logits > q_continue_logits)

                # Exploration
                min_halt_steps = (torch.rand_like(q_halt_logits) < self.config.halt_exploration_prob) * torch.randint_like(new_steps, low=2, high=self.config.halt_max_steps + 1)

                halted = halted & (new_steps >= min_halt_steps)

                # Compute target Q
                # NOTE: No replay buffer and target networks for computing target Q-value.
                # As batch_size is large, there're many parallel envs.
                # Similar concept as PQN https://arxiv.org/abs/2407.04811
                next_q_halt_logits, next_q_continue_logits = self.inner(new_inner_carry, new_current_data)[-1]
                
                outputs["target_q_continue"] = torch.sigmoid(torch.where(is_last_step, next_q_halt_logits, torch.maximum(next_q_halt_logits, next_q_continue_logits)))

        return HierarchicalReasoningModel_ACTV1Carry(new_inner_carry, new_steps, halted, new_current_data), outputs



In [ ]:
%%writefile checkpoints/Sudoku-extreme-1k-aug-1000 ACT-torch/HierarchicalReasoningModel_ACTV1 precise-cricket/losses.py
from typing import Any, Tuple, Dict, Sequence, Optional

import torch
import torch.nn.functional as F
from torch import nn


IGNORE_LABEL_ID = -100


def s(x, epsilon=1e-30):
    return torch.where(
        x<0,
        1/(1-x+ epsilon),
        x + 1
    )


def log_stablemax(x, dim=-1):
    s_x = s(x)
    return torch.log(s_x/torch.sum(s_x, dim=dim, keepdim=True))


def stablemax_cross_entropy(logits, labels, ignore_index: int = -100):
    logprobs = log_stablemax(logits.to(torch.float64), dim=-1)

    valid_mask = labels != ignore_index
    transformed_labels = torch.where(valid_mask, labels, 0)
    prediction_logprobs = torch.gather(logprobs, index=transformed_labels.to(torch.long).unsqueeze(-1), dim=-1).squeeze(-1)

    return -torch.where(valid_mask, prediction_logprobs, 0)


def softmax_cross_entropy(logits, labels, ignore_index: int = -100):
    # Cast logits to f32
    # Flatten logits
    return F.cross_entropy(logits.to(torch.float32).view(-1, logits.shape[-1]), labels.to(torch.long).view(-1), ignore_index=ignore_index, reduction="none").view(labels.shape)


class ACTLossHead(nn.Module):
    def __init__(self, model: nn.Module, loss_type: str):
        super().__init__()
        self.model = model
        self.loss_fn = globals()[loss_type]
        
    def initial_carry(self, *args, **kwargs):
        return self.model.initial_carry(*args, **kwargs)  # type: ignore

    def forward(
        self,
        return_keys: Sequence[str],
        # Model args
        **model_kwargs,
    ) -> Tuple[Any, torch.Tensor, Dict[str, torch.Tensor], Optional[Dict[str, torch.Tensor]], torch.Tensor]:
        # Model logits
        # B x SeqLen x D
        new_carry, outputs = self.model(**model_kwargs)
        labels = new_carry.current_data["labels"]

        # Correctness
        with torch.no_grad():
            mask = labels != IGNORE_LABEL_ID
            loss_counts = mask.sum(-1)
            loss_divisor = loss_counts.clamp_min(1).unsqueeze(-1)  # Avoid NaNs in division

            is_correct = mask & (torch.argmax(outputs["logits"], dim=-1) == labels)
            seq_is_correct = is_correct.sum(-1) == loss_counts
            
            # Metrics (halted)
            valid_metrics = new_carry.halted & (loss_counts > 0)
            metrics = {
                "count": valid_metrics.sum(),
                
                "accuracy":       torch.where(valid_metrics, (is_correct.to(torch.float32) / loss_divisor).sum(-1), 0).sum(),
                "exact_accuracy": (valid_metrics & seq_is_correct).sum(),

                "q_halt_accuracy": (valid_metrics & ((outputs["q_halt_logits"] >= 0) == seq_is_correct)).sum(),
                "steps":          torch.where(valid_metrics, new_carry.steps, 0).sum(),
            }

        # Losses
        # FIXME: Assuming the batch is always full
        lm_loss = (self.loss_fn(outputs["logits"], labels, ignore_index=IGNORE_LABEL_ID) / loss_divisor).sum()
        q_halt_loss = F.binary_cross_entropy_with_logits(outputs["q_halt_logits"], seq_is_correct.to(outputs["q_halt_logits"].dtype), reduction="sum")

        metrics.update({
            "lm_loss": lm_loss.detach(),
            "q_halt_loss": q_halt_loss.detach(),
        })

        # Q continue (bootstrapping target loss)
        q_continue_loss = 0
        if "target_q_continue" in outputs:
            q_continue_loss = F.binary_cross_entropy_with_logits(outputs["q_continue_logits"], outputs["target_q_continue"], reduction="sum")

            metrics["q_continue_loss"] = q_continue_loss.detach()

        # Filter outputs for return
        detached_outputs = {k: outputs[k].detach() for k in return_keys if k in outputs}

        return new_carry, lm_loss + 0.5 * (q_halt_loss + q_continue_loss), metrics, detached_outputs, new_carry.halted.all()



In [ ]:
%%writefile checkpoints/Sudoku-extreme-1k-aug-1000 ACT-torch/HierarchicalReasoningModel_ACTV1 ubiquitous-oyster/all_config.yaml
arch:
  H_cycles: 2
  H_layers: 4
  L_cycles: 2
  L_layers: 4
  expansion: 4
  halt_exploration_prob: 0.1
  halt_max_steps: 16
  hidden_size: 512
  loss:
    loss_type: stablemax_cross_entropy
    name: losses@ACTLossHead
  name: hrm.hrm_act_v1@HierarchicalReasoningModel_ACTV1
  num_heads: 8
  pos_encodings: rope
  puzzle_emb_ndim: 512
beta1: 0.9
beta2: 0.95
checkpoint_every_eval: true
checkpoint_path: checkpoints\Sudoku-extreme-1k-aug-1000 ACT-torch\HierarchicalReasoningModel_ACTV1
  ubiquitous-oyster
data_path: data/sudoku-extreme-1k-aug-1000
epochs: 2
eval_interval: 1
eval_save_outputs: []
global_batch_size: 8
load_checkpoint: checkpoints/HRM-checkpoint-sudoku-extreme/checkpoint
lr: 1.0e-05
lr_min_ratio: 1.0
lr_warmup_steps: 2000
project_name: Sudoku-extreme-1k-aug-1000 ACT-torch
puzzle_emb_lr: 1.0e-05
puzzle_emb_weight_decay: 1.0
run_name: HierarchicalReasoningModel_ACTV1 ubiquitous-oyster
seed: 0
weight_decay: 1.0



In [ ]:
%%writefile checkpoints/Sudoku-extreme-1k-aug-1000 ACT-torch/HierarchicalReasoningModel_ACTV1 ubiquitous-oyster/hrm_act_v1.py
from typing import Tuple, List, Dict, Optional
from dataclasses import dataclass
import math

import torch
import torch.nn.functional as F
from torch import nn
from pydantic import BaseModel

from models.common import trunc_normal_init_
from models.layers import rms_norm, SwiGLU, Attention, RotaryEmbedding, CosSin, CastedEmbedding, CastedLinear
from models.sparse_embedding import CastedSparseEmbedding


@dataclass
class HierarchicalReasoningModel_ACTV1InnerCarry:
    z_H: torch.Tensor
    z_L: torch.Tensor


@dataclass
class HierarchicalReasoningModel_ACTV1Carry:
    inner_carry: HierarchicalReasoningModel_ACTV1InnerCarry
    
    steps: torch.Tensor
    halted: torch.Tensor
    
    current_data: Dict[str, torch.Tensor]


class HierarchicalReasoningModel_ACTV1Config(BaseModel):
    batch_size: int
    seq_len: int
    puzzle_emb_ndim: int = 0
    num_puzzle_identifiers: int
    vocab_size: int

    H_cycles: int
    L_cycles: int

    H_layers: int
    L_layers: int

    # Transformer config
    hidden_size: int
    expansion: float
    num_heads: int
    pos_encodings: str

    rms_norm_eps: float = 1e-5
    rope_theta: float = 10000.0
    
    # Halting Q-learning config
    halt_max_steps: int
    halt_exploration_prob: float

    forward_dtype: str = "bfloat16"


class HierarchicalReasoningModel_ACTV1Block(nn.Module):
    def __init__(self, config: HierarchicalReasoningModel_ACTV1Config) -> None:
        super().__init__()

        self.self_attn = Attention(
            hidden_size=config.hidden_size,
            head_dim=config.hidden_size // config.num_heads,
            num_heads=config.num_heads,
            num_key_value_heads=config.num_heads,
            causal=False
        )
        self.mlp = SwiGLU(
            hidden_size=config.hidden_size,
            expansion=config.expansion,
        )
        self.norm_eps = config.rms_norm_eps

    def forward(self, cos_sin: CosSin, hidden_states: torch.Tensor) -> torch.Tensor:
        # Post Norm
        # Self Attention
        hidden_states = rms_norm(hidden_states + self.self_attn(cos_sin=cos_sin, hidden_states=hidden_states), variance_epsilon=self.norm_eps)
        # Fully Connected
        hidden_states = rms_norm(hidden_states + self.mlp(hidden_states), variance_epsilon=self.norm_eps)
        return hidden_states


class HierarchicalReasoningModel_ACTV1ReasoningModule(nn.Module):
    def __init__(self, layers: List[HierarchicalReasoningModel_ACTV1Block]):
        super().__init__()

        self.layers = torch.nn.ModuleList(layers)

    def forward(self, hidden_states: torch.Tensor, input_injection: torch.Tensor, **kwargs) -> torch.Tensor:
        # Input injection (add)
        hidden_states = hidden_states + input_injection
        # Layers
        for layer in self.layers:
            hidden_states = layer(hidden_states=hidden_states, **kwargs)

        return hidden_states


class HierarchicalReasoningModel_ACTV1_Inner(nn.Module):
    def __init__(self, config: HierarchicalReasoningModel_ACTV1Config) -> None:
        super().__init__()
        self.config = config
        self.forward_dtype = getattr(torch, self.config.forward_dtype)

        # I/O
        self.embed_scale  = math.sqrt(self.config.hidden_size)
        embed_init_std = 1.0 / self.embed_scale

        self.embed_tokens = CastedEmbedding(self.config.vocab_size, self.config.hidden_size, init_std=embed_init_std, cast_to=self.forward_dtype)
        self.lm_head      = CastedLinear(self.config.hidden_size, self.config.vocab_size, bias=False)
        self.q_head       = CastedLinear(self.config.hidden_size, 2, bias=True)

        self.puzzle_emb_len = -(self.config.puzzle_emb_ndim // -self.config.hidden_size)  # ceil div
        if self.config.puzzle_emb_ndim > 0:
            # Zero init puzzle embeddings
            self.puzzle_emb = CastedSparseEmbedding(self.config.num_puzzle_identifiers, self.config.puzzle_emb_ndim,
                                                    batch_size=self.config.batch_size, init_std=0, cast_to=self.forward_dtype)

        # LM Blocks
        if self.config.pos_encodings == "rope":
            self.rotary_emb = RotaryEmbedding(dim=self.config.hidden_size // self.config.num_heads,
                                              max_position_embeddings=self.config.seq_len + self.puzzle_emb_len,
                                              base=self.config.rope_theta)
        elif self.config.pos_encodings == "learned":
            self.embed_pos = CastedEmbedding(self.config.seq_len + self.puzzle_emb_len, self.config.hidden_size, init_std=embed_init_std, cast_to=self.forward_dtype)
        else:
            raise NotImplementedError()

        # Reasoning Layers
        self.H_level = HierarchicalReasoningModel_ACTV1ReasoningModule(layers=[HierarchicalReasoningModel_ACTV1Block(self.config) for _i in range(self.config.H_layers)])
        self.L_level = HierarchicalReasoningModel_ACTV1ReasoningModule(layers=[HierarchicalReasoningModel_ACTV1Block(self.config) for _i in range(self.config.L_layers)])
        
        # Initial states
        self.H_init = nn.Buffer(trunc_normal_init_(torch.empty(self.config.hidden_size, dtype=self.forward_dtype), std=1), persistent=True)
        self.L_init = nn.Buffer(trunc_normal_init_(torch.empty(self.config.hidden_size, dtype=self.forward_dtype), std=1), persistent=True)

        # Q head special init
        # Init Q to (almost) zero for faster learning during bootstrapping
        with torch.no_grad():
            self.q_head.weight.zero_()
            self.q_head.bias.fill_(-5)  # type: ignore

    def _input_embeddings(self, input: torch.Tensor, puzzle_identifiers: torch.Tensor):
        # Token embedding
        embedding = self.embed_tokens(input.to(torch.int32))

        # Puzzle embeddings
        if self.config.puzzle_emb_ndim > 0:
            puzzle_embedding = self.puzzle_emb(puzzle_identifiers)
            
            pad_count = self.puzzle_emb_len * self.config.hidden_size - puzzle_embedding.shape[-1]
            if pad_count > 0:
                puzzle_embedding = F.pad(puzzle_embedding, (0, pad_count))

            embedding = torch.cat((puzzle_embedding.view(-1, self.puzzle_emb_len, self.config.hidden_size), embedding), dim=-2)

        # Position embeddings
        if self.config.pos_encodings == "learned":
            # scale by 1/sqrt(2) to maintain forward variance
            embedding = 0.707106781 * (embedding + self.embed_pos.embedding_weight.to(self.forward_dtype))

        # Scale
        return self.embed_scale * embedding

    def empty_carry(self, batch_size: int):
        return HierarchicalReasoningModel_ACTV1InnerCarry(
            z_H=torch.empty(batch_size, self.config.seq_len + self.puzzle_emb_len, self.config.hidden_size, dtype=self.forward_dtype),
            z_L=torch.empty(batch_size, self.config.seq_len + self.puzzle_emb_len, self.config.hidden_size, dtype=self.forward_dtype),
        )
        
    def reset_carry(self, reset_flag: torch.Tensor, carry: HierarchicalReasoningModel_ACTV1InnerCarry):
        return HierarchicalReasoningModel_ACTV1InnerCarry(
            z_H=torch.where(reset_flag.view(-1, 1, 1), self.H_init, carry.z_H),
            z_L=torch.where(reset_flag.view(-1, 1, 1), self.L_init, carry.z_L),
        )

    def forward(self, carry: HierarchicalReasoningModel_ACTV1InnerCarry, batch: Dict[str, torch.Tensor]) -> Tuple[HierarchicalReasoningModel_ACTV1InnerCarry, torch.Tensor, Tuple[torch.Tensor, torch.Tensor]]:
        seq_info = dict(
            cos_sin=self.rotary_emb() if hasattr(self, "rotary_emb") else None,
        )

        # Input encoding
        input_embeddings = self._input_embeddings(batch["inputs"], batch["puzzle_identifiers"])

        # Forward iterations
        with torch.no_grad():
            z_H, z_L = carry.z_H, carry.z_L

            for _H_step in range(self.config.H_cycles):
                for _L_step in range(self.config.L_cycles):
                    if not ((_H_step == self.config.H_cycles - 1) and (_L_step == self.config.L_cycles - 1)):
                        z_L = self.L_level(z_L, z_H + input_embeddings, **seq_info)

                if not (_H_step == self.config.H_cycles - 1):
                    z_H = self.H_level(z_H, z_L, **seq_info)

        assert not z_H.requires_grad and not z_L.requires_grad

        # 1-step grad
        z_L = self.L_level(z_L, z_H + input_embeddings, **seq_info)
        z_H = self.H_level(z_H, z_L, **seq_info)

        # LM Outputs
        new_carry = HierarchicalReasoningModel_ACTV1InnerCarry(z_H=z_H.detach(), z_L=z_L.detach())  # New carry no grad
        output = self.lm_head(z_H)[:, self.puzzle_emb_len:]

        # Q head
        q_logits = self.q_head(z_H[:, 0]).to(torch.float32)
        
        return new_carry, output, (q_logits[..., 0], q_logits[..., 1])


class HierarchicalReasoningModel_ACTV1(nn.Module):
    """ACT wrapper."""

    def __init__(self, config_dict: dict):
        super().__init__()
        self.config = HierarchicalReasoningModel_ACTV1Config(**config_dict)
        self.inner = HierarchicalReasoningModel_ACTV1_Inner(self.config)

    @property
    def puzzle_emb(self):
        return self.inner.puzzle_emb

    def initial_carry(self, batch: Dict[str, torch.Tensor]):
        batch_size = batch["inputs"].shape[0]

        return HierarchicalReasoningModel_ACTV1Carry(
            inner_carry=self.inner.empty_carry(batch_size),  # Empty is expected, it will be reseted in first pass as all sequences are halted.
            
            steps=torch.zeros((batch_size, ), dtype=torch.int32),
            halted=torch.ones((batch_size, ), dtype=torch.bool),  # Default to halted
            
            current_data={k: torch.empty_like(v) for k, v in batch.items()}
        )
        
    def forward(self, carry: HierarchicalReasoningModel_ACTV1Carry, batch: Dict[str, torch.Tensor]) -> Tuple[HierarchicalReasoningModel_ACTV1Carry, Dict[str, torch.Tensor]]:
        # Update data, carry (removing halted sequences)
        new_inner_carry = self.inner.reset_carry(carry.halted, carry.inner_carry)
        
        new_steps = torch.where(carry.halted, 0, carry.steps)

        new_current_data = {k: torch.where(carry.halted.view((-1, ) + (1, ) * (batch[k].ndim - 1)), batch[k], v) for k, v in carry.current_data.items()}

        # Forward inner model
        new_inner_carry, logits, (q_halt_logits, q_continue_logits) = self.inner(new_inner_carry, new_current_data)

        outputs = {
            "logits": logits,
            "q_halt_logits": q_halt_logits,
            "q_continue_logits": q_continue_logits
        }
        
        with torch.no_grad():
            # Step
            new_steps = new_steps + 1
            is_last_step = new_steps >= self.config.halt_max_steps
            
            halted = is_last_step

            # if training, and ACT is enabled
            if self.training and (self.config.halt_max_steps > 1):
                # Halt signal
                # NOTE: During evaluation, always use max steps, this is to guarantee the same halting steps inside a batch for batching purposes
                halted = halted | (q_halt_logits > q_continue_logits)

                # Exploration
                min_halt_steps = (torch.rand_like(q_halt_logits) < self.config.halt_exploration_prob) * torch.randint_like(new_steps, low=2, high=self.config.halt_max_steps + 1)

                halted = halted & (new_steps >= min_halt_steps)

                # Compute target Q
                # NOTE: No replay buffer and target networks for computing target Q-value.
                # As batch_size is large, there're many parallel envs.
                # Similar concept as PQN https://arxiv.org/abs/2407.04811
                next_q_halt_logits, next_q_continue_logits = self.inner(new_inner_carry, new_current_data)[-1]
                
                outputs["target_q_continue"] = torch.sigmoid(torch.where(is_last_step, next_q_halt_logits, torch.maximum(next_q_halt_logits, next_q_continue_logits)))

        return HierarchicalReasoningModel_ACTV1Carry(new_inner_carry, new_steps, halted, new_current_data), outputs



In [ ]:
%%writefile checkpoints/Sudoku-extreme-1k-aug-1000 ACT-torch/HierarchicalReasoningModel_ACTV1 ubiquitous-oyster/losses.py
from typing import Any, Tuple, Dict, Sequence, Optional

import torch
import torch.nn.functional as F
from torch import nn


IGNORE_LABEL_ID = -100


def s(x, epsilon=1e-30):
    return torch.where(
        x<0,
        1/(1-x+ epsilon),
        x + 1
    )


def log_stablemax(x, dim=-1):
    s_x = s(x)
    return torch.log(s_x/torch.sum(s_x, dim=dim, keepdim=True))


def stablemax_cross_entropy(logits, labels, ignore_index: int = -100):
    logprobs = log_stablemax(logits.to(torch.float64), dim=-1)

    valid_mask = labels != ignore_index
    transformed_labels = torch.where(valid_mask, labels, 0)
    prediction_logprobs = torch.gather(logprobs, index=transformed_labels.to(torch.long).unsqueeze(-1), dim=-1).squeeze(-1)

    return -torch.where(valid_mask, prediction_logprobs, 0)


def softmax_cross_entropy(logits, labels, ignore_index: int = -100):
    # Cast logits to f32
    # Flatten logits
    return F.cross_entropy(logits.to(torch.float32).view(-1, logits.shape[-1]), labels.to(torch.long).view(-1), ignore_index=ignore_index, reduction="none").view(labels.shape)


class ACTLossHead(nn.Module):
    def __init__(self, model: nn.Module, loss_type: str):
        super().__init__()
        self.model = model
        self.loss_fn = globals()[loss_type]
        
    def initial_carry(self, *args, **kwargs):
        return self.model.initial_carry(*args, **kwargs)  # type: ignore

    def forward(
        self,
        return_keys: Sequence[str],
        # Model args
        **model_kwargs,
    ) -> Tuple[Any, torch.Tensor, Dict[str, torch.Tensor], Optional[Dict[str, torch.Tensor]], torch.Tensor]:
        # Model logits
        # B x SeqLen x D
        new_carry, outputs = self.model(**model_kwargs)
        labels = new_carry.current_data["labels"]

        # Correctness
        with torch.no_grad():
            mask = labels != IGNORE_LABEL_ID
            loss_counts = mask.sum(-1)
            loss_divisor = loss_counts.clamp_min(1).unsqueeze(-1)  # Avoid NaNs in division

            is_correct = mask & (torch.argmax(outputs["logits"], dim=-1) == labels)
            seq_is_correct = is_correct.sum(-1) == loss_counts
            
            # Metrics (halted)
            valid_metrics = new_carry.halted & (loss_counts > 0)
            metrics = {
                "count": valid_metrics.sum(),
                
                "accuracy":       torch.where(valid_metrics, (is_correct.to(torch.float32) / loss_divisor).sum(-1), 0).sum(),
                "exact_accuracy": (valid_metrics & seq_is_correct).sum(),

                "q_halt_accuracy": (valid_metrics & ((outputs["q_halt_logits"] >= 0) == seq_is_correct)).sum(),
                "steps":          torch.where(valid_metrics, new_carry.steps, 0).sum(),
            }

        # Losses
        # FIXME: Assuming the batch is always full
        lm_loss = (self.loss_fn(outputs["logits"], labels, ignore_index=IGNORE_LABEL_ID) / loss_divisor).sum()
        q_halt_loss = F.binary_cross_entropy_with_logits(outputs["q_halt_logits"], seq_is_correct.to(outputs["q_halt_logits"].dtype), reduction="sum")

        metrics.update({
            "lm_loss": lm_loss.detach(),
            "q_halt_loss": q_halt_loss.detach(),
        })

        # Q continue (bootstrapping target loss)
        q_continue_loss = 0
        if "target_q_continue" in outputs:
            q_continue_loss = F.binary_cross_entropy_with_logits(outputs["q_continue_logits"], outputs["target_q_continue"], reduction="sum")

            metrics["q_continue_loss"] = q_continue_loss.detach()

        # Filter outputs for return
        detached_outputs = {k: outputs[k].detach() for k in return_keys if k in outputs}

        return new_carry, lm_loss + 0.5 * (q_halt_loss + q_continue_loss), metrics, detached_outputs, new_carry.halted.all()



In [ ]:
%%writefile config/cfg_pretrain.yaml
# ARC training config

defaults:
  - arch: hrm_v1
  - _self_

hydra:
  output_subdir: null

# Data path
data_path: data/arc-aug-1000

# Hyperparams - Training
global_batch_size: 768

epochs: 100000
eval_interval: 10000
checkpoint_every_eval: True

lr: 1e-4
lr_min_ratio: 1.0
lr_warmup_steps: 2000

# Standard hyperparameter settings for LM, as used in Llama
beta1: 0.9
beta2: 0.95
weight_decay: 0.1
puzzle_emb_weight_decay: 0.1

# Hyperparams - Puzzle embeddings training
puzzle_emb_lr: 1e-2



In [ ]:
%%writefile config/arch/hrm_deep_v1.yaml
name: hrm.hrm_deep_v1@HierarchicalReasoningModel_DeepV1
loss:
  name: losses@ACTLossHead
  loss_type: stablemax_cross_entropy

halt_exploration_prob: 0.0
halt_max_steps: 3  # Fixed step = 3

H_cycles: 2
L_cycles: 2

H_layers: 4
L_layers: 4

hidden_size: 512
num_heads: 8
expansion: 4

puzzle_emb_ndim: ${.hidden_size}

pos_encodings: rope



In [ ]:
%%writefile config/arch/hrm_hrb_v1.yaml
name: hrm.hrm_hrb_v1@HierarchicalReasoningModel_HRBV1
loss:
  name: losses@ACTLossHead
  loss_type: stablemax_cross_entropy

halt_exploration_prob: 0.0
halt_max_steps: 1

H_cycles: 1
L_cycles: 1

H_layers: 2
L_layers: 1

hidden_size: 512
num_heads: 8
expansion: 4

puzzle_emb_ndim: ${.hidden_size}

pos_encodings: rope



In [ ]:
%%writefile config/arch/hrm_v1.yaml
name: hrm.hrm_act_v1@HierarchicalReasoningModel_ACTV1
loss:
  name: losses@ACTLossHead
  loss_type: stablemax_cross_entropy

halt_exploration_prob: 0.1
halt_max_steps: 16

H_cycles: 2
L_cycles: 2

H_layers: 4
L_layers: 4

hidden_size: 512
num_heads: 8  # min(2, hidden_size // 64)
expansion: 4

puzzle_emb_ndim: ${.hidden_size}

pos_encodings: rope



In [ ]:
%%writefile dataset/build_6x6_sudoku_dataset.py
import os
import json
import random
import numpy as np
from tqdm import tqdm
from argdantic import ArgParser
from pydantic import BaseModel

try:
    from dataset.common import PuzzleDatasetMetadata
except ImportError:
    from common import PuzzleDatasetMetadata

cli = ArgParser()

class DataProcessConfig(BaseModel):
    output_dir: str = "data/sudoku-6x6"
    num_train: int = 1000
    num_test: int = 100
    holes: int = 20  # Number of blanks

def is_valid(board, r, c, val):
    if val in board[r]: return False
    if val in board[:, c]: return False
    br, bc = 2 * (r // 2), 3 * (c // 3)
    if val in board[br:br+2, bc:bc+3]: return False
    return True

def solve(board):
    for r in range(6):
        for c in range(6):
            if board[r, c] == 0:
                nums = list(range(1, 7))
                random.shuffle(nums)
                for val in nums:
                    if is_valid(board, r, c, val):
                        board[r, c] = val
                        if solve(board):
                            return True
                        board[r, c] = 0
                return False
    return True

def generate_6x6_board():
    board = np.zeros((6, 6), dtype=int)
    solve(board)
    return board

def generate_puzzles(num_puzzles, num_holes):
    inputs = []
    labels = []
    for _ in tqdm(range(num_puzzles), desc="Generating 6x6 puzzles"):
        solution = generate_6x6_board()
        puzzle = solution.copy()
        
        # Poke holes
        cells = [(r, c) for r in range(6) for c in range(6)]
        random.shuffle(cells)
        for r, c in cells[:num_holes]:
            puzzle[r, c] = 0
            
        inputs.append(puzzle)
        labels.append(solution)
    return inputs, labels

def convert_subset(set_name: str, config: DataProcessConfig):
    num_puzzles = config.num_train if set_name == "train" else config.num_test
    
    inputs, labels = generate_puzzles(num_puzzles, config.holes)
    
    results = {k: [] for k in ["inputs", "labels", "puzzle_identifiers", "puzzle_indices", "group_indices"]}
    
    results["puzzle_indices"].append(0)
    results["group_indices"].append(0)
    
    for i, (inp, out) in enumerate(zip(inputs, labels)):
        # 0 becomes 1 (blank), 1-6 becomes 2-7
        results["inputs"].append(inp.flatten() + 1)
        results["labels"].append(out.flatten() + 1)
        
        results["puzzle_indices"].append(i + 1)
        results["puzzle_identifiers"].append(0)
        results["group_indices"].append(i + 1)
        
    results = {
        "inputs": np.array(results["inputs"], dtype=np.int32),
        "labels": np.array(results["labels"], dtype=np.int32),
        "group_indices": np.array(results["group_indices"], dtype=np.int32),
        "puzzle_indices": np.array(results["puzzle_indices"], dtype=np.int32),
        "puzzle_identifiers": np.array(results["puzzle_identifiers"], dtype=np.int32),
    }

    metadata = PuzzleDatasetMetadata(
        seq_len=36,
        vocab_size=8,  # PAD + "0" + "1"..."6"
        pad_id=0,
        ignore_label_id=0,
        blank_identifier_id=1,
        num_puzzle_identifiers=1,
        total_groups=len(results["group_indices"]) - 1,
        mean_puzzle_examples=1,
        sets=["all"]
    )

    save_dir = os.path.join(config.output_dir, set_name)
    os.makedirs(save_dir, exist_ok=True)
    
    with open(os.path.join(save_dir, "dataset.json"), "w") as f:
        json.dump(metadata.model_dump(), f)
        
    for k, v in results.items():
        np.save(os.path.join(save_dir, f"all__{k}.npy"), v)

    with open(os.path.join(config.output_dir, "identifiers.json"), "w") as f:
        json.dump(["<blank>"], f)

@cli.command(singleton=True)
def preprocess_data(config: DataProcessConfig):
    convert_subset("train", config)
    convert_subset("test", config)

if __name__ == "__main__":
    cli()



In [ ]:
%%writefile dataset/build_arc_dataset.py
from typing import List, Optional, Tuple, Dict
from dataclasses import dataclass
from pathlib import Path
import os
import json
import hashlib
import numpy as np
from glob import glob

from argdantic import ArgParser
from pydantic import BaseModel

from common import PuzzleDatasetMetadata, dihedral_transform


cli = ArgParser()


class DataProcessConfig(BaseModel):
    # ARC-1
    dataset_dirs: List[str] = ["dataset/raw-data/ARC-AGI/data", "dataset/raw-data/ConceptARC/corpus"]
    output_dir: str = "data/arc-aug-1000"
    
    # ARC-2
    # dataset_dirs: List[str] = ["dataset/raw-data/ARC-AGI-2/data"]
    # output_dir: str = "data/arc-2-aug-1000"

    seed: int = 42
    num_aug: int = 1000
    
    
ARCMaxGridSize = 30
ARCAugmentRetriesFactor = 5
    

@dataclass
class ARCPuzzle:
    id: str

    examples: List[Tuple[np.ndarray, np.ndarray]]

    
def arc_grid_to_np(grid: List[List[int]]):
    arr = np.array(grid)

    # Shape check
    assert arr.ndim == 2
    assert arr.shape[0] <= ARCMaxGridSize and arr.shape[1] <= ARCMaxGridSize
    # Element check
    assert np.all((arr >= 0) & (arr <= 9))
    return arr.astype(np.uint8)


def np_grid_to_seq_translational_augment(inp: np.ndarray, out: np.ndarray, do_translation: bool):
    # PAD: 0, <eos>: 1, digits: 2 ... 11
    # Compute random top-left pad
    if do_translation:
        pad_r = np.random.randint(0, ARCMaxGridSize - max(inp.shape[0], out.shape[0]) + 1)
        pad_c = np.random.randint(0, ARCMaxGridSize - max(inp.shape[1], out.shape[1]) + 1)
    else:
        pad_r = pad_c = 0

    # Pad grid
    result = []
    for grid in [inp, out]:
        nrow, ncol = grid.shape
        grid = np.pad(grid + 2, ((pad_r, ARCMaxGridSize - pad_r - nrow), (pad_c, ARCMaxGridSize - pad_c - ncol)), constant_values=0)

        # Add <eos>
        eos_row, eos_col = pad_r + nrow, pad_c + ncol
        if eos_row < ARCMaxGridSize:
            grid[eos_row, pad_c:eos_col] = 1
        if eos_col < ARCMaxGridSize:
            grid[pad_r:eos_row, eos_col] = 1

        result.append(grid.flatten())

    return result


def puzzle_hash(puzzle: dict):
    # Hash the puzzle for checking equivalence
    def _grid_hash(grid: np.ndarray):
        buffer = [x.to_bytes(1) for x in grid.shape]
        buffer.append(grid.tobytes())
        
        return hashlib.sha256(b"".join(buffer)).hexdigest()
    
    hashes = []
    for example_type, example in puzzle.items():
        for input, label in example.examples:
            hashes.append(f"{_grid_hash(input)}|{_grid_hash(label)}")
            
    hashes.sort()
    return hashlib.sha256("|".join(hashes).encode()).hexdigest()


def convert_single_arc_puzzle(results: dict, default_name: str, puzzle: dict, aug_count: int, dest_mapping: Dict[str, Tuple[str, str]]):
    # Remove "name"
    name = puzzle.pop("name", default_name)
    
    # Convert
    dests = set(dest_mapping.values())
    converted = {dest: ARCPuzzle(name, []) for dest in dests}
    for example_type, examples in puzzle.items():
        dest = dest_mapping[example_type]
        converted[dest].examples.extend([(arc_grid_to_np(example["input"]), arc_grid_to_np(example["output"])) for example in examples])

    group = [converted]
    
    # Augment
    if aug_count > 0:
        hashes = {puzzle_hash(converted)}

        for _trial in range(ARCAugmentRetriesFactor * aug_count):
            # Augment plan
            trans_id = np.random.randint(0, 8)
            mapping = np.concatenate([np.arange(0, 1, dtype=np.uint8), np.random.permutation(np.arange(1, 10, dtype=np.uint8))])  # Permute colors, Excluding "0" (black)
            
            aug_repr = f"t{trans_id}_{''.join(str(x) for x in mapping)}"

            def _map_grid(grid: np.ndarray):
                return dihedral_transform(mapping[grid], trans_id)
            
            # Check duplicate
            augmented = {dest: ARCPuzzle(f"{puzzle.id}_{aug_repr}", [(_map_grid(input), _map_grid(label)) for (input, label) in puzzle.examples]) for dest, puzzle in converted.items()}
            h = puzzle_hash(augmented)
            if h not in hashes:
                hashes.add(h)
                group.append(augmented)
                
            if len(group) >= aug_count + 1:
                break
            
        if len(group) < aug_count + 1:
            print (f"[Puzzle {name}] augmentation not full, only {len(group)}")

    # Append
    for dest in dests:
        # Convert the examples
        dest_split, dest_set = dest

        results.setdefault(dest_split, {})
        results[dest_split].setdefault(dest_set, [])
        results[dest_split][dest_set].append([converted[dest] for converted in group])


def load_puzzles_arcagi(results: dict, dataset_path: str, config: DataProcessConfig):
    train_examples_dest = ("train", "all")
    test_examples_map = {
        "evaluation": [(1.0, ("test", "all"))],
        "_default": [(1.0, ("train", "all"))]
    }
    
    total_puzzles = 0
    for subdir in os.scandir(dataset_path):
        if subdir.is_dir():
            # Load all puzzles in this directory
            puzzles = []
            for filename in glob(os.path.join(subdir.path, "*.json")):
                with open(filename, "r") as f:
                    puzzles.append((Path(filename).stem, json.load(f)))
                    
            # Shuffle puzzles
            np.random.shuffle(puzzles)
            
            # Assign by fraction
            for idx, (default_name, puzzle) in enumerate(puzzles):
                fraction = idx / len(puzzles)
                test_examples_dest = None
                for f, dest in test_examples_map.get(subdir.name, test_examples_map["_default"]):
                    if fraction < f:
                        test_examples_dest = dest
                        break
                        
                assert test_examples_dest is not None
                
                convert_single_arc_puzzle(results, default_name, puzzle, config.num_aug, {"train": train_examples_dest, "test": test_examples_dest})
                total_puzzles += 1

    print (f"[{dataset_path}] total puzzles: {total_puzzles}")


def convert_dataset(config: DataProcessConfig):
    np.random.seed(config.seed)
    
    # Read dataset
    data = {}
    for dataset_dir in config.dataset_dirs:
        load_puzzles_arcagi(data, dataset_dir, config)
    
    # Map global puzzle identifiers
    num_identifiers = 1  # 0 is blank
    identifier_map = {}
    for split_name, split in data.items():
        for subset_name, subset in split.items():
            for group in subset:
                for puzzle in group:
                    if puzzle.id not in identifier_map:
                        identifier_map[puzzle.id] = num_identifiers
                        num_identifiers += 1

    print (f"Total puzzle IDs (including <blank>): {num_identifiers}")

    # Save
    for split_name, split in data.items():
        os.makedirs(os.path.join(config.output_dir, split_name), exist_ok=True)
        
        # Translational augmentations
        enable_translational_augment = split_name == "train"

        # Statistics
        total_examples = 0
        total_puzzles = 0
        total_groups = 0
        
        for subset_name, subset in split.items():
            # Construct subset
            results = {k: [] for k in ["inputs", "labels", "puzzle_identifiers", "puzzle_indices", "group_indices"]}
            results["puzzle_indices"].append(0)
            results["group_indices"].append(0)
            
            example_id = 0
            puzzle_id = 0
            
            for group in subset:
                for puzzle in group:
                    # Push puzzle
                    no_aug_id = np.random.randint(0, len(puzzle.examples))
                    for _idx_ex, (inp, out) in enumerate(puzzle.examples):
                        inp, out = np_grid_to_seq_translational_augment(inp, out, do_translation=enable_translational_augment and _idx_ex != no_aug_id)
                            
                        results["inputs"].append(inp)
                        results["labels"].append(out)
                        example_id += 1
                        
                        total_examples += 1

                    results["puzzle_indices"].append(example_id)
                    results["puzzle_identifiers"].append(identifier_map[puzzle.id])
                    
                    puzzle_id += 1
                    
                    total_puzzles += 1
                    
                # Push group
                results["group_indices"].append(puzzle_id)
                total_groups += 1
            
            for k, v in results.items():
                if k in {"inputs", "labels"}:
                    v = np.stack(v, 0)
                else:
                    v = np.array(v, dtype=np.int32)
                
                np.save(os.path.join(config.output_dir, split_name, f"{subset_name}__{k}.npy"), v)
        
        # Metadata
        metadata = PuzzleDatasetMetadata(
            seq_len=ARCMaxGridSize * ARCMaxGridSize,
            vocab_size=10 + 2,  # PAD + EOS + "0" ... "9"
            
            pad_id=0,
            ignore_label_id=0,
            
            blank_identifier_id=0,
            num_puzzle_identifiers=num_identifiers,
            
            total_groups=total_groups,
            mean_puzzle_examples=total_examples / total_puzzles,
            sets=list(split.keys())
        )

        # Save metadata as JSON.
        with open(os.path.join(config.output_dir, split_name, "dataset.json"), "w") as f:
            json.dump(metadata.model_dump(), f)
            
    # Save IDs mapping
    with open(os.path.join(config.output_dir, "identifiers.json"), "w") as f:
        ids_mapping = {v: k for k, v in identifier_map.items()}
        
        json.dump([ids_mapping.get(i, "<blank>") for i in range(num_identifiers)], f)


@cli.command(singleton=True)
def main(config: DataProcessConfig):
    convert_dataset(config)


if __name__ == "__main__":
    cli()



In [ ]:
%%writefile dataset/build_maze_dataset.py
from typing import Optional
import math
import os
import csv
import json
import numpy as np

from argdantic import ArgParser
from pydantic import BaseModel
from tqdm import tqdm
from huggingface_hub import hf_hub_download

from common import PuzzleDatasetMetadata, dihedral_transform


CHARSET = "# SGo"


cli = ArgParser()


class DataProcessConfig(BaseModel):
    source_repo: str = "sapientinc/maze-30x30-hard-1k"
    output_dir: str = "data/maze-30x30-hard-1k"

    subsample_size: Optional[int] = None
    aug: bool = False


def convert_subset(set_name: str, config: DataProcessConfig):
    # Read CSV
    all_chars = set()
    grid_size = None
    inputs = []
    labels = []
    
    with open(hf_hub_download(config.source_repo, f"{set_name}.csv", repo_type="dataset"), newline="") as csvfile:  # type: ignore
        reader = csv.reader(csvfile)
        next(reader)  # Skip header
        for source, q, a, rating in reader:
            all_chars.update(q)
            all_chars.update(a)

            if grid_size is None:
                n = int(len(q) ** 0.5)
                grid_size = (n, n)
                
            inputs.append(np.frombuffer(q.encode(), dtype=np.uint8).reshape(grid_size))
            labels.append(np.frombuffer(a.encode(), dtype=np.uint8).reshape(grid_size))

    # If subsample_size is specified for the training set,
    # randomly sample the desired number of examples.
    if set_name == "train" and config.subsample_size is not None:
        total_samples = len(inputs)
        if config.subsample_size < total_samples:
            indices = np.random.choice(total_samples, size=config.subsample_size, replace=False)
            inputs = [inputs[i] for i in indices]
            labels = [labels[i] for i in indices]

    # Generate dataset
    results = {k: [] for k in ["inputs", "labels", "puzzle_identifiers", "puzzle_indices", "group_indices"]}
    puzzle_id = 0
    example_id = 0
    
    results["puzzle_indices"].append(0)
    results["group_indices"].append(0)
    
    for inp, out in zip(tqdm(inputs), labels):
        # Dihedral transformations for augmentation
        for aug_idx in range(8 if (set_name == "train" and config.aug) else 1):
            results["inputs"].append(dihedral_transform(inp, aug_idx))
            results["labels"].append(dihedral_transform(out, aug_idx))
            example_id += 1
            puzzle_id += 1
            
            results["puzzle_indices"].append(example_id)
            results["puzzle_identifiers"].append(0)
            
        # Push group
        results["group_indices"].append(puzzle_id)
            
    # Char mappings
    assert len(all_chars - set(CHARSET)) == 0
    
    char2id = np.zeros(256, np.uint8)
    char2id[np.array(list(map(ord, CHARSET)))] = np.arange(len(CHARSET)) + 1

    # To Numpy
    def _seq_to_numpy(seq):
        arr = np.vstack([char2id[s.reshape(-1)] for s in seq])
        
        return arr
    
    results = {
        "inputs": _seq_to_numpy(results["inputs"]),
        "labels": _seq_to_numpy(results["labels"]),
        
        "group_indices": np.array(results["group_indices"], dtype=np.int32),
        "puzzle_indices": np.array(results["puzzle_indices"], dtype=np.int32),
        "puzzle_identifiers": np.array(results["puzzle_identifiers"], dtype=np.int32),
    }

    # Metadata
    metadata = PuzzleDatasetMetadata(
        seq_len=int(math.prod(grid_size)),  # type: ignore
        vocab_size=len(CHARSET) + 1,  # PAD + Charset
        
        pad_id=0,
        ignore_label_id=0,
        
        blank_identifier_id=0,
        num_puzzle_identifiers=1,
        
        total_groups=len(results["group_indices"]) - 1,
        mean_puzzle_examples=1,
        sets=["all"]
    )

    # Save metadata as JSON.
    save_dir = os.path.join(config.output_dir, set_name)
    os.makedirs(save_dir, exist_ok=True)
    
    with open(os.path.join(save_dir, "dataset.json"), "w") as f:
        json.dump(metadata.model_dump(), f)
        
    # Save data
    for k, v in results.items():
        np.save(os.path.join(save_dir, f"all__{k}.npy"), v)
        
    # Save IDs mapping (for visualization only)
    with open(os.path.join(config.output_dir, "identifiers.json"), "w") as f:
        json.dump(["<blank>"], f)


@cli.command(singleton=True)
def preprocess_data(config: DataProcessConfig):
    convert_subset("train", config)
    convert_subset("test", config)


if __name__ == "__main__":
    cli()



In [ ]:
%%writefile dataset/build_sudoku_dataset.py
from typing import Optional
import os
import csv
import json
import numpy as np

from argdantic import ArgParser
from pydantic import BaseModel
from tqdm import tqdm
from huggingface_hub import hf_hub_download

from common import PuzzleDatasetMetadata


cli = ArgParser()


class DataProcessConfig(BaseModel):
    source_repo: str = "sapientinc/sudoku-extreme"
    output_dir: str = "data/sudoku-extreme-full"

    subsample_size: Optional[int] = None
    min_difficulty: Optional[int] = None
    num_aug: int = 0


def shuffle_sudoku(board: np.ndarray, solution: np.ndarray):
    # Create a random digit mapping: a permutation of 1..9, with zero (blank) unchanged
    digit_map = np.pad(np.random.permutation(np.arange(1, 10)), (1, 0))
    
    # Randomly decide whether to transpose.
    transpose_flag = np.random.rand() < 0.5

    # Generate a valid row permutation:
    # - Shuffle the 3 bands (each band = 3 rows) and for each band, shuffle its 3 rows.
    bands = np.random.permutation(3)
    row_perm = np.concatenate([b * 3 + np.random.permutation(3) for b in bands])

    # Similarly for columns (stacks).
    stacks = np.random.permutation(3)
    col_perm = np.concatenate([s * 3 + np.random.permutation(3) for s in stacks])

    # Build an 81->81 mapping. For each new cell at (i, j)
    # (row index = i // 9, col index = i % 9),
    # its value comes from old row = row_perm[i//9] and old col = col_perm[i%9].
    mapping = np.array([row_perm[i // 9] * 9 + col_perm[i % 9] for i in range(81)])

    def apply_transformation(x: np.ndarray) -> np.ndarray:
        # Apply transpose flag
        if transpose_flag:
            x = x.T
        # Apply the position mapping.
        new_board = x.flatten()[mapping].reshape(9, 9).copy()
        # Apply digit mapping
        return digit_map[new_board]

    return apply_transformation(board), apply_transformation(solution)


def convert_subset(set_name: str, config: DataProcessConfig):
    # Read CSV
    inputs = []
    labels = []
    
    # Count total rows to sample indices without loading into memory
    csv_path = hf_hub_download(config.source_repo, f"{set_name}.csv", repo_type="dataset")
    
    selected_indices = None
    if set_name == "train" and config.subsample_size is not None:
        # Instead of counting (which takes time but no RAM), we assume a large number or just read the first N
        # Or better: count lines
        with open(csv_path, 'rb') as f:
            total_samples = sum(1 for _ in f) - 1 # minus header
            
        if config.subsample_size < total_samples:
            selected_indices = set(np.random.choice(total_samples, size=config.subsample_size, replace=False))

    with open(csv_path, newline="") as csvfile:
        reader = csv.reader(csvfile)
        next(reader)  # Skip header
        for i, (source, q, a, rating) in enumerate(reader):
            if selected_indices is not None and i not in selected_indices:
                continue
                
            if (config.min_difficulty is None) or (int(rating) >= config.min_difficulty):
                assert len(q) == 81 and len(a) == 81
                
                inputs.append(np.frombuffer(q.replace('.', '0').encode(), dtype=np.uint8).reshape(9, 9) - ord('0'))
                labels.append(np.frombuffer(a.encode(), dtype=np.uint8).reshape(9, 9) - ord('0'))
                
            if selected_indices is None and config.subsample_size is not None and len(inputs) >= config.subsample_size:
                 break

    # Generate dataset
    num_augments = config.num_aug if set_name == "train" else 0

    results = {k: [] for k in ["inputs", "labels", "puzzle_identifiers", "puzzle_indices", "group_indices"]}
    puzzle_id = 0
    example_id = 0
    
    results["puzzle_indices"].append(0)
    results["group_indices"].append(0)
    
    for orig_inp, orig_out in zip(tqdm(inputs), labels):
        for aug_idx in range(1 + num_augments):
            # First index is not augmented
            if aug_idx == 0:
                inp, out = orig_inp, orig_out
            else:
                inp, out = shuffle_sudoku(orig_inp, orig_out)

            # Push puzzle (only single example)
            results["inputs"].append(inp)
            results["labels"].append(out)
            example_id += 1
            puzzle_id += 1
            
            results["puzzle_indices"].append(example_id)
            results["puzzle_identifiers"].append(0)
            
        # Push group
        results["group_indices"].append(puzzle_id)
        
    # To Numpy
    def _seq_to_numpy(seq):
        arr = np.concatenate(seq).reshape(len(seq), -1)
        
        assert np.all((arr >= 0) & (arr <= 9))
        return arr + 1
    
    results = {
        "inputs": _seq_to_numpy(results["inputs"]),
        "labels": _seq_to_numpy(results["labels"]),
        
        "group_indices": np.array(results["group_indices"], dtype=np.int32),
        "puzzle_indices": np.array(results["puzzle_indices"], dtype=np.int32),
        "puzzle_identifiers": np.array(results["puzzle_identifiers"], dtype=np.int32),
    }

    # Metadata
    metadata = PuzzleDatasetMetadata(
        seq_len=81,
        vocab_size=10 + 1,  # PAD + "0" ... "9"
        
        pad_id=0,
        ignore_label_id=0,
        
        blank_identifier_id=0,
        num_puzzle_identifiers=1,
        
        total_groups=len(results["group_indices"]) - 1,
        mean_puzzle_examples=1,
        sets=["all"]
    )

    # Save metadata as JSON.
    save_dir = os.path.join(config.output_dir, set_name)
    os.makedirs(save_dir, exist_ok=True)
    
    with open(os.path.join(save_dir, "dataset.json"), "w") as f:
        json.dump(metadata.model_dump(), f)
        
    # Save data
    for k, v in results.items():
        np.save(os.path.join(save_dir, f"all__{k}.npy"), v)
        
    # Save IDs mapping (for visualization only)
    with open(os.path.join(config.output_dir, "identifiers.json"), "w") as f:
        json.dump(["<blank>"], f)


@cli.command(singleton=True)
def preprocess_data(config: DataProcessConfig):
    convert_subset("train", config)
    convert_subset("test", config)


if __name__ == "__main__":
    cli()



In [ ]:
%%writefile dataset/common.py
from typing import List, Optional

import pydantic
import numpy as np


# Global list mapping each dihedral transform id to its inverse.
# Index corresponds to the original tid, and the value is its inverse.
DIHEDRAL_INVERSE = [0, 3, 2, 1, 4, 5, 6, 7]


class PuzzleDatasetMetadata(pydantic.BaseModel):
    pad_id: int
    ignore_label_id: Optional[int]
    blank_identifier_id: int
    
    vocab_size: int
    seq_len: int
    num_puzzle_identifiers: int
    
    total_groups: int
    mean_puzzle_examples: float

    sets: List[str]


def dihedral_transform(arr: np.ndarray, tid: int) -> np.ndarray:
    """8 dihedral symmetries by rotate, flip and mirror"""
    
    if tid == 0:
        return arr  # identity
    elif tid == 1:
        return np.rot90(arr, k=1)
    elif tid == 2:
        return np.rot90(arr, k=2)
    elif tid == 3:
        return np.rot90(arr, k=3)
    elif tid == 4:
        return np.fliplr(arr)       # horizontal flip
    elif tid == 5:
        return np.flipud(arr)       # vertical flip
    elif tid == 6:
        return arr.T                # transpose (reflection along main diagonal)
    elif tid == 7:
        return np.fliplr(np.rot90(arr, k=1))  # anti-diagonal reflection
    else:
        return arr
    
    
def inverse_dihedral_transform(arr: np.ndarray, tid: int) -> np.ndarray:
    return dihedral_transform(arr, DIHEDRAL_INVERSE[tid])



In [ ]:
%%writefile models/common.py
import math

import torch
from torch import nn


def trunc_normal_init_(tensor: torch.Tensor, std: float = 1.0, lower: float = -2.0, upper: float = 2.0):
    # NOTE: PyTorch nn.init.trunc_normal_ is not mathematically correct, the std dev is not actually the std dev of initialized tensor
    # This function is a PyTorch version of jax truncated normal init (default init method in flax)
    # https://github.com/jax-ml/jax/blob/main/jax/_src/random.py#L807-L848
    # https://github.com/jax-ml/jax/blob/main/jax/_src/nn/initializers.py#L162-L199

    with torch.no_grad():
        if std == 0:
            tensor.zero_()
        else:
            sqrt2 = math.sqrt(2)
            a = math.erf(lower / sqrt2)
            b = math.erf(upper / sqrt2)
            z = (b - a) / 2

            c = (2 * math.pi) ** -0.5
            pdf_u = c * math.exp(-0.5 * lower ** 2)
            pdf_l = c * math.exp(-0.5 * upper ** 2)
            comp_std = std / math.sqrt(1 - (upper * pdf_u - lower * pdf_l) / z - ((pdf_u - pdf_l) / z) ** 2)

            tensor.uniform_(a, b)
            tensor.erfinv_()
            tensor.mul_(sqrt2 * comp_std)
            tensor.clip_(lower * comp_std, upper * comp_std)

    return tensor



In [ ]:
%%writefile models/layers.py
from typing import Tuple

import torch
from torch import nn
import torch.nn.functional as F

try:
    from flash_attn_interface import flash_attn_func  # type: ignore[import]
except ImportError:
    # Fallback to FlashAttention 2
    try:
        from flash_attn import flash_attn_func  # type: ignore[import]
    except ImportError:
        # Fallback to PyTorch native SDPA
        def flash_attn_func(q, k, v, causal=False):
            q = q.transpose(1, 2)
            k = k.transpose(1, 2)
            v = v.transpose(1, 2)
            
            # Cast to float32 to avoid bfloat16 cuBLAS bugs on Turing (T4)
            orig_dtype = q.dtype
            q = q.to(torch.float32)
            k = k.to(torch.float32)
            v = v.to(torch.float32)
            
            # Manual attention
            L = q.size(-2)
            scale = q.size(-1) ** -0.5
            attn = (q @ k.transpose(-2, -1)) * scale
            if causal:
                mask = torch.ones(L, L, dtype=torch.bool, device=q.device).tril(diagonal=0)
                attn.masked_fill_(~mask, float('-inf'))
            attn = F.softmax(attn, dim=-1)
            out = attn @ v
            
            out = out.to(orig_dtype)
            return out.transpose(1, 2).contiguous()

from models.common import trunc_normal_init_


CosSin = Tuple[torch.Tensor, torch.Tensor]


def _find_multiple(a, b):
    return (-(a // -b)) * b


def rotate_half(x: torch.Tensor):
    """Rotates half the hidden dims of the input."""
    x1 = x[..., : x.shape[-1] // 2]
    x2 = x[..., x.shape[-1] // 2 :]
    return torch.cat((-x2, x1), dim=-1)


def apply_rotary_pos_emb(q: torch.Tensor, k: torch.Tensor, cos: torch.Tensor, sin: torch.Tensor):
    # q, k: [bs, seq_len, num_heads, head_dim]
    # cos, sin: [seq_len, head_dim]
    orig_dtype = q.dtype
    q = q.to(cos.dtype)
    k = k.to(cos.dtype)

    q_embed = (q * cos.unsqueeze(-2)) + (rotate_half(q) * sin.unsqueeze(-2))
    k_embed = (k * cos.unsqueeze(-2)) + (rotate_half(k) * sin.unsqueeze(-2))

    return q_embed.to(orig_dtype), k_embed.to(orig_dtype)


class CastedLinear(nn.Module):
    def __init__(self,
                 in_features: int,
                 out_features: int,
                 bias: bool):
        super().__init__()
        self.weight = nn.Parameter(
            trunc_normal_init_(torch.empty((out_features, in_features)), std=1.0 / (in_features ** 0.5))
        )
        self.bias = None
        if bias:
            self.bias = nn.Parameter(torch.zeros((out_features, )))

    def forward(self, input: torch.Tensor) -> torch.Tensor:
        orig_dtype = input.dtype
        # T4 GPUs (Turing) do not have hardware bfloat16 and their cuBLAS emulation is very buggy in eval mode.
        # Cast to float32 before F.linear to prevent CUBLAS_STATUS_EXECUTION_FAILED and device-side asserts.
        if orig_dtype == torch.bfloat16:
            input_f32 = input.to(torch.float32)
            weight_f32 = self.weight.to(torch.float32)
            bias_f32 = self.bias.to(torch.float32) if self.bias is not None else None
            out = F.linear(input_f32, weight_f32, bias=bias_f32)
            return out.to(orig_dtype)
        else:
            return F.linear(input, self.weight.to(orig_dtype), bias=self.bias.to(orig_dtype) if self.bias is not None else None)


class CastedEmbedding(nn.Module):
    def __init__(self,
                 num_embeddings: int,
                 embedding_dim: int,
                 init_std: float,
                 cast_to: torch.dtype):
        super().__init__()
        self.cast_to = cast_to

        # Truncated LeCun normal init
        self.embedding_weight = nn.Parameter(
            trunc_normal_init_(torch.empty((num_embeddings, embedding_dim)), std=init_std)
        )
        
    def forward(self, input: torch.Tensor) -> torch.Tensor:
        return F.embedding(input, self.embedding_weight.to(self.cast_to))


class RotaryEmbedding(nn.Module):
    def __init__(self, dim, max_position_embeddings, base, device=None):
        super().__init__()

        # RoPE
        inv_freq = 1.0 / (base ** (torch.arange(0, dim, 2, dtype=torch.float32, device=device) / dim))
        t = torch.arange(max_position_embeddings, dtype=torch.float32, device=device)
        freqs = torch.outer(t, inv_freq)

        # Different from paper, but it uses a different permutation in order to obtain the same calculation
        emb = torch.cat((freqs, freqs), dim=-1)
        self.cos_cached = nn.Buffer(emb.cos(), persistent=False)
        self.sin_cached = nn.Buffer(emb.sin(), persistent=False)

    def forward(self):
        return self.cos_cached, self.sin_cached


class Attention(nn.Module):
    def __init__(self, hidden_size, head_dim, num_heads, num_key_value_heads, causal=False):
        super().__init__()

        self.hidden_size = hidden_size
        self.head_dim = head_dim
        self.output_size = head_dim * num_heads
        self.num_heads = num_heads
        self.num_key_value_heads = num_key_value_heads
        self.causal = causal

        self.qkv_proj = CastedLinear(self.hidden_size, (self.num_heads + 2 * self.num_key_value_heads) * self.head_dim, bias=False)
        self.o_proj = CastedLinear(self.output_size, self.hidden_size, bias=False)

    def forward(self, cos_sin: CosSin, hidden_states: torch.Tensor) -> torch.Tensor:
        batch_size, seq_len, _ = hidden_states.shape

        # hidden_states: [bs, seq_len, num_heads, head_dim]
        qkv = self.qkv_proj(hidden_states)

        # Split head
        qkv = qkv.view(batch_size, seq_len, self.num_heads + 2 * self.num_key_value_heads, self.head_dim)
        query = qkv[:, :, :self.num_heads]
        key = qkv[:, :, self.num_heads: self.num_heads + self.num_key_value_heads]
        value = qkv[:, :, self.num_heads + self.num_key_value_heads:]

        # RoPE
        if cos_sin is not None:
            cos, sin = cos_sin
            query, key = apply_rotary_pos_emb(query, key, cos, sin)

        # flash attn
        attn_output = flash_attn_func(q=query, k=key, v=value, causal=self.causal)
        if isinstance(attn_output, tuple):  # fa2 and fa3 compatibility
            attn_output = attn_output[0]

        attn_output = attn_output.view(batch_size, seq_len, self.output_size)  # type: ignore
        return self.o_proj(attn_output)


class SwiGLU(nn.Module):
    def __init__(self, hidden_size: int, expansion: float):
        super().__init__()
        inter = _find_multiple(round(expansion * hidden_size * 2 / 3), 256)

        self.gate_up_proj = CastedLinear(hidden_size, inter * 2, bias=False)
        self.down_proj    = CastedLinear(inter, hidden_size, bias=False)

    def forward(self, x):
        gate, up = self.gate_up_proj(x).chunk(2, dim=-1)
        return self.down_proj(F.silu(gate) * up)


def rms_norm(hidden_states: torch.Tensor, variance_epsilon: float) -> torch.Tensor:
    input_dtype = hidden_states.dtype
    hidden_states = hidden_states.to(torch.float32)

    variance = hidden_states.square().mean(-1, keepdim=True)
    hidden_states = hidden_states * torch.rsqrt(variance + variance_epsilon)
    return hidden_states.to(input_dtype)



In [ ]:
%%writefile models/losses.py
from typing import Any, Tuple, Dict, Sequence, Optional

import torch
import torch.nn.functional as F
from torch import nn


IGNORE_LABEL_ID = -100


def s(x, epsilon=1e-30):
    return torch.where(
        x<0,
        1/(1-x+ epsilon),
        x + 1
    )


def log_stablemax(x, dim=-1):
    s_x = s(x)
    return torch.log(s_x/torch.sum(s_x, dim=dim, keepdim=True))


def stablemax_cross_entropy(logits, labels, ignore_index: int = -100):
    logprobs = log_stablemax(logits.to(torch.float64), dim=-1)

    valid_mask = labels != ignore_index
    transformed_labels = torch.where(valid_mask, labels, 0)
    prediction_logprobs = torch.gather(logprobs, index=transformed_labels.to(torch.long).unsqueeze(-1), dim=-1).squeeze(-1)

    return -torch.where(valid_mask, prediction_logprobs, 0)


def softmax_cross_entropy(logits, labels, ignore_index: int = -100):
    # Cast logits to f32
    # Flatten logits
    return F.cross_entropy(logits.to(torch.float32).view(-1, logits.shape[-1]), labels.to(torch.long).view(-1), ignore_index=ignore_index, reduction="none").view(labels.shape)


class ACTLossHead(nn.Module):
    def __init__(self, model: nn.Module, loss_type: str):
        super().__init__()
        self.model = model
        self.loss_fn = globals()[loss_type]
        
    def initial_carry(self, *args, **kwargs):
        return self.model.initial_carry(*args, **kwargs)  # type: ignore

    def forward(
        self,
        return_keys: Sequence[str],
        # Model args
        **model_kwargs,
    ) -> Tuple[Any, torch.Tensor, Dict[str, torch.Tensor], Optional[Dict[str, torch.Tensor]], torch.Tensor]:
        # Model logits
        # B x SeqLen x D
        new_carry, outputs = self.model(**model_kwargs)
        labels = new_carry.current_data["labels"]

        # Correctness
        with torch.no_grad():
            mask = labels != IGNORE_LABEL_ID
            loss_counts = mask.sum(-1)
            loss_divisor = loss_counts.clamp_min(1).unsqueeze(-1)  # Avoid NaNs in division

            is_correct = mask & (torch.argmax(outputs["logits"], dim=-1) == labels)
            seq_is_correct = is_correct.sum(-1) == loss_counts
            
            # Metrics (halted)
            valid_metrics = new_carry.halted & (loss_counts > 0)
            metrics = {
                "count": valid_metrics.sum(),
                
                "accuracy":       torch.where(valid_metrics, (is_correct.to(torch.float32) / loss_divisor).sum(-1), 0).sum(),
                "exact_accuracy": (valid_metrics & seq_is_correct).sum(),

                "q_halt_accuracy": (valid_metrics & ((outputs["q_halt_logits"] >= 0) == seq_is_correct)).sum(),
                "steps":          torch.where(valid_metrics, new_carry.steps, 0).sum(),
            }

        # Losses
        # FIXME: Assuming the batch is always full
        lm_loss = (self.loss_fn(outputs["logits"], labels, ignore_index=IGNORE_LABEL_ID) / loss_divisor).sum()
        q_halt_loss = F.binary_cross_entropy_with_logits(outputs["q_halt_logits"], seq_is_correct.to(outputs["q_halt_logits"].dtype), reduction="sum")

        metrics.update({
            "lm_loss": lm_loss.detach(),
            "q_halt_loss": q_halt_loss.detach(),
        })

        # Q continue (bootstrapping target loss)
        q_continue_loss = 0
        if "target_q_continue" in outputs:
            q_continue_loss = F.binary_cross_entropy_with_logits(outputs["q_continue_logits"], outputs["target_q_continue"], reduction="sum")

            metrics["q_continue_loss"] = q_continue_loss.detach()

        # Filter outputs for return
        detached_outputs = {k: outputs[k].detach() for k in return_keys if k in outputs}

        return new_carry, lm_loss + 0.5 * (q_halt_loss + q_continue_loss), metrics, detached_outputs, new_carry.halted.all()



In [ ]:
%%writefile models/sparse_embedding.py
from typing import Union

import torch
from torch import nn
import torch.distributed as dist
from torch.optim.optimizer import Optimizer, ParamsT

from models.common import trunc_normal_init_


class CastedSparseEmbedding(nn.Module):
    def __init__(self, num_embeddings: int, embedding_dim: int, batch_size: int, init_std: float, cast_to: torch.dtype):
        super().__init__()
        self.cast_to = cast_to

        # Real Weights
        # Truncated LeCun normal init
        self.weights = nn.Buffer(
            trunc_normal_init_(torch.empty((num_embeddings, embedding_dim)), std=init_std), persistent=True
        )

        # Local weights and IDs
        # Local embeddings, with gradient, not persistent
        self.local_weights = nn.Buffer(torch.zeros(batch_size, embedding_dim, requires_grad=True), persistent=False)
        # Local embedding IDs, not persistent
        self.local_ids = nn.Buffer(torch.zeros(batch_size, dtype=torch.int32), persistent=False)

    def forward(self, inputs: torch.Tensor) -> torch.Tensor:
        # Clamp indices to valid range to prevent OOB access from padded puzzle identifiers
        # (padding uses blank_identifier_id which may exceed num_embeddings)
        inputs = inputs.clamp(max=self.weights.shape[0] - 1)

        if not self.training:
            # Test mode, no gradient
            return self.weights[inputs].to(self.cast_to)
            
        # Training mode, fill puzzle embedding from weights
        with torch.no_grad():
            self.local_weights.copy_(self.weights[inputs])
            self.local_ids.copy_(inputs)

        return self.local_weights.to(self.cast_to)


class CastedSparseEmbeddingSignSGD_Distributed(Optimizer):
    def __init__(
        self,
        params: ParamsT,

        world_size: int,
        lr: Union[float, torch.Tensor] = 1e-3,
        weight_decay: float = 1e-2,
    ):
        if not 0.0 <= lr:
            raise ValueError(f"Invalid learning rate: {lr}")
        if not 0.0 <= weight_decay:
            raise ValueError(f"Invalid weight_decay value: {weight_decay}")

        defaults = dict(
            lr=lr,
            weight_decay=weight_decay,
            world_size=world_size
        )
        super().__init__(params, defaults)

    @torch.no_grad
    def step(self, closure=None):  # type: ignore
        for group in self.param_groups:
            # Find the sparse embedding weights
            local_weights_grad = None
            local_ids = None
            weights = None
            
            assert len(group["params"]) == 3
            for p in group["params"]:
                if p.requires_grad:
                    local_weights_grad = p.grad
                elif p.ndim == 1:
                    local_ids = p
                elif p.ndim == 2:
                    weights = p
                else:
                    assert False
                
            assert local_weights_grad is not None
            assert local_ids is not None
            assert weights is not None
        
            # Apply SignSGD
            # Adam ≈ SignSGD if gradient is very sparse
            _sparse_emb_signsgd_dist(
                local_weights_grad,
                local_ids,
                weights,
                
                lr=group["lr"],
                weight_decay=group["weight_decay"],
                world_size=group["world_size"]
            )


def _sparse_emb_signsgd_dist(
    local_weights_grad: torch.Tensor,
    local_ids: torch.Tensor,
    weights: torch.Tensor,
    
    lr: float,
    weight_decay: float,
    world_size: int
) -> None:
    N, D = local_weights_grad.shape
    
    # All-gather
    all_weights_grad = local_weights_grad
    all_ids = local_ids

    if world_size > 1:
        all_weights_grad = torch.empty((world_size * N, D), dtype=local_weights_grad.dtype, device=local_weights_grad.device)
        all_ids = torch.empty(world_size * N,               dtype=local_ids.dtype,          device=local_ids.device)
    
        dist.all_gather_into_tensor(all_weights_grad, local_weights_grad)
        dist.all_gather_into_tensor(all_ids,          local_ids)

    # Unique
    grad_ids, inv = all_ids.unique(return_inverse=True)

    grad = torch.zeros((grad_ids.shape[0], D), dtype=all_weights_grad.dtype, device=all_weights_grad.device)
    grad.scatter_add_(0, inv.unsqueeze(-1).expand(-1, D), all_weights_grad)

    # SignSGD with decoupled weight decay
    p = weights[grad_ids]

    p.mul_(1.0 - lr * weight_decay).add_(torch.sign(grad), alpha=-lr)

    # Write updated slices back
    weights[grad_ids] = p



In [ ]:
%%writefile models/hrm/hrm_act_v1.py
from typing import Tuple, List, Dict, Optional
from dataclasses import dataclass
import math

import torch
import torch.nn.functional as F
from torch import nn
from pydantic import BaseModel

from models.common import trunc_normal_init_
from models.layers import rms_norm, SwiGLU, Attention, RotaryEmbedding, CosSin, CastedEmbedding, CastedLinear
from models.sparse_embedding import CastedSparseEmbedding


@dataclass
class HierarchicalReasoningModel_ACTV1InnerCarry:
    z_H: torch.Tensor
    z_L: torch.Tensor


@dataclass
class HierarchicalReasoningModel_ACTV1Carry:
    inner_carry: HierarchicalReasoningModel_ACTV1InnerCarry
    
    steps: torch.Tensor
    halted: torch.Tensor
    
    current_data: Dict[str, torch.Tensor]


class HierarchicalReasoningModel_ACTV1Config(BaseModel):
    batch_size: int
    seq_len: int
    puzzle_emb_ndim: int = 0
    num_puzzle_identifiers: int
    vocab_size: int

    H_cycles: int
    L_cycles: int

    H_layers: int
    L_layers: int

    # Transformer config
    hidden_size: int
    expansion: float
    num_heads: int
    pos_encodings: str

    rms_norm_eps: float = 1e-5
    rope_theta: float = 10000.0
    
    # Halting Q-learning config
    halt_max_steps: int
    halt_exploration_prob: float

    forward_dtype: str = "bfloat16"


class HierarchicalReasoningModel_ACTV1Block(nn.Module):
    def __init__(self, config: HierarchicalReasoningModel_ACTV1Config) -> None:
        super().__init__()

        self.self_attn = Attention(
            hidden_size=config.hidden_size,
            head_dim=config.hidden_size // config.num_heads,
            num_heads=config.num_heads,
            num_key_value_heads=config.num_heads,
            causal=False
        )
        self.mlp = SwiGLU(
            hidden_size=config.hidden_size,
            expansion=config.expansion,
        )
        self.norm_eps = config.rms_norm_eps

    def forward(self, cos_sin: CosSin, hidden_states: torch.Tensor) -> torch.Tensor:
        # Post Norm
        # Self Attention
        hidden_states = rms_norm(hidden_states + self.self_attn(cos_sin=cos_sin, hidden_states=hidden_states), variance_epsilon=self.norm_eps)
        # Fully Connected
        hidden_states = rms_norm(hidden_states + self.mlp(hidden_states), variance_epsilon=self.norm_eps)
        return hidden_states


class HierarchicalReasoningModel_ACTV1ReasoningModule(nn.Module):
    def __init__(self, layers: List[HierarchicalReasoningModel_ACTV1Block]):
        super().__init__()

        self.layers = torch.nn.ModuleList(layers)

    def forward(self, hidden_states: torch.Tensor, input_injection: torch.Tensor, **kwargs) -> torch.Tensor:
        # Input injection (add)
        hidden_states = hidden_states + input_injection
        # Layers
        for layer in self.layers:
            hidden_states = layer(hidden_states=hidden_states, **kwargs)

        return hidden_states


class HierarchicalReasoningModel_ACTV1_Inner(nn.Module):
    def __init__(self, config: HierarchicalReasoningModel_ACTV1Config) -> None:
        super().__init__()
        self.config = config
        self.forward_dtype = getattr(torch, self.config.forward_dtype)

        # I/O
        self.embed_scale  = math.sqrt(self.config.hidden_size)
        embed_init_std = 1.0 / self.embed_scale

        self.embed_tokens = CastedEmbedding(self.config.vocab_size, self.config.hidden_size, init_std=embed_init_std, cast_to=self.forward_dtype)
        self.lm_head      = CastedLinear(self.config.hidden_size, self.config.vocab_size, bias=False)
        self.q_head       = CastedLinear(self.config.hidden_size, 2, bias=True)

        self.puzzle_emb_len = -(self.config.puzzle_emb_ndim // -self.config.hidden_size)  # ceil div
        if self.config.puzzle_emb_ndim > 0:
            # Zero init puzzle embeddings
            self.puzzle_emb = CastedSparseEmbedding(self.config.num_puzzle_identifiers, self.config.puzzle_emb_ndim,
                                                    batch_size=self.config.batch_size, init_std=0, cast_to=self.forward_dtype)

        # LM Blocks
        if self.config.pos_encodings == "rope":
            self.rotary_emb = RotaryEmbedding(dim=self.config.hidden_size // self.config.num_heads,
                                              max_position_embeddings=self.config.seq_len + self.puzzle_emb_len,
                                              base=self.config.rope_theta)
        elif self.config.pos_encodings == "learned":
            self.embed_pos = CastedEmbedding(self.config.seq_len + self.puzzle_emb_len, self.config.hidden_size, init_std=embed_init_std, cast_to=self.forward_dtype)
        else:
            raise NotImplementedError()

        # Reasoning Layers
        self.H_level = HierarchicalReasoningModel_ACTV1ReasoningModule(layers=[HierarchicalReasoningModel_ACTV1Block(self.config) for _i in range(self.config.H_layers)])
        self.L_level = HierarchicalReasoningModel_ACTV1ReasoningModule(layers=[HierarchicalReasoningModel_ACTV1Block(self.config) for _i in range(self.config.L_layers)])
        
        # Initial states
        self.H_init = nn.Buffer(trunc_normal_init_(torch.empty(self.config.hidden_size, dtype=self.forward_dtype), std=1), persistent=True)
        self.L_init = nn.Buffer(trunc_normal_init_(torch.empty(self.config.hidden_size, dtype=self.forward_dtype), std=1), persistent=True)

        # Q head special init
        # Init Q to (almost) zero for faster learning during bootstrapping
        with torch.no_grad():
            self.q_head.weight.zero_()
            self.q_head.bias.fill_(-5)  # type: ignore

    def _input_embeddings(self, input: torch.Tensor, puzzle_identifiers: torch.Tensor):
        # Token embedding
        embedding = self.embed_tokens(input.to(torch.int32))

        # Puzzle embeddings
        if self.config.puzzle_emb_ndim > 0:
            puzzle_embedding = self.puzzle_emb(puzzle_identifiers)
            
            pad_count = self.puzzle_emb_len * self.config.hidden_size - puzzle_embedding.shape[-1]
            if pad_count > 0:
                puzzle_embedding = F.pad(puzzle_embedding, (0, pad_count))

            embedding = torch.cat((puzzle_embedding.view(-1, self.puzzle_emb_len, self.config.hidden_size), embedding), dim=-2)

        # Position embeddings
        if self.config.pos_encodings == "learned":
            # scale by 1/sqrt(2) to maintain forward variance
            embedding = 0.707106781 * (embedding + self.embed_pos.embedding_weight.to(self.forward_dtype))

        # Scale
        return self.embed_scale * embedding

    def empty_carry(self, batch_size: int):
        return HierarchicalReasoningModel_ACTV1InnerCarry(
            z_H=torch.empty(batch_size, self.config.seq_len + self.puzzle_emb_len, self.config.hidden_size, dtype=self.forward_dtype),
            z_L=torch.empty(batch_size, self.config.seq_len + self.puzzle_emb_len, self.config.hidden_size, dtype=self.forward_dtype),
        )
        
    def reset_carry(self, reset_flag: torch.Tensor, carry: HierarchicalReasoningModel_ACTV1InnerCarry):
        return HierarchicalReasoningModel_ACTV1InnerCarry(
            z_H=torch.where(reset_flag.view(-1, 1, 1), self.H_init, carry.z_H),
            z_L=torch.where(reset_flag.view(-1, 1, 1), self.L_init, carry.z_L),
        )

    def forward(self, carry: HierarchicalReasoningModel_ACTV1InnerCarry, batch: Dict[str, torch.Tensor]) -> Tuple[HierarchicalReasoningModel_ACTV1InnerCarry, torch.Tensor, Tuple[torch.Tensor, torch.Tensor]]:
        seq_info = dict(
            cos_sin=self.rotary_emb() if hasattr(self, "rotary_emb") else None,
        )

        # Input encoding
        input_embeddings = self._input_embeddings(batch["inputs"], batch["puzzle_identifiers"])

        # Forward iterations
        with torch.no_grad():
            z_H, z_L = carry.z_H, carry.z_L

            for _H_step in range(self.config.H_cycles):
                for _L_step in range(self.config.L_cycles):
                    if not ((_H_step == self.config.H_cycles - 1) and (_L_step == self.config.L_cycles - 1)):
                        z_L = self.L_level(z_L, z_H + input_embeddings, **seq_info)

                if not (_H_step == self.config.H_cycles - 1):
                    z_H = self.H_level(z_H, z_L, **seq_info)

        assert not z_H.requires_grad and not z_L.requires_grad

        # 1-step grad
        z_L = self.L_level(z_L, z_H + input_embeddings, **seq_info)
        z_H = self.H_level(z_H, z_L, **seq_info)

        # LM Outputs
        new_carry = HierarchicalReasoningModel_ACTV1InnerCarry(z_H=z_H.detach(), z_L=z_L.detach())  # New carry no grad
        output = self.lm_head(z_H)[:, self.puzzle_emb_len:]

        # Q head
        q_logits = self.q_head(z_H[:, 0]).to(torch.float32)
        
        return new_carry, output, (q_logits[..., 0], q_logits[..., 1])


class HierarchicalReasoningModel_ACTV1(nn.Module):
    """ACT wrapper."""

    def __init__(self, config_dict: dict):
        super().__init__()
        self.config = HierarchicalReasoningModel_ACTV1Config(**config_dict)
        self.inner = HierarchicalReasoningModel_ACTV1_Inner(self.config)

    @property
    def puzzle_emb(self):
        return self.inner.puzzle_emb

    def initial_carry(self, batch: Dict[str, torch.Tensor]):
        batch_size = batch["inputs"].shape[0]

        return HierarchicalReasoningModel_ACTV1Carry(
            inner_carry=self.inner.empty_carry(batch_size),  # Empty is expected, it will be reseted in first pass as all sequences are halted.
            
            steps=torch.zeros((batch_size, ), dtype=torch.int32),
            halted=torch.ones((batch_size, ), dtype=torch.bool),  # Default to halted
            
            current_data={k: torch.empty_like(v) for k, v in batch.items()}
        )
        
    def forward(self, carry: HierarchicalReasoningModel_ACTV1Carry, batch: Dict[str, torch.Tensor]) -> Tuple[HierarchicalReasoningModel_ACTV1Carry, Dict[str, torch.Tensor]]:
        # Update data, carry (removing halted sequences)
        new_inner_carry = self.inner.reset_carry(carry.halted, carry.inner_carry)
        
        new_steps = torch.where(carry.halted, 0, carry.steps)

        new_current_data = {k: torch.where(carry.halted.view((-1, ) + (1, ) * (batch[k].ndim - 1)), batch[k], v) for k, v in carry.current_data.items()}

        # Forward inner model
        new_inner_carry, logits, (q_halt_logits, q_continue_logits) = self.inner(new_inner_carry, new_current_data)

        outputs = {
            "logits": logits,
            "q_halt_logits": q_halt_logits,
            "q_continue_logits": q_continue_logits
        }
        
        with torch.no_grad():
            # Step
            new_steps = new_steps + 1
            is_last_step = new_steps >= self.config.halt_max_steps
            
            halted = is_last_step

            # if training, and ACT is enabled
            if self.training and (self.config.halt_max_steps > 1):
                # Halt signal
                # NOTE: During evaluation, always use max steps, this is to guarantee the same halting steps inside a batch for batching purposes
                halted = halted | (q_halt_logits > q_continue_logits)

                # Exploration
                min_halt_steps = (torch.rand_like(q_halt_logits) < self.config.halt_exploration_prob) * torch.randint_like(new_steps, low=2, high=self.config.halt_max_steps + 1)

                halted = halted & (new_steps >= min_halt_steps)

                # Compute target Q
                # NOTE: No replay buffer and target networks for computing target Q-value.
                # As batch_size is large, there're many parallel envs.
                # Similar concept as PQN https://arxiv.org/abs/2407.04811
                next_q_halt_logits, next_q_continue_logits = self.inner(new_inner_carry, new_current_data)[-1]
                
                outputs["target_q_continue"] = torch.sigmoid(torch.where(is_last_step, next_q_halt_logits, torch.maximum(next_q_halt_logits, next_q_continue_logits)))

        return HierarchicalReasoningModel_ACTV1Carry(new_inner_carry, new_steps, halted, new_current_data), outputs



In [ ]:
%%writefile models/hrm/hrm_deep_v1.py
from typing import Tuple, List, Dict, Optional
from dataclasses import dataclass
import math

import torch
import torch.nn.functional as F
from torch import nn
from pydantic import BaseModel

from models.common import trunc_normal_init_
from models.layers import rms_norm, SwiGLU, Attention, RotaryEmbedding, CosSin, CastedEmbedding, CastedLinear
from models.sparse_embedding import CastedSparseEmbedding


@dataclass
class HierarchicalReasoningModel_DeepV1InnerCarry:
    z_H: torch.Tensor
    z_L: torch.Tensor


@dataclass
class HierarchicalReasoningModel_DeepV1Carry:
    inner_carry: HierarchicalReasoningModel_DeepV1InnerCarry
    
    steps: torch.Tensor
    halted: torch.Tensor
    
    current_data: Dict[str, torch.Tensor]


class HierarchicalReasoningModel_DeepV1Config(BaseModel):
    batch_size: int
    seq_len: int
    puzzle_emb_ndim: int = 0
    num_puzzle_identifiers: int
    vocab_size: int

    H_cycles: int
    L_cycles: int

    H_layers: int
    L_layers: int

    # Transformer config
    hidden_size: int
    expansion: float
    num_heads: int
    pos_encodings: str

    rms_norm_eps: float = 1e-5
    rope_theta: float = 10000.0
    
    # Deep config
    halt_max_steps: int
    halt_exploration_prob: Optional[float] = None # unused

    forward_dtype: str = "bfloat16"


class HierarchicalReasoningModel_DeepV1Block(nn.Module):
    def __init__(self, config: HierarchicalReasoningModel_DeepV1Config) -> None:
        super().__init__()

        self.self_attn = Attention(
            hidden_size=config.hidden_size,
            head_dim=config.hidden_size // config.num_heads,
            num_heads=config.num_heads,
            num_key_value_heads=config.num_heads,
            causal=False
        )
        self.mlp = SwiGLU(
            hidden_size=config.hidden_size,
            expansion=config.expansion,
        )
        self.norm_eps = config.rms_norm_eps

    def forward(self, cos_sin: CosSin, hidden_states: torch.Tensor) -> torch.Tensor:
        hidden_states = rms_norm(hidden_states + self.self_attn(cos_sin=cos_sin, hidden_states=hidden_states), variance_epsilon=self.norm_eps)
        hidden_states = rms_norm(hidden_states + self.mlp(hidden_states), variance_epsilon=self.norm_eps)
        return hidden_states


class HierarchicalReasoningModel_DeepV1ReasoningModule(nn.Module):
    def __init__(self, layers: List[HierarchicalReasoningModel_DeepV1Block]):
        super().__init__()
        self.layers = torch.nn.ModuleList(layers)

    def forward(self, hidden_states: torch.Tensor, input_injection: torch.Tensor, **kwargs) -> torch.Tensor:
        hidden_states = hidden_states + input_injection
        for layer in self.layers:
            hidden_states = layer(hidden_states=hidden_states, **kwargs)
        return hidden_states


class HierarchicalReasoningModel_DeepV1_Inner(nn.Module):
    def __init__(self, config: HierarchicalReasoningModel_DeepV1Config) -> None:
        super().__init__()
        self.config = config
        self.forward_dtype = getattr(torch, self.config.forward_dtype)

        self.embed_scale  = math.sqrt(self.config.hidden_size)
        embed_init_std = 1.0 / self.embed_scale

        self.embed_tokens = CastedEmbedding(self.config.vocab_size, self.config.hidden_size, init_std=embed_init_std, cast_to=self.forward_dtype)
        self.lm_head      = CastedLinear(self.config.hidden_size, self.config.vocab_size, bias=False)
        self.q_head       = CastedLinear(self.config.hidden_size, 2, bias=True)

        self.puzzle_emb_len = -(self.config.puzzle_emb_ndim // -self.config.hidden_size)
        if self.config.puzzle_emb_ndim > 0:
            self.puzzle_emb = CastedSparseEmbedding(self.config.num_puzzle_identifiers, self.config.puzzle_emb_ndim,
                                                    batch_size=self.config.batch_size, init_std=0, cast_to=self.forward_dtype)

        if self.config.pos_encodings == "rope":
            self.rotary_emb = RotaryEmbedding(dim=self.config.hidden_size // self.config.num_heads,
                                              max_position_embeddings=self.config.seq_len + self.puzzle_emb_len,
                                              base=self.config.rope_theta)
        elif self.config.pos_encodings == "learned":
            self.embed_pos = CastedEmbedding(self.config.seq_len + self.puzzle_emb_len, self.config.hidden_size, init_std=embed_init_std, cast_to=self.forward_dtype)
        else:
            raise NotImplementedError()

        self.H_level = HierarchicalReasoningModel_DeepV1ReasoningModule(layers=[HierarchicalReasoningModel_DeepV1Block(self.config) for _i in range(self.config.H_layers)])
        self.L_level = HierarchicalReasoningModel_DeepV1ReasoningModule(layers=[HierarchicalReasoningModel_DeepV1Block(self.config) for _i in range(self.config.L_layers)])
        
        self.H_init = nn.Buffer(trunc_normal_init_(torch.empty(self.config.hidden_size, dtype=self.forward_dtype), std=1), persistent=True)
        self.L_init = nn.Buffer(trunc_normal_init_(torch.empty(self.config.hidden_size, dtype=self.forward_dtype), std=1), persistent=True)

        with torch.no_grad():
            self.q_head.weight.zero_()
            self.q_head.bias.fill_(-5)

    def _input_embeddings(self, input: torch.Tensor, puzzle_identifiers: torch.Tensor):
        embedding = self.embed_tokens(input.to(torch.int32))
        if self.config.puzzle_emb_ndim > 0:
            puzzle_embedding = self.puzzle_emb(puzzle_identifiers)
            pad_count = self.puzzle_emb_len * self.config.hidden_size - puzzle_embedding.shape[-1]
            if pad_count > 0:
                puzzle_embedding = F.pad(puzzle_embedding, (0, pad_count))
            embedding = torch.cat((puzzle_embedding.view(-1, self.puzzle_emb_len, self.config.hidden_size), embedding), dim=-2)

        if self.config.pos_encodings == "learned":
            embedding = 0.707106781 * (embedding + self.embed_pos.embedding_weight.to(self.forward_dtype))

        return self.embed_scale * embedding

    def empty_carry(self, batch_size: int):
        return HierarchicalReasoningModel_DeepV1InnerCarry(
            z_H=torch.empty(batch_size, self.config.seq_len + self.puzzle_emb_len, self.config.hidden_size, dtype=self.forward_dtype),
            z_L=torch.empty(batch_size, self.config.seq_len + self.puzzle_emb_len, self.config.hidden_size, dtype=self.forward_dtype),
        )
        
    def reset_carry(self, reset_flag: torch.Tensor, carry: HierarchicalReasoningModel_DeepV1InnerCarry):
        return HierarchicalReasoningModel_DeepV1InnerCarry(
            z_H=torch.where(reset_flag.view(-1, 1, 1), self.H_init, carry.z_H),
            z_L=torch.where(reset_flag.view(-1, 1, 1), self.L_init, carry.z_L),
        )

    def forward(self, carry: HierarchicalReasoningModel_DeepV1InnerCarry, batch: Dict[str, torch.Tensor]) -> Tuple[HierarchicalReasoningModel_DeepV1InnerCarry, torch.Tensor, Tuple[torch.Tensor, torch.Tensor]]:
        seq_info = dict(
            cos_sin=self.rotary_emb() if hasattr(self, "rotary_emb") else None,
        )

        input_embeddings = self._input_embeddings(batch["inputs"], batch["puzzle_identifiers"])

        with torch.no_grad():
            z_H, z_L = carry.z_H, carry.z_L

            for _H_step in range(self.config.H_cycles):
                for _L_step in range(self.config.L_cycles):
                    if not ((_H_step == self.config.H_cycles - 1) and (_L_step == self.config.L_cycles - 1)):
                        z_L = self.L_level(z_L, z_H + input_embeddings, **seq_info)

                if not (_H_step == self.config.H_cycles - 1):
                    z_H = self.H_level(z_H, z_L, **seq_info)

        assert not z_H.requires_grad and not z_L.requires_grad

        z_L = self.L_level(z_L, z_H + input_embeddings, **seq_info)
        z_H = self.H_level(z_H, z_L, **seq_info)

        new_carry = HierarchicalReasoningModel_DeepV1InnerCarry(z_H=z_H.detach(), z_L=z_L.detach())
        output = self.lm_head(z_H)[:, self.puzzle_emb_len:]
        q_logits = self.q_head(z_H[:, 0]).to(torch.float32)
        
        return new_carry, output, (q_logits[..., 0], q_logits[..., 1])


class HierarchicalReasoningModel_DeepV1(nn.Module):
    """DeepReasoner wrapper (Fixed iterations, no early halting)."""

    def __init__(self, config_dict: dict):
        super().__init__()
        self.config = HierarchicalReasoningModel_DeepV1Config(**config_dict)
        self.inner = HierarchicalReasoningModel_DeepV1_Inner(self.config)

    @property
    def puzzle_emb(self):
        return self.inner.puzzle_emb

    def initial_carry(self, batch: Dict[str, torch.Tensor]):
        batch_size = batch["inputs"].shape[0]

        return HierarchicalReasoningModel_DeepV1Carry(
            inner_carry=self.inner.empty_carry(batch_size),
            steps=torch.zeros((batch_size, ), dtype=torch.int32),
            halted=torch.ones((batch_size, ), dtype=torch.bool),
            current_data={k: torch.empty_like(v) for k, v in batch.items()}
        )
        
    def forward(self, carry: HierarchicalReasoningModel_DeepV1Carry, batch: Dict[str, torch.Tensor]) -> Tuple[HierarchicalReasoningModel_DeepV1Carry, Dict[str, torch.Tensor]]:
        new_inner_carry = self.inner.reset_carry(carry.halted, carry.inner_carry)
        new_steps = torch.where(carry.halted, 0, carry.steps)
        new_current_data = {k: torch.where(carry.halted.view((-1, ) + (1, ) * (batch[k].ndim - 1)), batch[k], v) for k, v in carry.current_data.items()}

        new_inner_carry, logits, (q_halt_logits, q_continue_logits) = self.inner(new_inner_carry, new_current_data)

        outputs = {
            "logits": logits,
            "q_halt_logits": q_halt_logits,
            "q_continue_logits": q_continue_logits
        }
        
        with torch.no_grad():
            new_steps = new_steps + 1
            is_last_step = new_steps >= self.config.halt_max_steps
            halted = is_last_step # NO EARLY HALTING, fixed iterations!

            outputs["target_q_continue"] = torch.zeros_like(q_continue_logits)

        return HierarchicalReasoningModel_DeepV1Carry(new_inner_carry, new_steps, halted, new_current_data), outputs



In [ ]:
%%writefile models/hrm/hrm_hrb_v1.py
from typing import Tuple, List, Dict, Optional
from dataclasses import dataclass
import math

import torch
import torch.nn.functional as F
from torch import nn
from pydantic import BaseModel

from models.common import trunc_normal_init_
from models.layers import rms_norm, SwiGLU, Attention, RotaryEmbedding, CosSin, CastedEmbedding, CastedLinear
from models.sparse_embedding import CastedSparseEmbedding


@dataclass
class HierarchicalReasoningModel_HRBV1InnerCarry:
    pass


@dataclass
class HierarchicalReasoningModel_HRBV1Carry:
    inner_carry: HierarchicalReasoningModel_HRBV1InnerCarry
    steps: torch.Tensor
    halted: torch.Tensor
    current_data: Dict[str, torch.Tensor]


class HierarchicalReasoningModel_HRBV1Config(BaseModel):
    batch_size: int
    seq_len: int
    puzzle_emb_ndim: int = 0
    num_puzzle_identifiers: int
    vocab_size: int

    H_layers: int = 2
    L_layers: int = 1

    hidden_size: int
    expansion: float
    num_heads: int
    pos_encodings: str

    rms_norm_eps: float = 1e-5
    rope_theta: float = 10000.0

    forward_dtype: str = "bfloat16"
    
    halt_exploration_prob: Optional[float] = None
    halt_max_steps: Optional[int] = None
    H_cycles: Optional[int] = None
    L_cycles: Optional[int] = None


class HierarchicalReasoningModel_HRBV1Block(nn.Module):
    def __init__(self, config: HierarchicalReasoningModel_HRBV1Config) -> None:
        super().__init__()

        self.self_attn = Attention(
            hidden_size=config.hidden_size,
            head_dim=config.hidden_size // config.num_heads,
            num_heads=config.num_heads,
            num_key_value_heads=config.num_heads,
            causal=False
        )
        self.mlp = SwiGLU(
            hidden_size=config.hidden_size,
            expansion=config.expansion,
        )
        self.norm_eps = config.rms_norm_eps

    def forward(self, cos_sin: CosSin, hidden_states: torch.Tensor) -> torch.Tensor:
        hidden_states = rms_norm(hidden_states + self.self_attn(cos_sin=cos_sin, hidden_states=hidden_states), variance_epsilon=self.norm_eps)
        hidden_states = rms_norm(hidden_states + self.mlp(hidden_states), variance_epsilon=self.norm_eps)
        return hidden_states


class HierarchicalReasoningModel_HRBV1ReasoningModule(nn.Module):
    def __init__(self, layers: List[HierarchicalReasoningModel_HRBV1Block]):
        super().__init__()
        self.layers = torch.nn.ModuleList(layers)

    def forward(self, hidden_states: torch.Tensor, **kwargs) -> torch.Tensor:
        for layer in self.layers:
            hidden_states = layer(hidden_states=hidden_states, **kwargs)
        return hidden_states


class HierarchicalReasoningModel_HRBV1_Inner(nn.Module):
    def __init__(self, config: HierarchicalReasoningModel_HRBV1Config) -> None:
        super().__init__()
        self.config = config
        self.forward_dtype = getattr(torch, self.config.forward_dtype)

        self.embed_scale  = math.sqrt(self.config.hidden_size)
        embed_init_std = 1.0 / self.embed_scale

        self.embed_tokens = CastedEmbedding(self.config.vocab_size, self.config.hidden_size, init_std=embed_init_std, cast_to=self.forward_dtype)
        self.lm_head      = CastedLinear(self.config.hidden_size, self.config.vocab_size, bias=False)
        self.q_head       = CastedLinear(self.config.hidden_size, 2, bias=True)

        self.puzzle_emb_len = -(self.config.puzzle_emb_ndim // -self.config.hidden_size)
        if self.config.puzzle_emb_ndim > 0:
            self.puzzle_emb = CastedSparseEmbedding(self.config.num_puzzle_identifiers, self.config.puzzle_emb_ndim,
                                                    batch_size=self.config.batch_size, init_std=0, cast_to=self.forward_dtype)

        if self.config.pos_encodings == "rope":
            self.rotary_emb = RotaryEmbedding(dim=self.config.hidden_size // self.config.num_heads,
                                              max_position_embeddings=self.config.seq_len + self.puzzle_emb_len,
                                              base=self.config.rope_theta)
        elif self.config.pos_encodings == "learned":
            self.embed_pos = CastedEmbedding(self.config.seq_len + self.puzzle_emb_len, self.config.hidden_size, init_std=embed_init_std, cast_to=self.forward_dtype)
        else:
            raise NotImplementedError()

        self.fast_path = HierarchicalReasoningModel_HRBV1ReasoningModule(layers=[HierarchicalReasoningModel_HRBV1Block(self.config) for _i in range(self.config.L_layers)])
        self.deep_path = HierarchicalReasoningModel_HRBV1ReasoningModule(layers=[HierarchicalReasoningModel_HRBV1Block(self.config) for _i in range(self.config.H_layers)])
        
        self.gate = nn.Parameter(torch.tensor([0.05], dtype=self.forward_dtype))

        with torch.no_grad():
            self.q_head.weight.zero_()
            self.q_head.bias.fill_(-5)

    def _input_embeddings(self, input: torch.Tensor, puzzle_identifiers: torch.Tensor):
        embedding = self.embed_tokens(input.to(torch.int32))
        if self.config.puzzle_emb_ndim > 0:
            puzzle_embedding = self.puzzle_emb(puzzle_identifiers)
            pad_count = self.puzzle_emb_len * self.config.hidden_size - puzzle_embedding.shape[-1]
            if pad_count > 0:
                puzzle_embedding = F.pad(puzzle_embedding, (0, pad_count))
            embedding = torch.cat((puzzle_embedding.view(-1, self.puzzle_emb_len, self.config.hidden_size), embedding), dim=-2)

        if self.config.pos_encodings == "learned":
            embedding = 0.707106781 * (embedding + self.embed_pos.embedding_weight.to(self.forward_dtype))

        return self.embed_scale * embedding

    def empty_carry(self, batch_size: int):
        return HierarchicalReasoningModel_HRBV1InnerCarry()
        
    def reset_carry(self, reset_flag: torch.Tensor, carry: HierarchicalReasoningModel_HRBV1InnerCarry):
        return carry

    def forward(self, carry: HierarchicalReasoningModel_HRBV1InnerCarry, batch: Dict[str, torch.Tensor]) -> Tuple[HierarchicalReasoningModel_HRBV1InnerCarry, torch.Tensor, Tuple[torch.Tensor, torch.Tensor]]:
        seq_info = dict(
            cos_sin=self.rotary_emb() if hasattr(self, "rotary_emb") else None,
        )

        input_embeddings = self._input_embeddings(batch["inputs"], batch["puzzle_identifiers"])

        fast_out = self.fast_path(input_embeddings, **seq_info)
        deep_out = self.deep_path(input_embeddings, **seq_info)
        
        gate_val = torch.sigmoid(self.gate.to(torch.float32)).to(input_embeddings.dtype)
        hidden_states = fast_out + gate_val * deep_out

        output = self.lm_head(hidden_states)[:, self.puzzle_emb_len:]
        q_logits = self.q_head(hidden_states[:, 0]).to(torch.float32)
        
        return carry, output, (q_logits[..., 0], q_logits[..., 1])


class HierarchicalReasoningModel_HRBV1(nn.Module):
    def __init__(self, config_dict: dict):
        super().__init__()
        self.config = HierarchicalReasoningModel_HRBV1Config(**config_dict)
        self.inner = HierarchicalReasoningModel_HRBV1_Inner(self.config)

    @property
    def puzzle_emb(self):
        return self.inner.puzzle_emb

    def initial_carry(self, batch: Dict[str, torch.Tensor]):
        batch_size = batch["inputs"].shape[0]

        return HierarchicalReasoningModel_HRBV1Carry(
            inner_carry=self.inner.empty_carry(batch_size),
            steps=torch.zeros((batch_size, ), dtype=torch.int32),
            halted=torch.ones((batch_size, ), dtype=torch.bool),
            current_data={k: torch.empty_like(v) for k, v in batch.items()}
        )
        
    def forward(self, carry: HierarchicalReasoningModel_HRBV1Carry, batch: Dict[str, torch.Tensor]) -> Tuple[HierarchicalReasoningModel_HRBV1Carry, Dict[str, torch.Tensor]]:
        new_inner_carry = self.inner.reset_carry(carry.halted, carry.inner_carry)
        new_steps = torch.where(carry.halted, 0, carry.steps)
        new_current_data = {k: torch.where(carry.halted.view((-1, ) + (1, ) * (batch[k].ndim - 1)), batch[k], v) for k, v in carry.current_data.items()}

        new_inner_carry, logits, (q_halt_logits, q_continue_logits) = self.inner(new_inner_carry, new_current_data)

        outputs = {
            "logits": logits,
            "q_halt_logits": q_halt_logits,
            "q_continue_logits": q_continue_logits
        }
        
        with torch.no_grad():
            new_steps = new_steps + 1
            halted = torch.ones_like(carry.halted) # HRB always halts in 1 step!
            outputs["target_q_continue"] = torch.zeros_like(q_continue_logits) # Not used

        return HierarchicalReasoningModel_HRBV1Carry(new_inner_carry, new_steps, halted, new_current_data), outputs



In [ ]:
%%writefile utils/functions.py
import importlib
import inspect


def load_model_class(identifier: str, prefix: str = "models."):
    module_path, class_name = identifier.split('@')

    # Import the module
    module = importlib.import_module(prefix + module_path)
    cls = getattr(module, class_name)
    
    return cls


def get_model_source_path(identifier: str, prefix: str = "models."):
    module_path, class_name = identifier.split('@')

    module = importlib.import_module(prefix + module_path)
    return inspect.getsourcefile(module)



## Step 1: Install Dependencies & Download Checkpoint

In [ ]:
!pip install -r requirements.txt peft huggingface_hub wandb
import os
from huggingface_hub import snapshot_download
os.makedirs('checkpoints/HRM-checkpoint-sudoku-extreme', exist_ok=True)
snapshot_download(repo_id='sapientinc/HRM-checkpoint-sudoku-extreme', local_dir='checkpoints/HRM-checkpoint-sudoku-extreme')


## Step 2: Generate 6x6 Sudoku Dataset

In [ ]:
!python dataset/build_6x6_sudoku_dataset.py --output-dir data/sudoku-6x6 --num-train 1000 --num-test 100 --holes 20


## Step 3: Fine-Tune HRM on 6x6 Sudoku using LoRA

Using architecture: hrm_hrb_v1

In [ ]:
import os
os.environ['WANDB_MODE'] = 'offline'

!python finetune_lora.py \
    arch=hrm_hrb_v1 \
    data_path=data/sudoku-6x6 \
    epochs=5000 \
    eval_interval=500 \
    checkpoint_every_eval=True \
    global_batch_size=64 \
    lr=7e-5 \
    puzzle_emb_lr=7e-5 \
    weight_decay=1.0 \
    puzzle_emb_weight_decay=1.0 \
    +load_checkpoint=checkpoints/HRM-checkpoint-sudoku-extreme/checkpoint


## Step 4: Evaluate the Trained Model

Finds the latest checkpoint and runs evaluation.

In [ ]:
import os

latest_step = -1
best_checkpoint_path = None

for root, dirs, files in os.walk('checkpoints'):
    for file in files:
        if file.startswith('step_') and 'all_preds' not in file:
            try:
                step_num = int(file.split('_')[1])
                if step_num > latest_step:
                    latest_step = step_num
                    best_checkpoint_path = os.path.join(root, file)
            except ValueError:
                pass

if best_checkpoint_path is None:
    print('ERROR: No checkpoint found!')
else:
    print(f'Latest checkpoint: {best_checkpoint_path}  (step {latest_step})')
    os.environ['DISABLE_COMPILE'] = '1'
    !python evaluate.py checkpoint="{best_checkpoint_path}"
